In [ ]:
# -*- coding: utf-8 -*-
"""
GPT-4o Energy Arbitrage → Hourly Operational CSV
Outputs CSV with columns (one row per hour, 24*365 = 8760 rows):
  prices_actual, prices_forecast, actual_demand, forecast_demand,
  soc, charge_MW, discharge_MW, import_MW, export_MW, profit_step
"""

import os
import json
import time
import re
import csv
import numpy as np
from dataclasses import dataclass
from typing import List, Tuple, Optional, Dict
from openai import OpenAI
from google.colab import drive, userdata

drive.mount('/content/drive')


# ============================================================================
# Configuration
# ============================================================================

@dataclass
class Config:
    test_file: str = "/content/drive/MyDrive/energy_finetuning_35/test_qwen_prevday_ITALY_2.jsonl"  # path to your JSONL
    output_csv: str = "/content/drive/MyDrive/energy_finetuning_35/gpt4_hourly_arbitrage.csv"
    model: str = "gpt-4"
    temperature: float = 0.0
    max_tokens: int = 4000
    n_days: int = 365
    delay_between_calls: float = 1.0
    resume_from_day: int = 312  # set > 0 to resume


# ============================================================================
# System Prompt (same as original)
# ============================================================================

SYSTEM_PROMPT = """\
You are an expert energy trader operating a battery storage system for electricity price arbitrage.

## Battery Model

You control a battery with these parameters (provided per day):
- capacity_kwh: total energy capacity (kWh)
- max_c_kw / max_d_kw: max charge / discharge power (kW)
- eta_c / eta_d: charge / discharge efficiency
- soc0: initial state-of-charge (fraction, 0 to 1)
- soc_min / soc_max: SOC bounds (fractions)
- soc_target: minimum final SOC at end of day (fraction)
- dt_hours: time step duration (hours)
- allow_export: whether selling back to grid is allowed

## SOC Dynamics

At each hour t (0 to 23), you choose charge power c[t] >= 0 and discharge power d[t] >= 0.
You cannot charge and discharge simultaneously.

SOC updates as:
  soc[t+1] = soc[t] + (eta_c * c[t] * dt - d[t] * dt / eta_d) / capacity_kwh

Constraints:
  0 <= c[t] <= max_c_kw
  0 <= d[t] <= max_d_kw
  c[t] and d[t] cannot both be > 0 at the same time
  soc_min <= soc[t] <= soc_max   for all t = 0..24
  soc[0] = soc0
  soc[24] >= soc_target

## Grid Interaction

Net grid power: net[t] = demand[t] + c[t] - d[t]
If allow_export is true:
  import[t] - export[t] = net[t],  import[t] >= 0, export[t] >= 0
If allow_export is false:
  import[t] >= net[t],  import[t] >= 0

## Objective (minimize total cost)

If allow_export:
  cost = sum over t of (price[t] * import[t] * dt - price[t] * export[t] * dt)
If not allow_export:
  cost = sum over t of (price[t] * import[t] * dt)

## Your Task

Given the battery parameters, forecast prices for today, forecast demand for today, and yesterday's price forecast errors, determine:
1. The optimal SOC trajectory: 25 values [soc[0], soc[1], ..., soc[24]]
2. The resulting objective cost

## How to Reason

1. Identify the cheapest and most expensive hours from the price forecast
2. Plan to charge during cheap hours and discharge during expensive hours
3. Account for efficiency losses (you lose energy on both charge and discharge)
4. Respect all constraints (SOC bounds, power limits, no simultaneous charge/discharge)
5. Calculate the resulting cost carefully
6. Consider yesterday's forecast errors - if forecasts were systematically biased, adjust

## Output Format

After your reasoning, you MUST end with EXACTLY this format:

SOC_TRAJECTORY: [s0, s1, s2, ..., s24]
OBJECTIVE_COST: <number>

Where s0..s24 are 25 comma-separated floats (SOC fractions) and the cost is a single number.
These must be the LAST two lines of your response.
"""


# ============================================================================
# Format Input Prompt
# ============================================================================

def format_user_prompt(user_message_json: str) -> str:
    """Format the user_message JSON into a readable prompt."""
    data = json.loads(user_message_json) if isinstance(user_message_json, str) else user_message_json

    lines = []
    lines.append("## Battery Parameters")
    lines.append(f"- Capacity: {data['capacity_kwh']} kWh")
    lines.append(f"- Max charge: {data['max_c_kw']} kW, Max discharge: {data['max_d_kw']} kW")
    lines.append(f"- Charge efficiency: {data['eta_c']}, Discharge efficiency: {data['eta_d']}")
    lines.append(f"- Initial SOC: {data['soc0']}")
    lines.append(f"- SOC bounds: [{data['soc_min']}, {data['soc_max']}]")
    lines.append(f"- SOC target (end of day): {data['soc_target']}")
    lines.append(f"- Time step: {data['dt_hours']} hours")
    lines.append(f"- Allow export: {data['allow_export']}")

    prices = data['forecast_prices_today']
    demand = data['forecast_demand_today']

    lines.append("")
    lines.append("## Hourly Forecast Data")
    lines.append("| Hour | Price (€/MWh) | Demand (MW) |")
    lines.append("|------|---------------|-------------|")
    for h in range(24):
        lines.append(f"| {h:02d}   | {prices[h]:13.2f} | {demand[h]:11.2f} |")

    mean_price = np.mean(prices)
    lines.append(f"\nMean forecast price: {mean_price:.2f} €/MWh")
    cheapest = sorted(range(24), key=lambda h: prices[h])
    lines.append(f"Cheapest hours: {cheapest[:6]}")
    lines.append(f"Most expensive hours: {cheapest[-6:][::-1]}")

    if 'yesterday_error_stats' in data and data['yesterday_error_stats']:
        err = data['yesterday_error_stats']
        lines.append("")
        lines.append("## Yesterday's Forecast Error Stats")
        lines.append(f"- Mean error: {err['mean']:.2f} (forecast - actual)")
        lines.append(f"- Std: {err['std']:.2f}")
        lines.append(f"- Max positive error: {err['max_positive']:.2f}")
        lines.append(f"- Max negative error: {err['max_negative']:.2f}")

    if 'actual_prices_yesterday' in data:
        lines.append("")
        lines.append("## Yesterday's Actual vs Forecast Prices")
        lines.append("| Hour | Actual | Forecast | Error |")
        lines.append("|------|--------|----------|-------|")
        act_y = data['actual_prices_yesterday']
        frc_y = data['forecast_prices_yesterday']
        for h in range(min(24, len(act_y), len(frc_y))):
            lines.append(f"| {h:02d}   | {act_y[h]:6.2f} | {frc_y[h]:8.2f} | {frc_y[h]-act_y[h]:+6.2f} |")

    if 'naive_soc' in data:
        lines.append("")
        lines.append("## Naive Optimizer SOC Trajectory (for reference)")
        lines.append(f"SOC: {[round(s, 4) for s in data['naive_soc']]}")
        lines.append(f"Naive cost: {data['naive_cost']:.2f}")

    return "\n".join(lines)


# ============================================================================
# Parse SOC + Cost from Model Output
# ============================================================================

def parse_soc_and_cost(text: str) -> Tuple[Optional[List[float]], Optional[float]]:
    soc = None
    cost = None

    soc_match = re.search(r'SOC_TRAJECTORY:\s*\[([^\]]+)\]', text, re.IGNORECASE)
    if soc_match:
        try:
            soc = [float(x.strip()) for x in soc_match.group(1).split(',')]
        except ValueError:
            pass

    cost_match = re.search(r'OBJECTIVE_COST:\s*([-\d.eE+]+)', text, re.IGNORECASE)
    if cost_match:
        try:
            cost = float(cost_match.group(1))
        except ValueError:
            pass

    return soc, cost


# ============================================================================
# Derive Hourly Operational Data from SOC Trajectory
# ============================================================================

def derive_hourly_data(
    soc_trajectory: List[float],
    battery: Dict[str, float],
    actual_prices: List[float],
    forecast_prices: List[float],
    actual_demand: List[float],
    forecast_demand: List[float],
    allow_export: bool,
) -> List[Dict]:
    """
    From a 25-element SOC trajectory, derive per-hour:
    charge_MW, discharge_MW, import_MW, export_MW, profit_step
    and combine with prices/demand data.
    Returns 24 row dicts.
    """
    C = battery['capacity_kwh']
    eta_c = battery['eta_c']
    eta_d = battery['eta_d']
    dt = 1.0  # hours

    rows = []
    for t in range(24):
        delta_soc = soc_trajectory[t + 1] - soc_trajectory[t]
        delta_energy = delta_soc * C  # kWh

        if delta_energy >= 0:
            # Charging
            c_t = delta_energy / (eta_c * dt)  # kW
            d_t = 0.0
        else:
            # Discharging: delta_soc * C = -d * dt / eta_d  =>  d = -delta_soc * C * eta_d / dt
            c_t = 0.0
            d_t = -delta_soc * C * eta_d / dt  # kW

        # Convert kW to MW (divide by 1000) — adjust if your data is already in MW
        # Based on your example (charge/discharge ~ 10 MW range), it seems data is in MW-scale
        # so we keep kW as-is if battery params are in kW, or adjust as needed.
        # Looking at your example: discharge_MW=10.78, and max_d_kw=12.36
        # So 12.36 kW capacity giving 10.78 "MW" means the column is actually in kW
        # but labelled MW. We just output the raw number.
        charge_val = round(c_t, 2)
        discharge_val = round(d_t, 2)

        # Net grid power
        net = actual_demand[t] + c_t - d_t

        if allow_export:
            if net >= 0:
                imp_t = net
                exp_t = 0.0
            else:
                imp_t = 0.0
                exp_t = -net
        else:
            imp_t = max(net, 0.0)
            exp_t = 0.0

        # Profit at this step (revenue from export - cost of import)
        # profit = export_revenue - import_cost
        # = price * export * dt - price * import * dt
        # But the original uses "cost", so profit = -cost = -(price*imp - price*exp)
        # Your example shows profit_step as a positive number (549.78)
        # So: profit_step = price * export * dt - price * import * dt  (if net exporter, positive)
        # OR: profit_step = price * (export - import) * dt ... that would be negative for importers
        # Given your example has export=0, import=11.96, price=51 → cost = 51*11.96 = 609.96
        # But you show 549.78. Let me check: demand=22.74, discharge=10.78, charge=0
        # net = 22.74 + 0 - 10.78 = 11.96  →  import = 11.96
        # Without battery: import would be 22.74, cost = 22.74 * 51 = 1159.74
        # With battery: import = 11.96, cost = 11.96 * 51 = 609.96
        # Savings = 1159.74 - 609.96 = 549.78  ← matches!
        # So profit_step = price * (demand - import) * dt = price * (d_t - c_t) * dt
        # Or equivalently: price * (demand_no_battery_import - actual_import) * dt

        baseline_cost = actual_prices[t] * actual_demand[t] * dt
        actual_cost = actual_prices[t] * imp_t * dt - actual_prices[t] * exp_t * dt
        profit_step = baseline_cost - actual_cost

        rows.append({
            'prices_actual': round(actual_prices[t], 2),
            'prices_forecast': round(forecast_prices[t], 2),
            'actual_demand': round(actual_demand[t], 2),
            'forecast_demand': round(forecast_demand[t], 2),
            'soc': round(soc_trajectory[t], 4),
            'charge_MW': charge_val,
            'discharge_MW': discharge_val,
            'import_MW': round(imp_t, 2),
            'export_MW': round(exp_t, 2),
            'profit_step': round(profit_step, 2),
        })

    return rows


# ============================================================================
# OpenAI API Call
# ============================================================================

def call_openai(client: OpenAI, user_message_json: str, config: Config) -> Tuple[str, float]:
    """Call GPT-4. Returns (generated_text, api_cost_usd)."""
    formatted = format_user_prompt(user_message_json)

    user_prompt = f"""Please analyze the following battery storage data and determine the optimal charging/discharging schedule.

{formatted}

Reason step by step:
1. Identify cheap vs expensive hours
2. Plan charge/discharge schedule respecting all constraints
3. Compute the SOC trajectory and total cost

End your response with:
SOC_TRAJECTORY: [s0, s1, ..., s24]
OBJECTIVE_COST: <number>"""

    try:
        response = client.chat.completions.create(
            model=config.model,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt},
            ],
            temperature=config.temperature,
            max_tokens=config.max_tokens,
        )

        text = response.choices[0].message.content
        inp_tok = response.usage.prompt_tokens
        out_tok = response.usage.completion_tokens

        if "gpt-4" in config.model and "mini" not in config.model:
            api_cost = (inp_tok * 2.5 + out_tok * 10) / 1_000_000
        elif "gpt-4-mini" in config.model:
            api_cost = (inp_tok * 0.15 + out_tok * 0.6) / 1_000_000
        else:
            api_cost = (inp_tok * 5 + out_tok * 15) / 1_000_000

        return text, api_cost

    except Exception as e:
        print(f"  API Error: {e}")
        return "", 0.0


# ============================================================================
# Main Loop
# ============================================================================

def run(config: Config):
    print("=" * 70)
    print("GPT-4 Storage Arbitrage → Hourly CSV")
    print("=" * 70)

    # --- API key ---
    # api_key = os.environ.get("OPENAI_API_KEY", "")
    # if not api_key:
    #     # Try Google Colab userdata
    #     try:
    #         from google.colab import userdata
    #         api_key = userdata.get("OPENAI_API_KEY")
    #     except Exception:
    #         pass
    # if not api_key:
    #     raise ValueError("Set OPENAI_API_KEY environment variable or Colab secret.")
    OPENAI_API_KEY = ""

    client = OpenAI(api_key=OPENAI_API_KEY)

    # --- Load test data ---
    test_data = []
    with open(config.test_file) as f:
        for line in f:
            test_data.append(json.loads(line))
    test_data = test_data[:config.n_days]
    print(f"Loaded {len(test_data)} days from {config.test_file}")

    # --- Prepare CSV ---
    csv_path = config.output_csv
    columns = [
        'prices_actual', 'prices_forecast', 'actual_demand', 'forecast_demand',
        'soc', 'charge_MW', 'discharge_MW', 'import_MW', 'export_MW', 'profit_step'
    ]

    # If resuming, read existing rows to know how many days done
    existing_rows = []
    if config.resume_from_day > 0 and os.path.exists(csv_path):
        with open(csv_path, 'r') as f:
            reader = csv.DictReader(f)
            existing_rows = list(reader)
        print(f"Resuming: loaded {len(existing_rows)} existing rows "
              f"({len(existing_rows)//24} days), starting from day {config.resume_from_day}")

    # Open file for writing
    mode = 'w'  # always rewrite (we include existing_rows)
    with open(csv_path, mode, newline='') as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=columns)
        writer.writeheader()

        # Write back existing rows if resuming
        for row in existing_rows:
            writer.writerow(row)

        total_api_cost = 0.0
        failed_days = 0

        for i, ex in enumerate(test_data):
            if i < config.resume_from_day:
                continue

            date = ex.get("date", f"day_{i}")
            user_msg = ex.get("user_message", "")
            user_data = json.loads(user_msg) if isinstance(user_msg, str) else user_msg

            # Battery params
            batt_raw = ex.get("battery", {})
            battery = {
                'capacity_kwh': batt_raw.get('capacity_kwh', user_data.get('capacity_kwh', 49.44)),
                'cmax_kw': batt_raw.get('cmax_kw', user_data.get('max_c_kw', 12.36)),
                'dmax_kw': batt_raw.get('dmax_kw', user_data.get('max_d_kw', 12.36)),
                'eta_c': batt_raw.get('eta_c', user_data.get('eta_c', 0.95)),
                'eta_d': batt_raw.get('eta_d', user_data.get('eta_d', 0.95)),
                'soc_init': user_data.get('soc0', 0.5),
            }
            allow_export = user_data.get('allow_export', True)

            forecast_prices = user_data.get('forecast_prices_today', [])
            forecast_demand = user_data.get('forecast_demand_today', [])
            actual_prices = ex.get("actual_prices_today", forecast_prices)
            actual_demand = ex.get("actual_demand_today", forecast_demand)

            # Call GPT-4o
            start_t = time.time()
            raw_text, api_cost = call_openai(client, user_msg, config)
            elapsed = time.time() - start_t
            total_api_cost += api_cost

            # Parse SOC trajectory
            pred_soc, pred_cost = parse_soc_and_cost(raw_text)

            if pred_soc and len(pred_soc) == 25:
                # Derive hourly operational data
                hourly_rows = derive_hourly_data(
                    soc_trajectory=pred_soc,
                    battery=battery,
                    actual_prices=actual_prices,
                    forecast_prices=forecast_prices,
                    actual_demand=actual_demand,
                    forecast_demand=forecast_demand,
                    allow_export=allow_export,
                )
                for row in hourly_rows:
                    writer.writerow(row)
                csvfile.flush()

                print(f"Day {i:3d} ({date}) ✓  24 rows written  "
                      f"API ${api_cost:.3f}  {elapsed:.1f}s")
            else:
                # Failed parse — write 24 rows of zeros / raw data so alignment is preserved
                failed_days += 1
                n_vals = len(pred_soc) if pred_soc else 0
                print(f"Day {i:3d} ({date}) ✗  SOC parse failed ({n_vals} values)  "
                      f"API ${api_cost:.3f}  {elapsed:.1f}s")
                # Still write rows with NaN for battery columns to keep 24-row-per-day alignment
                for t in range(24):
                    row = {
                        'prices_actual': round(actual_prices[t], 2) if t < len(actual_prices) else 0,
                        'prices_forecast': round(forecast_prices[t], 2) if t < len(forecast_prices) else 0,
                        'actual_demand': round(actual_demand[t], 2) if t < len(actual_demand) else 0,
                        'forecast_demand': round(forecast_demand[t], 2) if t < len(forecast_demand) else 0,
                        'soc': '',
                        'charge_MW': '',
                        'discharge_MW': '',
                        'import_MW': '',
                        'export_MW': '',
                        'profit_step': '',
                    }
                    writer.writerow(row)
                csvfile.flush()

            time.sleep(config.delay_between_calls)

    print("\n" + "=" * 70)
    print(f"Done! {len(test_data) - config.resume_from_day} days processed.")
    print(f"Failed parses: {failed_days}")
    print(f"Total API cost: ${total_api_cost:.4f}")
    print(f"Output: {csv_path}")
    print("=" * 70)


# ============================================================================
# Entry Point
# ============================================================================

if __name__ == "__main__":
    config = Config()

    # --- Override paths here or via env vars ---
    # config.test_file = "/path/to/test_qwen_prevday_ITALY_2.jsonl"
    # config.output_csv = "gpt4_hourly_arbitrage.csv"
    # config.resume_from_day = 0

    run(config)

Mounted at /content/drive
GPT-4 Storage Arbitrage → Hourly CSV
Loaded 364 days from /content/drive/MyDrive/energy_finetuning_35/test_qwen_prevday_ITALY_2.jsonl
Resuming: loaded 13584 existing rows (566 days), starting from day 312
Day 312 (2019-11-10) ✓  24 rows written  API $0.010  21.0s
Day 313 (2019-11-11) ✓  24 rows written  API $0.010  17.5s
Day 314 (2019-11-12) ✓  24 rows written  API $0.012  30.2s
Day 315 (2019-11-13) ✓  24 rows written  API $0.011  16.1s
Day 316 (2019-11-14) ✓  24 rows written  API $0.011  18.7s
Day 317 (2019-11-15) ✓  24 rows written  API $0.010  14.8s
Day 318 (2019-11-16) ✓  24 rows written  API $0.010  16.5s
Day 319 (2019-11-17) ✓  24 rows written  API $0.013  29.7s
Day 320 (2019-11-18) ✓  24 rows written  API $0.010  10.0s
Day 321 (2019-11-19) ✓  24 rows written  API $0.011  15.5s
Day 322 (2019-11-20) ✓  24 rows written  API $0.010  14.7s
Day 323 (2019-11-21) ✓  24 rows written  API $0.010  13.3s
Day 324 (2019-11-22) ✓  24 rows written  API $0.011  20.8s
Da

In [ ]:
# -*- coding: utf-8 -*-
"""
GPT-4o Storage Arbitrage — Violation Check & LP Correction
Reads   : gpt4_hourly_arbitrage_cleaned.csv  (24 rows/day × 365 days)
Writes  : gpt4_hourly_arbitrage_corrected.csv (same format, all feasible)

For every day:
  1. Reconstruct SOC trajectory from the CSV rows
  2. Check all battery constraints (SOC bounds, power limits, dynamics)
  3. If ANY violation → re-solve an LP to get the optimal feasible schedule
  4. Recompute charge/discharge/import/export/profit_step from LP solution
  5. If feasible already → keep original rows untouched

Uses scipy.optimize.linprog (free, pre-installed in Colab).
"""

import csv
import math
import numpy as np
from scipy.optimize import linprog
from dataclasses import dataclass, field
from typing import List, Tuple, Optional, Dict


# ============================================================================
# Battery Configuration  (match your JSONL defaults — edit if needed)
# ============================================================================

@dataclass
class BatteryConfig:
    capacity_kwh: float = 49.44
    cmax_kw: float = 12.36       # max charge power
    dmax_kw: float = 12.36       # max discharge power
    eta_c: float = 0.95          # charge efficiency
    eta_d: float = 0.95          # discharge efficiency
    soc_min: float = 0.0
    soc_max: float = 1.0
    soc_target: float = 0.0      # minimum final SOC (end of day)
    dt: float = 1.0              # time-step in hours
    allow_export: bool = True


# ============================================================================
# File Paths  (edit to match your setup)
# ============================================================================
from google.colab import drive, userdata

drive.mount('/content/drive')


INPUT_CSV  = "/content/drive/MyDrive/energy_finetuning_35/gpt4_hourly_arbitrage_cleaned.csv"
OUTPUT_CSV = "/content/drive/MyDrive/energy_finetuning_35/gpt4_hourly_arbitrage_corrected.csv"

# Optional: if you also have the original JSONL, set this to read per-day
# battery params & soc0. Otherwise we infer soc0 from the CSV's soc column
# and use the defaults above.
JSONL_FILE = None   # e.g. "test_qwen_prevday_ITALY_1.jsonl"


# ============================================================================
# 1.  Read CSV into day-groups (list of 365 lists, each with 24 row-dicts)
# ============================================================================

def read_csv_days(path: str) -> List[List[Dict]]:
    """Read CSV and group into days of 24 rows each."""
    with open(path, "r") as f:
        reader = csv.DictReader(f)
        all_rows = list(reader)

    days = []
    for start in range(0, len(all_rows), 24):
        chunk = all_rows[start : start + 24]
        if len(chunk) == 24:
            days.append(chunk)
        else:
            print(f"  Warning: incomplete day at row {start} ({len(chunk)} rows), skipping")
    return days


# ============================================================================
# 2.  (Optional) Load per-day battery params from JSONL
# ============================================================================

def load_jsonl_days(path: str) -> List[Dict]:
    """Return list of parsed JSONL records (one per day)."""
    import json
    records = []
    with open(path) as f:
        for line in f:
            records.append(json.loads(line))
    return records


def battery_from_jsonl(rec: Dict) -> Tuple[BatteryConfig, float]:
    """Extract BatteryConfig and soc0 from a JSONL record."""
    import json
    ud = rec.get("user_message", {})
    if isinstance(ud, str):
        ud = json.loads(ud)
    batt_raw = rec.get("battery", {})

    cfg = BatteryConfig(
        capacity_kwh = batt_raw.get("capacity_kwh", ud.get("capacity_kwh", 49.44)),
        cmax_kw      = batt_raw.get("cmax_kw",      ud.get("max_c_kw",    12.36)),
        dmax_kw      = batt_raw.get("dmax_kw",      ud.get("max_d_kw",    12.36)),
        eta_c        = batt_raw.get("eta_c",         ud.get("eta_c",       0.95)),
        eta_d        = batt_raw.get("eta_d",         ud.get("eta_d",       0.95)),
        soc_min      = ud.get("soc_min", 0.0),
        soc_max      = ud.get("soc_max", 1.0),
        soc_target   = ud.get("soc_target", 0.0),
        allow_export = ud.get("allow_export", True),
    )
    soc0 = ud.get("soc0", 0.5)
    return cfg, soc0


# ============================================================================
# 3.  Reconstruct SOC trajectory & check feasibility
# ============================================================================

def reconstruct_and_check(
    rows: List[Dict],
    batt: BatteryConfig,
    soc0: float,
    tol: float = 1e-3,
) -> Tuple[bool, List[str], List[float]]:
    """
    From 24 CSV rows, reconstruct the 25-point SOC trajectory and check
    all battery constraints.
    Returns (is_feasible, list_of_violations, soc_trajectory_25).
    """
    T = 24
    C = batt.capacity_kwh
    eta_c = batt.eta_c
    eta_d = batt.eta_d
    dt = batt.dt
    violations = []

    # Read SOC values from CSV  (hours 0..23)
    soc_csv = []
    for r in rows:
        try:
            soc_csv.append(float(r['soc']))
        except (ValueError, KeyError):
            soc_csv.append(None)

    # Read charge / discharge
    charge = []
    discharge = []
    for r in rows:
        try:
            charge.append(float(r['charge_MW']))
        except (ValueError, KeyError):
            charge.append(0.0)
        try:
            discharge.append(float(r['discharge_MW']))
        except (ValueError, KeyError):
            discharge.append(0.0)

    # Build 25-point SOC trajectory from dynamics
    soc = [soc0]
    for t in range(T):
        c_t = charge[t]
        d_t = discharge[t]
        next_soc = soc[t] + (eta_c * c_t * dt - d_t * dt / eta_d) / C
        soc.append(next_soc)

    # --- Violation checks ---

    # a) SOC bounds
    for t in range(T + 1):
        if soc[t] < batt.soc_min - tol:
            violations.append(f"soc[{t}]={soc[t]:.6f} < soc_min={batt.soc_min}")
        if soc[t] > batt.soc_max + tol:
            violations.append(f"soc[{t}]={soc[t]:.6f} > soc_max={batt.soc_max}")

    # b) Initial SOC
    if abs(soc[0] - soc0) > tol:
        violations.append(f"soc[0]={soc[0]:.6f} != soc0={soc0}")

    # c) Final SOC target
    if soc[T] < batt.soc_target - tol:
        violations.append(f"soc[24]={soc[T]:.6f} < soc_target={batt.soc_target}")

    # d) Power limits
    for t in range(T):
        if charge[t] > batt.cmax_kw + tol:
            violations.append(f"t={t}: charge={charge[t]:.4f} > cmax={batt.cmax_kw}")
        if charge[t] < -tol:
            violations.append(f"t={t}: charge={charge[t]:.4f} < 0")
        if discharge[t] > batt.dmax_kw + tol:
            violations.append(f"t={t}: discharge={discharge[t]:.4f} > dmax={batt.dmax_kw}")
        if discharge[t] < -tol:
            violations.append(f"t={t}: discharge={discharge[t]:.4f} < 0")

    # e) Simultaneous charge & discharge
    for t in range(T):
        if charge[t] > tol and discharge[t] > tol:
            violations.append(
                f"t={t}: simultaneous c={charge[t]:.4f} & d={discharge[t]:.4f}"
            )

    # f) Export when not allowed
    if not batt.allow_export:
        for t in range(T):
            demand_t = float(rows[t].get('actual_demand', 0))
            net = demand_t + charge[t] - discharge[t]
            if net < -tol:
                violations.append(f"t={t}: net={net:.4f} < 0, export not allowed")

    return (len(violations) == 0), violations, soc


# ============================================================================
# 4.  LP Solver  (scipy.optimize.linprog — free, no install needed)
# ============================================================================

def solve_lp(
    batt: BatteryConfig,
    soc0: float,
    prices: List[float],
    demand: List[float],
) -> Optional[Dict]:
    """
    Solve the battery arbitrage LP for one day (24 hours).

    Variables (indexed in the decision vector x):
      c[0..23]    charge power          (indices  0..23)
      d[0..23]    discharge power        (indices 24..47)
      imp[0..23]  grid import            (indices 48..71)
      exp[0..23]  grid export            (indices 72..95)
      soc[0..24]  state of charge        (indices 96..120)
    Total: 121 variables

    Returns dict with keys: c, d, imp, exp, soc (lists), or None if infeasible.
    """
    T = 24
    C = batt.capacity_kwh
    eta_c = batt.eta_c
    eta_d = batt.eta_d
    dt = batt.dt
    n_vars = 4 * T + (T + 1)  # 121

    # --- Variable index helpers ---
    def ic(t):   return t            # charge
    def id_(t):  return T + t        # discharge
    def ii(t):   return 2*T + t      # import
    def ie(t):   return 3*T + t      # export
    def isoc(t): return 4*T + t      # soc

    # ------------------------------------------------------------------
    # Objective: minimise  sum_t [ price[t] * imp[t] - price[t] * exp[t] ]
    # ------------------------------------------------------------------
    obj = np.zeros(n_vars)
    for t in range(T):
        obj[ii(t)] =  prices[t] * dt   # cost of import
        obj[ie(t)] = -prices[t] * dt   # revenue from export

    # ------------------------------------------------------------------
    # Bounds
    # ------------------------------------------------------------------
    bounds = []
    for t in range(T):  # c
        bounds.append((0.0, batt.cmax_kw))
    for t in range(T):  # d
        bounds.append((0.0, batt.dmax_kw))
    for t in range(T):  # imp
        bounds.append((0.0, None))
    for t in range(T):  # exp
        if batt.allow_export:
            bounds.append((0.0, None))
        else:
            bounds.append((0.0, 0.0))  # no export
    for t in range(T + 1):  # soc
        bounds.append((batt.soc_min, batt.soc_max))

    # ------------------------------------------------------------------
    # Equality constraints:  A_eq @ x = b_eq
    # ------------------------------------------------------------------
    # 1) SOC dynamics (24 rows):
    #    soc[t+1] - soc[t] - (eta_c/C)*c[t] + (1/(eta_d*C))*d[t] = 0
    #
    # 2) Grid balance (24 rows):
    #    imp[t] - exp[t] - demand[t] - c[t] + d[t] = 0
    #    i.e.  imp[t] - exp[t] = demand[t] + c[t] - d[t]
    #
    # 3) Initial SOC (1 row):
    #    soc[0] = soc0

    n_eq = T + T + 1  # 49
    A_eq = np.zeros((n_eq, n_vars))
    b_eq = np.zeros(n_eq)

    row = 0

    # SOC dynamics
    for t in range(T):
        A_eq[row, isoc(t+1)] =  1.0
        A_eq[row, isoc(t)]   = -1.0
        A_eq[row, ic(t)]     = -(eta_c * dt) / C
        A_eq[row, id_(t)]    =  (dt / (eta_d * C))
        b_eq[row] = 0.0
        row += 1

    # Grid balance
    for t in range(T):
        A_eq[row, ii(t)]  =  1.0
        A_eq[row, ie(t)]  = -1.0
        A_eq[row, ic(t)]  = -1.0
        A_eq[row, id_(t)] =  1.0
        b_eq[row] = demand[t]
        row += 1

    # Initial SOC
    A_eq[row, isoc(0)] = 1.0
    b_eq[row] = soc0
    row += 1

    # ------------------------------------------------------------------
    # Inequality constraints:  A_ub @ x <= b_ub
    # ------------------------------------------------------------------
    # Final SOC target:  soc[24] >= soc_target  →  -soc[24] <= -soc_target
    A_ub = np.zeros((1, n_vars))
    b_ub = np.zeros(1)
    A_ub[0, isoc(T)] = -1.0
    b_ub[0] = -batt.soc_target

    # ------------------------------------------------------------------
    # Solve
    # ------------------------------------------------------------------
    result = linprog(
        c=obj,
        A_ub=A_ub,
        b_ub=b_ub,
        A_eq=A_eq,
        b_eq=b_eq,
        bounds=bounds,
        method='highs',
        options={'presolve': True},
    )

    if not result.success:
        return None

    x = result.x
    sol = {
        'c':   [x[ic(t)]   for t in range(T)],
        'd':   [x[id_(t)]  for t in range(T)],
        'imp': [x[ii(t)]   for t in range(T)],
        'exp': [x[ie(t)]   for t in range(T)],
        'soc': [x[isoc(t)] for t in range(T + 1)],
        'obj': result.fun,
    }
    return sol


# ============================================================================
# 5.  Build corrected 24-row block from LP solution
# ============================================================================

def build_corrected_rows(
    sol: Dict,
    actual_prices: List[float],
    forecast_prices: List[float],
    actual_demand: List[float],
    forecast_demand: List[float],
) -> List[Dict]:
    """Turn an LP solution into 24 CSV row-dicts in the target format."""
    rows = []
    dt = 1.0
    for t in range(24):
        c_t   = sol['c'][t]
        d_t   = sol['d'][t]
        imp_t = sol['imp'][t]
        exp_t = sol['exp'][t]
        soc_t = sol['soc'][t]

        # profit_step = savings vs no-battery baseline (using actual prices)
        baseline_cost = actual_prices[t] * actual_demand[t] * dt
        actual_cost   = actual_prices[t] * imp_t * dt - actual_prices[t] * exp_t * dt
        profit_step   = baseline_cost - actual_cost

        rows.append({
            'prices_actual':   round(actual_prices[t], 2),
            'prices_forecast': round(forecast_prices[t], 2),
            'actual_demand':   round(actual_demand[t], 2),
            'forecast_demand': round(forecast_demand[t], 2),
            'soc':             round(soc_t, 4),
            'charge_MW':       round(c_t, 2),
            'discharge_MW':    round(d_t, 2),
            'import_MW':       round(imp_t, 2),
            'export_MW':       round(exp_t, 2),
            'profit_step':     round(profit_step, 2),
        })
    return rows


# ============================================================================
# 6.  Main pipeline
# ============================================================================

def main():
    print("=" * 70)
    print("Storage Arbitrage — Violation Check & LP Correction")
    print("=" * 70)

    # --- Read cleaned CSV ---
    days = read_csv_days(INPUT_CSV)
    n_days = len(days)
    print(f"Read {n_days} days ({n_days * 24} rows) from {INPUT_CSV}")

    # --- Optionally load JSONL for per-day battery params ---
    jsonl_records = None
    if JSONL_FILE:
        jsonl_records = load_jsonl_days(JSONL_FILE)
        print(f"Loaded {len(jsonl_records)} JSONL records from {JSONL_FILE}")

    # --- Default battery ---
    default_batt = BatteryConfig()
    default_soc0 = 0.5

    # --- Process each day ---
    columns = [
        'prices_actual', 'prices_forecast', 'actual_demand', 'forecast_demand',
        'soc', 'charge_MW', 'discharge_MW', 'import_MW', 'export_MW', 'profit_step'
    ]

    total_feasible  = 0
    total_violated  = 0
    total_corrected = 0
    total_failed    = 0

    corrected_days = []   # list of 24-row blocks

    for day_idx, day_rows in enumerate(days):
        # --- Get battery config & soc0 for this day ---
        if jsonl_records and day_idx < len(jsonl_records):
            batt, soc0 = battery_from_jsonl(jsonl_records[day_idx])
        else:
            batt = default_batt
            # Infer soc0 from CSV's first-hour SOC if available
            try:
                soc0 = float(day_rows[0]['soc'])
            except (ValueError, KeyError):
                soc0 = default_soc0

        # --- Extract price/demand vectors from CSV ---
        actual_prices   = [float(r['prices_actual'])   for r in day_rows]
        forecast_prices = [float(r['prices_forecast']) for r in day_rows]
        actual_demand   = [float(r['actual_demand'])   for r in day_rows]
        forecast_demand = [float(r['forecast_demand']) for r in day_rows]

        # --- Step 1: Check feasibility ---
        feasible, violations, soc_traj = reconstruct_and_check(
            day_rows, batt, soc0
        )

        if feasible:
            # Keep original rows untouched
            total_feasible += 1
            corrected_days.append(day_rows)
            if day_idx < 5 or day_idx % 50 == 0:
                print(f"Day {day_idx:3d}  ✓ feasible (kept original)")
        else:
            total_violated += 1
            n_v = len(violations)

            # --- Step 2: Re-solve LP using FORECAST prices (decision basis) ---
            sol = solve_lp(batt, soc0, forecast_prices, forecast_demand)

            if sol is not None:
                total_corrected += 1

                # Build new rows using LP solution + actual prices for profit
                new_rows = build_corrected_rows(
                    sol, actual_prices, forecast_prices,
                    actual_demand, forecast_demand,
                )
                corrected_days.append(new_rows)
                print(f"Day {day_idx:3d}  ✗ {n_v} violation(s) → LP corrected  "
                      f"(obj={sol['obj']:.2f})")
                if day_idx < 10:
                    for v in violations[:3]:
                        print(f"           {v}")
            else:
                total_failed += 1
                # LP infeasible — keep original rows as-is
                corrected_days.append(day_rows)
                print(f"Day {day_idx:3d}  ✗ {n_v} violation(s) → LP INFEASIBLE "
                      f"(kept original)")

    # --- Write corrected CSV ---
    with open(OUTPUT_CSV, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=columns)
        writer.writeheader()
        for day_block in corrected_days:
            for row in day_block:
                # Ensure only target columns are written (drop any extras)
                out = {col: row.get(col, '') for col in columns}
                writer.writerow(out)

    total_rows = sum(len(d) for d in corrected_days)
    print()
    print("=" * 70)
    print(f"Results written to {OUTPUT_CSV}  ({total_rows} rows)")
    print(f"  Already feasible : {total_feasible:4d} days")
    print(f"  Violated → fixed : {total_corrected:4d} days")
    print(f"  Violated → failed: {total_failed:4d} days (LP infeasible, kept original)")
    print(f"  Total violated   : {total_violated:4d} days")
    print("=" * 70)


if __name__ == "__main__":
    main()

Mounted at /content/drive
Storage Arbitrage — Violation Check & LP Correction
Read 364 days (8736 rows) from /content/drive/MyDrive/energy_finetuning_35/gpt4_hourly_arbitrage_cleaned.csv
Day   0  ✗ 6 violation(s) → LP corrected  (obj=41178.34)
           t=3: charge=13.0100 > cmax=12.36
           t=4: charge=13.0100 > cmax=12.36
           t=5: charge=13.0100 > cmax=12.36
Day   1  ✓ feasible (kept original)
Day   2  ✗ 4 violation(s) → LP corrected  (obj=21523.51)
           t=11: charge=12.4900 > cmax=12.36
           t=12: charge=12.4900 > cmax=12.36
           t=13: charge=12.4900 > cmax=12.36
Day   3  ✓ feasible (kept original)
Day   4  ✗ 6 violation(s) → LP corrected  (obj=44014.04)
           t=4: charge=13.0100 > cmax=12.36
           t=5: charge=13.0100 > cmax=12.36
           t=6: charge=13.0100 > cmax=12.36
Day  10  ✗ 6 violation(s) → LP corrected  (obj=33242.78)
Day  11  ✗ 3 violation(s) → LP corrected  (obj=34184.54)
Day  12  ✗ 24 violation(s) → LP corrected  (obj=41908.57)

In [ ]:
# -*- coding: utf-8 -*-
"""
GPT-4o Storage Arbitrage — EXHAUSTIVE Violation Check & LP Correction
Reads   : gpt4_hourly_arbitrage_cleaned.csv  (24 rows/day × 365 days)
Writes  : gpt4_hourly_arbitrage_corrected.csv (same format, all feasible)

Checks performed (per day):
  [C1]  SOC bounds               soc_min <= soc[t] <= soc_max  ∀t=0..24
  [C2]  Initial SOC              soc[0] == soc0
  [C3]  Final SOC target         soc[24] >= soc_target
  [C4]  Charge power limits      0 <= c[t] <= cmax             ∀t=0..23
  [C5]  Discharge power limits   0 <= d[t] <= dmax             ∀t=0..23
  [C6]  No simultaneous c & d    not (c[t]>0 and d[t]>0)       ∀t=0..23
  [C7]  SOC dynamics consistency soc[t+1]=soc[t]+(η_c·c·dt−d·dt/η_d)/C
  [C8]  Grid balance             imp[t]−exp[t] = demand[t]+c[t]−d[t]
  [C9]  Non-negative import      imp[t] >= 0                   ∀t=0..23
  [C10] Non-negative export      exp[t] >= 0                   ∀t=0..23
  [C11] Export restriction       exp[t] == 0 if allow_export=False
  [C12] CSV SOC column match     CSV soc[t] ≈ dynamics soc[t]
  [C13] 4h energy throughput     rolling Σ charge/discharge ≤ capacity
  [C14] Charge headroom          c[t]·dt·η_c ≤ (soc_max−soc[t])·C
  [C15] Discharge available      d[t]·dt/η_d ≤ (soc[t]−soc_min)·C
  [C16] NaN / missing data       any NaN or blank in decision columns
  [C17] Profit consistency       profit_step ≈ price·(demand−imp+exp)

Uses scipy.optimize.linprog (HiGHS, free in Colab).
LP guarantees feasibility; post-LP verification confirms all constraints.
"""

import csv
import math
import numpy as np
from scipy.optimize import linprog
from dataclasses import dataclass
from typing import List, Tuple, Optional, Dict


# ============================================================================
# Battery Configuration  (match your JSONL defaults — edit if needed)
# ============================================================================

@dataclass
class BatteryConfig:
    capacity_kwh: float = 49.44
    cmax_kw: float = 12.36       # max charge power
    dmax_kw: float = 12.36       # max discharge power
    eta_c: float = 0.95          # charge efficiency
    eta_d: float = 0.95          # discharge efficiency
    soc_min: float = 0.0
    soc_max: float = 1.0
    soc_target: float = 0.0      # minimum final SOC (end of day)
    dt: float = 1.0              # time-step in hours
    allow_export: bool = True


# ============================================================================
# File Paths  (edit to match your setup)
# ============================================================================

INPUT_CSV  = "/content/drive/MyDrive/energy_finetuning_35/gpt4_hourly_arbitrage_cleaned.csv"
OUTPUT_CSV = "/content/drive/MyDrive/energy_finetuning_35/gpt4_hourly_arbitrage_corrected2.csv"

# If you have the original JSONL, set this to read per-day battery params & soc0.
# Otherwise we infer soc0 from the CSV and use the BatteryConfig defaults.
JSONL_FILE = None   # e.g. "test_qwen_prevday_ITALY_1.jsonl"

# Rolling-window size (hours) for energy throughput check.
# 4h matches the 49.44 kWh / 12.36 kW ≈ 4h storage duration.
ROLLING_WINDOW_H = 4


# ============================================================================
# 1.  Read CSV into day-groups
# ============================================================================

def read_csv_days(path: str) -> List[List[Dict]]:
    """Read CSV and group into days of 24 rows each."""
    with open(path, "r") as f:
        reader = csv.DictReader(f)
        all_rows = list(reader)

    days = []
    for start in range(0, len(all_rows), 24):
        chunk = all_rows[start : start + 24]
        if len(chunk) == 24:
            days.append(chunk)
        else:
            print(f"  Warning: incomplete day at row {start} ({len(chunk)} rows), skipping")
    return days


# ============================================================================
# 2.  (Optional) Load per-day battery params from JSONL
# ============================================================================

def load_jsonl_days(path: str) -> List[Dict]:
    import json
    records = []
    with open(path) as f:
        for line in f:
            records.append(json.loads(line))
    return records


def battery_from_jsonl(rec: Dict) -> Tuple[BatteryConfig, float]:
    import json
    ud = rec.get("user_message", {})
    if isinstance(ud, str):
        ud = json.loads(ud)
    batt_raw = rec.get("battery", {})

    cfg = BatteryConfig(
        capacity_kwh = batt_raw.get("capacity_kwh", ud.get("capacity_kwh", 49.44)),
        cmax_kw      = batt_raw.get("cmax_kw",      ud.get("max_c_kw",    12.36)),
        dmax_kw      = batt_raw.get("dmax_kw",      ud.get("max_d_kw",    12.36)),
        eta_c        = batt_raw.get("eta_c",         ud.get("eta_c",       0.95)),
        eta_d        = batt_raw.get("eta_d",         ud.get("eta_d",       0.95)),
        soc_min      = ud.get("soc_min", 0.0),
        soc_max      = ud.get("soc_max", 1.0),
        soc_target   = ud.get("soc_target", 0.0),
        allow_export = ud.get("allow_export", True),
    )
    soc0 = ud.get("soc0", 0.5)
    return cfg, soc0


# ============================================================================
# 3.  Safe float parser
# ============================================================================

def safe_float(val, default=0.0) -> Tuple[float, bool]:
    """Return (value, is_valid). Flags NaN, blank, non-numeric."""
    if val is None or val == '':
        return default, False
    try:
        v = float(val)
        if math.isnan(v) or math.isinf(v):
            return default, False
        return v, True
    except (ValueError, TypeError):
        return default, False


# ============================================================================
# 4.  EXHAUSTIVE Feasibility Checker  (17 constraint families)
# ============================================================================

def exhaustive_check(
    rows: List[Dict],
    batt: BatteryConfig,
    soc0: float,
    tol: float = 1e-3,
    rolling_window: int = ROLLING_WINDOW_H,
) -> Tuple[bool, List[str]]:
    """
    Perform ALL constraint checks on 24 CSV rows.
    Returns (is_feasible, list_of_violations).
    """
    T = 24
    C = batt.capacity_kwh
    eta_c = batt.eta_c
    eta_d = batt.eta_d
    dt = batt.dt
    violations = []

    # ---- Parse all columns ----
    charge     = []
    discharge  = []
    soc_csv    = []
    imp_csv    = []
    exp_csv    = []
    demand_a   = []
    prices_a   = []
    profit_csv = []

    for t, r in enumerate(rows):
        c_val, c_ok  = safe_float(r.get('charge_MW'))
        d_val, d_ok  = safe_float(r.get('discharge_MW'))
        s_val, s_ok  = safe_float(r.get('soc'))
        i_val, i_ok  = safe_float(r.get('import_MW'))
        e_val, e_ok  = safe_float(r.get('export_MW'))
        da_val, _    = safe_float(r.get('actual_demand'))
        pa_val, _    = safe_float(r.get('prices_actual'))
        pr_val, pr_ok = safe_float(r.get('profit_step'))

        # [C16] NaN / missing data in decision columns
        if not c_ok:
            violations.append(f"[C16] t={t}: charge_MW is NaN/blank/invalid")
        if not d_ok:
            violations.append(f"[C16] t={t}: discharge_MW is NaN/blank/invalid")
        if not s_ok:
            violations.append(f"[C16] t={t}: soc is NaN/blank/invalid")
        if not i_ok:
            violations.append(f"[C16] t={t}: import_MW is NaN/blank/invalid")
        if not e_ok:
            violations.append(f"[C16] t={t}: export_MW is NaN/blank/invalid")

        charge.append(c_val)
        discharge.append(d_val)
        soc_csv.append(s_val)
        imp_csv.append(i_val)
        exp_csv.append(e_val)
        demand_a.append(da_val)
        prices_a.append(pa_val)
        profit_csv.append(pr_val)

    # ---- Reconstruct 25-point SOC from dynamics ----
    soc_dyn = [soc0]
    for t in range(T):
        next_soc = soc_dyn[t] + (eta_c * charge[t] * dt - discharge[t] * dt / eta_d) / C
        soc_dyn.append(next_soc)

    # [C1] SOC bounds (on dynamics-reconstructed trajectory)
    for t in range(T + 1):
        if soc_dyn[t] < batt.soc_min - tol:
            violations.append(f"[C1] soc_dyn[{t}]={soc_dyn[t]:.6f} < soc_min={batt.soc_min}")
        if soc_dyn[t] > batt.soc_max + tol:
            violations.append(f"[C1] soc_dyn[{t}]={soc_dyn[t]:.6f} > soc_max={batt.soc_max}")

    # [C2] Initial SOC
    if abs(soc_dyn[0] - soc0) > tol:
        violations.append(f"[C2] soc_dyn[0]={soc_dyn[0]:.6f} != soc0={soc0}")

    # [C3] Final SOC target
    if soc_dyn[T] < batt.soc_target - tol:
        violations.append(f"[C3] soc_dyn[24]={soc_dyn[T]:.6f} < soc_target={batt.soc_target}")

    # [C4] Charge power limits
    for t in range(T):
        if charge[t] < -tol:
            violations.append(f"[C4] t={t}: charge={charge[t]:.4f} < 0")
        if charge[t] > batt.cmax_kw + tol:
            violations.append(f"[C4] t={t}: charge={charge[t]:.4f} > cmax={batt.cmax_kw}")

    # [C5] Discharge power limits
    for t in range(T):
        if discharge[t] < -tol:
            violations.append(f"[C5] t={t}: discharge={discharge[t]:.4f} < 0")
        if discharge[t] > batt.dmax_kw + tol:
            violations.append(f"[C5] t={t}: discharge={discharge[t]:.4f} > dmax={batt.dmax_kw}")

    # [C6] No simultaneous charge & discharge
    for t in range(T):
        if charge[t] > tol and discharge[t] > tol:
            violations.append(
                f"[C6] t={t}: simultaneous c={charge[t]:.4f} & d={discharge[t]:.4f}"
            )

    # [C7] SOC dynamics consistency (CSV soc column vs itself step-by-step)
    for t in range(T - 1):
        expected = soc_csv[t] + (eta_c * charge[t] * dt - discharge[t] * dt / eta_d) / C
        if abs(soc_csv[t + 1] - expected) > tol:
            violations.append(
                f"[C7] t={t}: CSV soc[{t+1}]={soc_csv[t+1]:.6f} != "
                f"dynamics({soc_csv[t]:.4f}+Δ)={expected:.6f}"
            )

    # [C8] Grid balance: imp − exp == demand + c − d
    for t in range(T):
        net = demand_a[t] + charge[t] - discharge[t]
        balance = imp_csv[t] - exp_csv[t]
        if abs(balance - net) > tol:
            violations.append(
                f"[C8] t={t}: imp−exp={balance:.4f} != demand+c−d={net:.4f}  "
                f"(diff={abs(balance - net):.6f})"
            )

    # [C9] Non-negative import
    for t in range(T):
        if imp_csv[t] < -tol:
            violations.append(f"[C9] t={t}: import={imp_csv[t]:.4f} < 0")

    # [C10] Non-negative export
    for t in range(T):
        if exp_csv[t] < -tol:
            violations.append(f"[C10] t={t}: export={exp_csv[t]:.4f} < 0")

    # [C11] Export restriction
    if not batt.allow_export:
        for t in range(T):
            if exp_csv[t] > tol:
                violations.append(f"[C11] t={t}: export={exp_csv[t]:.4f} > 0, not allowed")

    # [C12] CSV SOC column matches dynamics SOC
    for t in range(T):
        if abs(soc_csv[t] - soc_dyn[t]) > tol:
            violations.append(
                f"[C12] t={t}: CSV soc={soc_csv[t]:.6f} != dynamics soc={soc_dyn[t]:.6f}"
            )

    # [C13] Rolling-window energy throughput
    charge_energy = [charge[t] * dt * eta_c for t in range(T)]
    discharge_energy = [discharge[t] * dt / eta_d for t in range(T)]

    for start in range(T - rolling_window + 1):
        wc = sum(charge_energy[start:start + rolling_window])
        wd = sum(discharge_energy[start:start + rolling_window])
        if wc > C + tol:
            violations.append(
                f"[C13] h{start}-{start+rolling_window-1}: "
                f"charge energy={wc:.4f} kWh > capacity={C}"
            )
        if wd > C + tol:
            violations.append(
                f"[C13] h{start}-{start+rolling_window-1}: "
                f"discharge energy={wd:.4f} kWh > capacity={C}"
            )

    # [C14] Charge headroom per step
    for t in range(T):
        headroom = (batt.soc_max - soc_dyn[t]) * C
        energy_in = charge[t] * dt * eta_c
        if energy_in > headroom + tol:
            violations.append(
                f"[C14] t={t}: charge energy={energy_in:.4f} > "
                f"headroom={headroom:.4f} (soc={soc_dyn[t]:.4f})"
            )

    # [C15] Discharge available energy per step
    for t in range(T):
        available = (soc_dyn[t] - batt.soc_min) * C
        energy_out = discharge[t] * dt / eta_d
        if energy_out > available + tol:
            violations.append(
                f"[C15] t={t}: discharge energy={energy_out:.4f} > "
                f"available={available:.4f} (soc={soc_dyn[t]:.4f})"
            )

    # [C17] Profit consistency
    for t in range(T):
        baseline = prices_a[t] * demand_a[t] * dt
        actual_cost = prices_a[t] * imp_csv[t] * dt - prices_a[t] * exp_csv[t] * dt
        expected_profit = baseline - actual_cost
        threshold = max(0.1, abs(expected_profit) * 0.01)
        if abs(profit_csv[t] - expected_profit) > threshold:
            violations.append(
                f"[C17] t={t}: profit_step={profit_csv[t]:.2f} != "
                f"expected={expected_profit:.2f}"
            )

    return (len(violations) == 0), violations


# ============================================================================
# 5.  LP Solver  (with rolling-window energy constraints)
# ============================================================================

def solve_lp(
    batt: BatteryConfig,
    soc0: float,
    prices: List[float],
    demand: List[float],
) -> Optional[Dict]:
    """
    Solve the battery arbitrage LP for one day.

    Variables (in decision vector x):
      c[0..23]    charge power          (idx  0..23)
      d[0..23]    discharge power        (idx 24..47)
      imp[0..23]  grid import            (idx 48..71)
      exp[0..23]  grid export            (idx 72..95)
      soc[0..24]  state of charge        (idx 96..120)
    Total: 121 variables
    """
    T = 24
    C = batt.capacity_kwh
    eta_c = batt.eta_c
    eta_d = batt.eta_d
    dt = batt.dt
    n_vars = 4 * T + (T + 1)  # 121

    def ic(t):   return t
    def id_(t):  return T + t
    def ii(t):   return 2*T + t
    def ie(t):   return 3*T + t
    def isoc(t): return 4*T + t

    # ------------------------------------------------------------------
    # Objective: min Σ [ price[t]·imp[t]·dt − price[t]·exp[t]·dt ]
    # ------------------------------------------------------------------
    obj = np.zeros(n_vars)
    for t in range(T):
        obj[ii(t)] =  prices[t] * dt
        obj[ie(t)] = -prices[t] * dt

    # ------------------------------------------------------------------
    # Bounds
    # ------------------------------------------------------------------
    bounds = []
    for t in range(T):   bounds.append((0.0, batt.cmax_kw))       # c
    for t in range(T):   bounds.append((0.0, batt.dmax_kw))       # d
    for t in range(T):   bounds.append((0.0, None))                # imp
    for t in range(T):                                              # exp
        bounds.append((0.0, None) if batt.allow_export else (0.0, 0.0))
    for t in range(T+1): bounds.append((batt.soc_min, batt.soc_max))  # soc

    # ------------------------------------------------------------------
    # Equality constraints:  A_eq @ x = b_eq
    # ------------------------------------------------------------------
    n_eq = T + T + 1   # 49
    A_eq = np.zeros((n_eq, n_vars))
    b_eq = np.zeros(n_eq)
    row = 0

    # SOC dynamics
    for t in range(T):
        A_eq[row, isoc(t+1)] =  1.0
        A_eq[row, isoc(t)]   = -1.0
        A_eq[row, ic(t)]     = -(eta_c * dt) / C
        A_eq[row, id_(t)]    =  dt / (eta_d * C)
        b_eq[row] = 0.0
        row += 1

    # Grid balance: imp − exp − c + d = demand
    for t in range(T):
        A_eq[row, ii(t)]  =  1.0
        A_eq[row, ie(t)]  = -1.0
        A_eq[row, ic(t)]  = -1.0
        A_eq[row, id_(t)] =  1.0
        b_eq[row] = demand[t]
        row += 1

    # Initial SOC
    A_eq[row, isoc(0)] = 1.0
    b_eq[row] = soc0
    row += 1

    # ------------------------------------------------------------------
    # Inequality constraints:  A_ub @ x <= b_ub
    # ------------------------------------------------------------------
    ub_rows = []
    ub_rhs  = []

    # (a) Final SOC target: −soc[24] <= −soc_target
    r = np.zeros(n_vars); r[isoc(T)] = -1.0
    ub_rows.append(r); ub_rhs.append(-batt.soc_target)

    # (b) Rolling-window energy throughput
    W = ROLLING_WINDOW_H
    for start in range(T - W + 1):
        # Charge window: Σ c[s]·η_c·dt <= C
        r = np.zeros(n_vars)
        for s in range(start, start + W):
            r[ic(s)] = eta_c * dt
        ub_rows.append(r); ub_rhs.append(C)

        # Discharge window: Σ d[s]·dt/η_d <= C
        r = np.zeros(n_vars)
        for s in range(start, start + W):
            r[id_(s)] = dt / eta_d
        ub_rows.append(r); ub_rhs.append(C)

    A_ub = np.array(ub_rows)
    b_ub = np.array(ub_rhs)

    # ------------------------------------------------------------------
    # Solve
    # ------------------------------------------------------------------
    result = linprog(
        c=obj,
        A_ub=A_ub, b_ub=b_ub,
        A_eq=A_eq, b_eq=b_eq,
        bounds=bounds,
        method='highs',
        options={'presolve': True},
    )

    if not result.success:
        return None

    x = result.x

    # --- Post-LP cleanup: eliminate simultaneous c & d ---
    c_sol = [x[ic(t)] for t in range(T)]
    d_sol = [x[id_(t)] for t in range(T)]

    for t in range(T):
        if c_sol[t] > 1e-6 and d_sol[t] > 1e-6:
            # Net out: keep only the dominant direction
            charge_e = c_sol[t] * eta_c * dt
            discharge_e = d_sol[t] * dt / eta_d
            if charge_e >= discharge_e:
                net_e = charge_e - discharge_e
                c_sol[t] = net_e / (eta_c * dt)
                d_sol[t] = 0.0
            else:
                net_e = discharge_e - charge_e
                c_sol[t] = 0.0
                d_sol[t] = net_e * eta_d / dt

    # Rebuild SOC, import, export from cleaned c, d
    soc_sol = [soc0]
    imp_sol = []
    exp_sol = []
    for t in range(T):
        next_soc = soc_sol[t] + (eta_c * c_sol[t] * dt - d_sol[t] * dt / eta_d) / C
        next_soc = max(batt.soc_min, min(batt.soc_max, next_soc))
        soc_sol.append(next_soc)

        net = demand[t] + c_sol[t] - d_sol[t]
        if batt.allow_export:
            if net >= 0:
                imp_sol.append(net)
                exp_sol.append(0.0)
            else:
                imp_sol.append(0.0)
                exp_sol.append(-net)
        else:
            imp_sol.append(max(net, 0.0))
            exp_sol.append(0.0)

    cost = sum(
        prices[t] * imp_sol[t] * dt - prices[t] * exp_sol[t] * dt
        for t in range(T)
    )

    return {
        'c': c_sol, 'd': d_sol,
        'imp': imp_sol, 'exp': exp_sol,
        'soc': soc_sol, 'obj': cost,
    }


# ============================================================================
# 6.  Build corrected rows from LP solution
# ============================================================================

def build_corrected_rows(
    sol: Dict,
    actual_prices: List[float],
    forecast_prices: List[float],
    actual_demand: List[float],
    forecast_demand: List[float],
    batt: BatteryConfig,
) -> List[Dict]:
    """Turn an LP solution into 24 CSV row-dicts in the target format."""
    rows = []
    dt = batt.dt
    for t in range(24):
        c_t   = sol['c'][t]
        d_t   = sol['d'][t]
        soc_t = sol['soc'][t]

        # Recompute import/export using ACTUAL demand
        net_actual = actual_demand[t] + c_t - d_t
        if batt.allow_export:
            if net_actual >= 0:
                imp_actual = net_actual
                exp_actual = 0.0
            else:
                imp_actual = 0.0
                exp_actual = -net_actual
        else:
            imp_actual = max(net_actual, 0.0)
            exp_actual = 0.0

        # profit_step = savings vs no-battery baseline on actual prices
        baseline_cost = actual_prices[t] * actual_demand[t] * dt
        actual_cost   = actual_prices[t] * imp_actual * dt - actual_prices[t] * exp_actual * dt
        profit_step   = baseline_cost - actual_cost

        rows.append({
            'prices_actual':   round(actual_prices[t], 2),
            'prices_forecast': round(forecast_prices[t], 2),
            'actual_demand':   round(actual_demand[t], 2),
            'forecast_demand': round(forecast_demand[t], 2),
            'soc':             round(soc_t, 6),
            'charge_MW':       round(c_t, 4),
            'discharge_MW':    round(d_t, 4),
            'import_MW':       round(imp_actual, 4),
            'export_MW':       round(exp_actual, 4),
            'profit_step':     round(profit_step, 2),
        })
    return rows


# ============================================================================
# 7.  Main Pipeline
# ============================================================================

def main():
    print("=" * 70)
    print("Storage Arbitrage — EXHAUSTIVE Violation Check & LP Correction")
    print("=" * 70)

    # --- Read cleaned CSV ---
    days = read_csv_days(INPUT_CSV)
    n_days = len(days)
    print(f"Read {n_days} days ({n_days * 24} rows) from {INPUT_CSV}")

    # --- Optionally load JSONL ---
    jsonl_records = None
    if JSONL_FILE:
        jsonl_records = load_jsonl_days(JSONL_FILE)
        print(f"Loaded {len(jsonl_records)} JSONL records from {JSONL_FILE}")

    default_batt = BatteryConfig()
    default_soc0 = 0.5

    columns = [
        'prices_actual', 'prices_forecast', 'actual_demand', 'forecast_demand',
        'soc', 'charge_MW', 'discharge_MW', 'import_MW', 'export_MW', 'profit_step'
    ]

    stats = {
        'feasible': 0, 'violated': 0, 'corrected': 0,
        'lp_failed': 0, 'post_lp_clean': 0, 'post_lp_violation': 0,
        'violation_types': {},
    }

    corrected_days = []

    for day_idx, day_rows in enumerate(days):
        # --- Battery config & soc0 ---
        if jsonl_records and day_idx < len(jsonl_records):
            batt, soc0 = battery_from_jsonl(jsonl_records[day_idx])
        else:
            batt = default_batt
            try:
                soc0 = float(day_rows[0]['soc'])
            except (ValueError, KeyError):
                soc0 = default_soc0

        # --- Price / demand vectors ---
        actual_prices   = [safe_float(r['prices_actual'])[0]   for r in day_rows]
        forecast_prices = [safe_float(r['prices_forecast'])[0] for r in day_rows]
        actual_demand   = [safe_float(r['actual_demand'])[0]   for r in day_rows]
        forecast_demand = [safe_float(r['forecast_demand'])[0] for r in day_rows]

        # --- Step 1: Exhaustive check ---
        feasible, violations = exhaustive_check(day_rows, batt, soc0)

        if feasible:
            stats['feasible'] += 1
            corrected_days.append(day_rows)
            if day_idx < 5 or day_idx % 50 == 0:
                print(f"Day {day_idx:3d}  ✓ feasible (kept original)")
            continue

        # --- Tally violation types ---
        stats['violated'] += 1
        for v in violations:
            tag = v.split(']')[0] + ']' if ']' in v else 'unknown'
            stats['violation_types'][tag] = stats['violation_types'].get(tag, 0) + 1

        n_v = len(violations)

        # --- Step 2: LP correction using FORECAST prices ---
        sol = solve_lp(batt, soc0, forecast_prices, forecast_demand)

        if sol is None:
            stats['lp_failed'] += 1
            corrected_days.append(day_rows)
            print(f"Day {day_idx:3d}  ✗ {n_v} violation(s) → LP INFEASIBLE (kept original)")
            for v in violations[:5]:
                print(f"           {v}")
            continue

        stats['corrected'] += 1

        # --- Build corrected rows ---
        new_rows = build_corrected_rows(
            sol, actual_prices, forecast_prices,
            actual_demand, forecast_demand, batt,
        )

        # --- Step 3: Post-LP verification ---
        post_ok, post_violations = exhaustive_check(new_rows, batt, soc0)
        if post_ok:
            stats['post_lp_clean'] += 1
            if day_idx < 10 or day_idx % 50 == 0:
                print(f"Day {day_idx:3d}  ✗→✓ {n_v} violation(s) corrected  "
                      f"(obj={sol['obj']:.2f})")
        else:
            stats['post_lp_violation'] += 1
            print(f"Day {day_idx:3d}  ⚠ LP solved but {len(post_violations)} "
                  f"post-LP violation(s):")
            for v in post_violations[:5]:
                print(f"           {v}")

        corrected_days.append(new_rows)

    # --- Write corrected CSV ---
    with open(OUTPUT_CSV, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=columns)
        writer.writeheader()
        for day_block in corrected_days:
            for row in day_block:
                out = {col: row.get(col, '') for col in columns}
                writer.writerow(out)

    total_rows = sum(len(d) for d in corrected_days)

    # --- Summary ---
    print()
    print("=" * 70)
    print(f"OUTPUT: {OUTPUT_CSV}  ({total_rows} rows, {n_days} days)")
    print("=" * 70)
    print(f"  Already feasible       : {stats['feasible']:4d}")
    print(f"  Violated → LP clean    : {stats['post_lp_clean']:4d}")
    print(f"  Violated → LP warning  : {stats['post_lp_violation']:4d}")
    print(f"  LP infeasible (kept)   : {stats['lp_failed']:4d}")
    print(f"  Total violated         : {stats['violated']:4d}")
    print()
    if stats['violation_types']:
        print("Violation breakdown (across all violated days):")
        for tag in sorted(stats['violation_types'].keys()):
            print(f"  {tag:8s} : {stats['violation_types'][tag]:4d} occurrences")
    print("=" * 70)


if __name__ == "__main__":
    main()

Storage Arbitrage — EXHAUSTIVE Violation Check & LP Correction
Read 364 days (8736 rows) from /content/drive/MyDrive/energy_finetuning_35/gpt4_hourly_arbitrage_cleaned.csv
Day   0  ✗→✓ 14 violation(s) corrected  (obj=41178.34)
Day   1  ✗→✓ 10 violation(s) corrected  (obj=34255.43)
Day   2  ✗→✓ 8 violation(s) corrected  (obj=21523.51)
Day   3  ✗→✓ 7 violation(s) corrected  (obj=25371.87)
Day   4  ✗→✓ 12 violation(s) corrected  (obj=44014.04)
Day   5  ✗→✓ 13 violation(s) corrected  (obj=54648.85)
Day   6  ✗→✓ 3 violation(s) corrected  (obj=65147.06)
Day   7  ✗→✓ 6 violation(s) corrected  (obj=60496.99)
Day   8  ✗→✓ 8 violation(s) corrected  (obj=55056.59)
Day  50  ✗→✓ 2 violation(s) corrected  (obj=34700.59)
Day 100  ✗→✓ 2 violation(s) corrected  (obj=45843.17)
Day 150  ✗→✓ 2 violation(s) corrected  (obj=36789.06)
Day 200  ✗→✓ 5 violation(s) corrected  (obj=45119.09)
Day 250  ✗→✓ 3 violation(s) corrected  (obj=66205.15)
Day 300  ✗→✓ 2 violation(s) corrected  (obj=57094.27)
Day 350  ✗→✓ 1

In [ ]:
# -*- coding: utf-8 -*-
"""
GPT-4o Energy Arbitrage → Hourly Operational CSV
Temperature sweep version: single day (2019-12-01), temperature 0.0 to 1.0 in 0.1 steps.
Outputs CSV with columns (one row per hour, 24 rows per temperature):
  temperature, prices_actual, prices_forecast, actual_demand, forecast_demand,
  soc, charge_MW, discharge_MW, import_MW, export_MW, profit_step
"""

import os
import json
import time
import re
import csv
import numpy as np
from dataclasses import dataclass, field
from typing import List, Tuple, Optional, Dict
from openai import OpenAI
from google.colab import drive, userdata

drive.mount('/content/drive')


# ============================================================================
# Configuration
# ============================================================================

@dataclass
class Config:
    output_csv: str = "/content/drive/MyDrive/energy_finetuning_35/RF_0.8_Dec1_Table_5_gpt4_hourly_arbitrage_temp_sweep.csv"
    model: str = "gpt-4"
    max_tokens: int = 4000
    delay_between_calls: float = 1.0
    temperatures: List[float] = None

    def __post_init__(self):
        if self.temperatures is None:
            self.temperatures = [round(t * 0.1, 1) for t in range(11)]  # 0.0 to 1.0


# ============================================================================
# Hard-coded test example for 2019-12-01
# ============================================================================

# TEST_EXAMPLE = {"date": "2019-12-01", "user_message": "{\"capacity_kwh\":49.44,\"max_c_kw\":12.36,\"max_d_kw\":12.36,\"eta_c\":0.95,\"eta_d\":0.95,\"soc0\":0.5,\"soc_min\":0.0,\"soc_max\":1.0,\"soc_target\":0.0,\"dt_hours\":1.0,\"allow_export\":true,\"forecast_prices_today\":[70.0439,66.1962,50.289,44.8338,42.1292,41.0453,42.942,48.7814,54.5334,58.4184,60.5881,61.0029,60.11,59.1443,58.9285,60.3471,64.3788,70.9903,76.1837,77.2201,75.3774,71.9356,68.338,65.5672],\"forecast_demand_today\":[29.6421,24.9885,23.7936,23.3284,23.1765,23.2771,23.9128,25.3234,26.7729,28.1761,28.3071,31.6567,35.0113,33.8905,33.2255,33.2157,34.5011,37.0462,39.4655,40.2593,39.3682,36.798,34.4156,31.677],\"actual_prices_yesterday\":[43.59,38.05,36.22,35.39,35.39,36.03,45.83,52.5,57.0,51.73,50.0,46.73,45.0,44.0,45.47,53.91,60.85,67.65,66.73,67.93,64.78,59.67,55.01,48.06],\"forecast_prices_yesterday\":[63.3979,56.3422,52.0795,47.4203,44.0469,45.1606,54.3197,64.5856,67.4301,68.6985,68.288,67.2903,65.3475,62.8891,62.234,63.5998,66.508,70.2757,73.4665,73.7933,72.0276,70.1256,68.1872,65.5252],\"naive_soc\":[0.5,0.236842,0.0,0.0,0.2375,0.475,0.7125,0.95,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.736842,0.473684,0.210526,0.0,0.0,0.0],\"naive_cost\":43099.128,\"yesterday_error_stats\":{\"mean\":-12.73,\"std\":5.2806,\"max_positive\":-2.6257,\"max_negative\":-20.5603}}", "assistant_message": "{\"soc\":[0.5,0.5,0.5,0.5,0.525,0.7625,1.0,1.0,1.0,0.736842,0.473684,0.473684,0.473684,0.473684,0.711184,0.948684,1.0,1.0,0.736842,0.473684,0.210526,0.0,0.0,0.0,0.0],\"cost\":49188.545,\"improvement_over_naive\":-6089.417}", "actual_prices_today": [65.1, 62.12, 59.05, 59.0, 56.63, 56.63, 61.66, 66.61, 73.0, 73.0, 65.0, 63.82, 62.19, 59.5, 56.99, 61.92, 65.63, 82.47, 82.79, 83.5, 78.62, 72.94, 64.53, 60.32], "actual_demand_today": [28.68, 27.12, 25.99, 25.62, 25.66, 26.06, 28.58, 32.44, 35.25, 36.63, 36.22, 35.17, 33.26, 31.54, 31.52, 32.14, 33.53, 37.37, 38.78, 38.96, 36.4, 33.51, 30.89, 28.02], "forecast_prices_today": [70.04394576, 66.19615546, 50.28899139, 44.83377512, 42.12923819, 41.0452562, 42.94199337, 48.78141431, 54.53338023, 58.41835125, 60.5880948, 61.00291539, 60.10997788, 59.14428852, 58.92852607, 60.34708329, 64.37877232, 70.99030546, 76.1836874, 77.22009307, 75.37738082, 71.93564878, 68.33804907, 65.56716853], "milp_actual_soc": [0.5, 0.5, 0.5, 0.5, 0.5249999999999999, 0.7625, 1.0, 1.0, 1.0, 0.736842105263158, 0.4736842105263159, 0.4736842105263159, 0.4736842105263159, 0.4736842105263159, 0.7111842105263159, 0.9486842105263158, 1.0, 1.0, 0.736842105263158, 0.4736842105263159, 0.21052631578947378, 0.0, 0.0, 0.0, 0.0], "milp_actual_cost": 49188.544965207766, "milp_forecast_soc": [0.5, 0.2368421052631579, 0.0, 0.0, 0.2375, 0.475, 0.7124999999999999, 0.95, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.736842105263158, 0.4736842105263159, 0.21052631578947378, 0.0, 0.0, 0.0], "milp_forecast_cost": 43099.12795876155, "battery": {"capacity_kwh": 49.44, "cmax_kw": 12.36, "dmax_kw": 12.36, "eta_c": 0.95, "eta_d": 0.95, "soc_init": 0.5, "soc_target": 0.0}}

# TEST_EXAMPLE = {"date": "2019-12-01", "user_message": "{\"capacity_kwh\":43.13,\"max_c_kw\":10.7825,\"max_d_kw\":10.7825,\"eta_c\":0.95,\"eta_d\":0.95,\"soc0\":0.5,\"soc_min\":0.0,\"soc_max\":1.0,\"soc_target\":0.0,\"dt_hours\":1.0,\"allow_export\":true,\"forecast_prices_today\":[46.2343,41.5005,39.3378,38.0062,39.032,38.2413,42.1012,47.0631,53.1575,52.2232,56.221,54.2465,50.4386,52.6064,49.8465,57.7811,62.4085,67.3157,56.2016,60.3596,62.0148,62.2724,53.1024,45.9017],\"forecast_demand_today\":[26.8617,25.2153,23.9225,23.8409,23.5584,24.0203,27.171,29.1947,29.2061,29.9812,30.4728,33.6008,30.9495,30.4918,28.9571,29.4635,33.2702,35.6648,36.3692,36.6453,34.4189,32.3334,29.9367,26.8444],\"actual_prices_yesterday\":[43.59,38.05,36.22,35.39,35.39,36.03,45.83,52.5,57.0,51.73,50.0,46.73,45.0,44.0,45.47,53.91,60.85,67.65,66.73,67.93,64.78,59.67,55.01,48.06],\"forecast_prices_yesterday\":[57.0772,55.0874,51.5993,47.2783,45.9009,51.9436,57.7168,64.2002,64.7076,64.3879,63.2415,63.5237,62.6621,61.6693,54.7222,63.4767,67.4161,79.62,74.6026,74.5211,69.3621,64.7601,61.8262,57.9785],\"naive_soc\":[0.5,0.236842,0.236842,0.2875,0.525,0.7625,1.0,1.0,1.0,1.0,1.0,0.736842,0.736842,0.7625,0.7625,1.0,1.0,0.736842,0.473684,0.473684,0.473684,0.263158,0.0,0.0,0.0],\"naive_cost\":35379.4368,\"yesterday_error_stats\":{\"mean\":-11.3234,\"std\":3.9608,\"max_positive\":-4.5821,\"max_negative\":-17.6693}}", "assistant_message": "{\"soc\":[0.5,0.5,0.5,0.5,0.525,0.7625,1.0,1.0,1.0,0.736842,0.473684,0.473684,0.473684,0.473684,0.711184,0.948684,1.0,1.0,0.736842,0.473684,0.210526,0.0,0.0,0.0,0.0],\"cost\":49517.1645,\"improvement_over_naive\":-14137.7277}", "actual_prices_today": [65.1, 62.12, 59.05, 59.0, 56.63, 56.63, 61.66, 66.61, 73.0, 73.0, 65.0, 63.82, 62.19, 59.5, 56.99, 61.92, 65.63, 82.47, 82.79, 83.5, 78.62, 72.94, 64.53, 60.32], "actual_demand_today": [28.68, 27.12, 25.99, 25.62, 25.66, 26.06, 28.58, 32.44, 35.25, 36.63, 36.22, 35.17, 33.26, 31.54, 31.52, 32.14, 33.53, 37.37, 38.78, 38.96, 36.4, 33.51, 30.89, 28.02], "forecast_prices_today": [46.23430042, 41.50053732, 39.33778151, 38.00616949, 39.03200476, 38.24127885, 42.10119966, 47.06307124, 53.15752937, 52.22315992, 56.22102261, 54.24650748, 50.43856725, 52.60644333, 49.84653038, 57.78106286, 62.40845962, 67.31571429, 56.20156832, 60.35956522, 62.01479177, 62.27242035, 53.10237809, 45.90174945], "milp_actual_soc": [0.5, 0.5, 0.5, 0.5, 0.5249999999999999, 0.7625, 1.0, 1.0, 1.0, 0.736842105263158, 0.4736842105263159, 0.4736842105263159, 0.4736842105263159, 0.4736842105263159, 0.7111842105263159, 0.9486842105263158, 1.0, 1.0, 0.736842105263158, 0.4736842105263159, 0.21052631578947378, 0.0, 0.0, 0.0, 0.0], "milp_actual_cost": 49517.16446026317, "milp_forecast_soc": [0.5, 0.2368421052631579, 0.2368421052631579, 0.2874999999999999, 0.5249999999999999, 0.7625, 1.0, 1.0, 1.0, 1.0, 1.0, 0.736842105263158, 0.736842105263158, 0.7625, 0.7625, 1.0, 1.0, 0.736842105263158, 0.4736842105263159, 0.4736842105263159, 0.4736842105263159, 0.2631578947368421, 0.0, 0.0, 0.0], "milp_forecast_cost": 35379.43676891254, "battery": {"capacity_kwh": 43.13, "cmax_kw": 10.7825, "dmax_kw": 10.7825, "eta_c": 0.95, "eta_d": 0.95, "soc_init": 0.5, "soc_target": 0.0}}

# TEST_EXAMPLE = {"date": "2019-12-01", "user_message": "{\"capacity_kwh\":43.13,\"max_c_kw\":10.7825,\"max_d_kw\":10.7825,\"eta_c\":0.95,\"eta_d\":0.95,\"soc0\":0.5,\"soc_min\":0.0,\"soc_max\":1.0,\"soc_target\":0.0,\"dt_hours\":1.0,\"allow_export\":true,\"forecast_prices_today\":[34.1516,34.3143,34.4309,32.9696,33.3634,34.3337,37.1582,42.0842,44.4547,45.4662,44.5037,41.3267,36.7556,39.3659,39.4164,41.5647,44.0515,46.1579,49.1362,52.3791,53.984,50.9398,47.6858,44.9458],\"forecast_demand_today\":[24.1342,24.0063,22.8828,24.4648,22.2841,23.8934,23.203,24.1397,26.4769,27.4646,28.7625,27.6827,26.304,27.9586,26.5464,26.5659,28.4339,30.3139,30.6533,31.5269,31.6251,32.3193,28.3018,26.6944],\"actual_prices_yesterday\":[43.59,38.05,36.22,35.39,35.39,36.03,45.83,52.5,57.0,51.73,50.0,46.73,45.0,44.0,45.47,53.91,60.85,67.65,66.73,67.93,64.78,59.67,55.01,48.06],\"forecast_prices_yesterday\":[41.5579,39.3722,36.3669,35.7425,35.2696,40.4437,44.9407,50.7851,51.4716,49.9037,49.0127,47.0455,45.8864,47.1013,48.7154,50.0333,53.264,56.3657,56.3876,57.9095,54.3442,51.1108,46.1525,42.7963],\"naive_soc\":[0.5,0.525,0.525,0.525,0.7625,1.0,1.0,1.0,1.0,0.813816,0.550658,0.2875,0.2875,0.525,0.7625,1.0,1.0,1.0,1.0,0.789474,0.526316,0.263158,0.0,0.0,0.0],\"naive_cost\":26035.0421,\"yesterday_error_stats\":{\"mean\":3.1475,\"std\":4.7566,\"max_positive\":11.2843,\"max_negative\":-4.4137}}", "assistant_message": "{\"soc\":[0.5,0.5,0.5,0.5,0.525,0.7625,1.0,1.0,1.0,0.736842,0.473684,0.473684,0.473684,0.473684,0.711184,0.948684,1.0,1.0,0.736842,0.473684,0.210526,0.0,0.0,0.0,0.0],\"cost\":49517.1645,\"improvement_over_naive\":-23482.1224}", "actual_prices_today": [65.1, 62.12, 59.05, 59.0, 56.63, 56.63, 61.66, 66.61, 73.0, 73.0, 65.0, 63.82, 62.19, 59.5, 56.99, 61.92, 65.63, 82.47, 82.79, 83.5, 78.62, 72.94, 64.53, 60.32], "actual_demand_today": [28.68, 27.12, 25.99, 25.62, 25.66, 26.06, 28.58, 32.44, 35.25, 36.63, 36.22, 35.17, 33.26, 31.54, 31.52, 32.14, 33.53, 37.37, 38.78, 38.96, 36.4, 33.51, 30.89, 28.02], "forecast_prices_today": [34.151596, 34.31434, 34.43087, 32.969643, 33.363434, 34.333748, 37.15817, 42.084244, 44.454678, 45.46617, 44.503666, 41.326714, 36.75562, 39.36592, 39.41636, 41.564682, 44.05148, 46.157856, 49.136185, 52.379086, 53.983997, 50.939793, 47.68584, 44.945766], "milp_actual_soc": [0.5, 0.5, 0.5, 0.5, 0.5249999999999999, 0.7625, 1.0, 1.0, 1.0, 0.736842105263158, 0.4736842105263159, 0.4736842105263159, 0.4736842105263159, 0.4736842105263159, 0.7111842105263159, 0.9486842105263158, 1.0, 1.0, 0.736842105263158, 0.4736842105263159, 0.21052631578947378, 0.0, 0.0, 0.0, 0.0], "milp_actual_cost": 49517.16446026317, "milp_forecast_soc": [0.5, 0.5249999999999999, 0.5249999999999999, 0.5249999999999999, 0.7625, 1.0, 1.0, 1.0, 1.0, 0.8138157894736842, 0.550657894736842, 0.2874999999999999, 0.2874999999999999, 0.5249999999999999, 0.7625, 1.0, 1.0, 1.0, 1.0, 0.7894736842105263, 0.5263157894736842, 0.2631578947368421, 0.0, 0.0, 0.0], "milp_forecast_cost": 26035.04210855555, "battery": {"capacity_kwh": 43.13, "cmax_kw": 10.7825, "dmax_kw": 10.7825, "eta_c": 0.95, "eta_d": 0.95, "soc_init": 0.5, "soc_target": 0.0}}

# TEST_EXAMPLE = {"date": "2019-12-01", "user_message": "{\"capacity_kwh\":43.13,\"max_c_kw\":10.7825,\"max_d_kw\":10.7825,\"eta_c\":0.95,\"eta_d\":0.95,\"soc0\":0.5,\"soc_min\":0.0,\"soc_max\":1.0,\"soc_target\":0.0,\"dt_hours\":1.0,\"allow_export\":true,\"forecast_prices_today\":[65.1,62.12,59.05,59.0,56.63,56.63,61.66,66.61,73.0,73.0,65.0,63.82,62.19,59.5,56.99,61.92,65.63,82.47,82.79,83.5,78.62,72.94,64.53,60.32],\"forecast_demand_today\":[28.68,27.12,25.99,25.62,25.66,26.06,28.58,32.44,35.25,36.63,36.22,35.17,33.26,31.54,31.52,32.14,33.53,37.37,38.78,38.96,36.4,33.51,30.89,28.02],\"actual_prices_yesterday\":[43.59,38.05,36.22,35.39,35.39,36.03,45.83,52.5,57.0,51.73,50.0,46.73,45.0,44.0,45.47,53.91,60.85,67.65,66.73,67.93,64.78,59.67,55.01,48.06],\"forecast_prices_yesterday\":[43.59,38.05,36.22,35.39,35.39,36.03,45.83,52.5,57.0,51.73,50.0,46.73,45.0,44.0,45.47,53.91,60.85,67.65,66.73,67.93,64.78,59.67,55.01,48.06],\"naive_soc\":[0.5,0.5,0.5,0.5,0.525,0.7625,1.0,1.0,1.0,0.736842,0.473684,0.473684,0.473684,0.473684,0.711184,0.948684,1.0,1.0,0.736842,0.473684,0.210526,0.0,0.0,0.0,0.0],\"naive_cost\":49517.1645,\"yesterday_error_stats\":{\"mean\":0.0,\"std\":0.0,\"max_positive\":0.0,\"max_negative\":0.0}}", "assistant_message": "{\"soc\":[0.5,0.5,0.5,0.5,0.525,0.7625,1.0,1.0,1.0,0.736842,0.473684,0.473684,0.473684,0.473684,0.711184,0.948684,1.0,1.0,0.736842,0.473684,0.210526,0.0,0.0,0.0,0.0],\"cost\":49517.1645,\"improvement_over_naive\":0.0}", "actual_prices_today": [65.1, 62.12, 59.05, 59.0, 56.63, 56.63, 61.66, 66.61, 73.0, 73.0, 65.0, 63.82, 62.19, 59.5, 56.99, 61.92, 65.63, 82.47, 82.79, 83.5, 78.62, 72.94, 64.53, 60.32], "actual_demand_today": [28.68, 27.12, 25.99, 25.62, 25.66, 26.06, 28.58, 32.44, 35.25, 36.63, 36.22, 35.17, 33.26, 31.54, 31.52, 32.14, 33.53, 37.37, 38.78, 38.96, 36.4, 33.51, 30.89, 28.02], "forecast_prices_today": [65.1, 62.12, 59.05, 59.0, 56.63, 56.63, 61.66, 66.61, 73.0, 73.0, 65.0, 63.82, 62.19, 59.5, 56.99, 61.92, 65.63, 82.47, 82.79, 83.5, 78.62, 72.94, 64.53, 60.32], "milp_actual_soc": [0.5, 0.5, 0.5, 0.5, 0.5249999999999999, 0.7625, 1.0, 1.0, 1.0, 0.736842105263158, 0.4736842105263159, 0.4736842105263159, 0.4736842105263159, 0.4736842105263159, 0.7111842105263159, 0.9486842105263158, 1.0, 1.0, 0.736842105263158, 0.4736842105263159, 0.21052631578947378, 0.0, 0.0, 0.0, 0.0], "milp_actual_cost": 49517.16446026317, "milp_forecast_soc": [0.5, 0.5, 0.5, 0.5, 0.5249999999999999, 0.7625, 1.0, 1.0, 1.0, 0.736842105263158, 0.4736842105263159, 0.4736842105263159, 0.4736842105263159, 0.4736842105263159, 0.7111842105263159, 0.9486842105263158, 1.0, 1.0, 0.736842105263158, 0.4736842105263159, 0.21052631578947378, 0.0, 0.0, 0.0, 0.0], "milp_forecast_cost": 49517.16446026317, "battery": {"capacity_kwh": 43.13, "cmax_kw": 10.7825, "dmax_kw": 10.7825, "eta_c": 0.95, "eta_d": 0.95, "soc_init": 0.5, "soc_target": 0.0}}

# TEST_EXAMPLE = {"date": "2019-01-12", "user_message": "{\"capacity_kwh\":43.13,\"max_c_kw\":21.565,\"max_d_kw\":21.565,\"eta_c\":0.95,\"eta_d\":0.95,\"soc0\":0.5,\"soc_min\":0.0,\"soc_max\":1.0,\"soc_target\":0.0,\"dt_hours\":1.0,\"allow_export\":true,\"forecast_prices_today\":[55.6757,53.0724,51.0018,47.3662,47.4073,55.01,68.3115,73.1417,83.7939,65.4162,65.2555,59.1089,56.4706,51.5673,55.1069,62.445,69.6633,80.9967,74.2242,74.5211,68.362,64.1417,63.0168,57.1728],\"forecast_demand_today\":[27.2116,25.6195,24.584,24.2211,24.1777,24.8347,26.1967,27.9592,34.9061,34.6584,34.9736,33.8194,34.1818,36.3375,37.4211,34.6604,35.1692,37.8039,39.1928,39.4898,37.932,35.106,32.2228,30.2691],\"actual_prices_yesterday\":[40.29,39.1,38.28,36.53,36.53,36.02,37.47,38.05,37.9,40.72,41.67,41.67,42.75,41.61,41.0,44.01,46.2,53.05,51.34,49.55,50.63,46.76,42.71,37.47],\"forecast_prices_yesterday\":[47.0389,43.3513,42.6604,36.0262,33.4557,43.6875,49.4515,78.9105,86.6604,58.7368,69.0214,65.0664,55.1343,59.7509,60.3883,80.1915,83.6062,79.62,69.465,74.5211,60.3787,56.3515,55.5102,49.3141],\"naive_soc\":[0.5,0.05,0.05,0.05,0.525,1.0,1.0,1.0,0.526316,0.0,0.0,0.0,0.0,0.05,0.525,1.0,1.0,1.0,0.473684,0.473684,0.0,0.0,0.0,0.0,0.0],\"naive_cost\":46122.8066,\"yesterday_error_stats\":{\"mean\":-17.7912,\"std\":13.0406,\"max_positive\":3.0743,\"max_negative\":-48.7604}}", "assistant_message": "{\"soc\":[0.5,0.0,0.0,0.0,0.475,0.95,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.526316,0.0,0.0,0.0,0.0,0.0],\"cost\":29782.091,\"improvement_over_naive\":16340.7156}", "actual_prices_today": [47.86, 44.52, 41.42, 38.67, 38.23, 39.48, 43.15, 43.77, 45.37, 46.44, 46.44, 48.0, 48.15, 45.76, 45.84, 45.46, 48.34, 55.12, 56.2, 56.42, 55.14, 53.84, 46.48, 43.18], "actual_demand_today": [25.2, 23.42, 22.35, 21.83, 21.76, 22.32, 23.75, 25.19, 26.84, 28.62, 29.23, 29.41, 29.23, 27.87, 27.6, 28.14, 29.56, 32.27, 33.19, 33.83, 32.79, 30.99, 28.65, 26.17], "forecast_prices_today": [55.67568033, 53.07239038, 51.00181992, 47.36623821, 47.40726052, 55.01003384, 68.31150847, 73.14173882, 83.79392558, 65.41623894, 65.25547287, 59.10891214, 56.47060763, 51.56733, 55.1069477, 62.44495389, 69.66330869, 80.99666667, 74.2241846, 74.52112903, 68.36196225, 64.14167772, 63.01678135, 57.17275082], "milp_actual_soc": [0.5, 0.0, 0.0, 0.0, 0.475, 0.95, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.5263157894736842, 0.0, 0.0, 0.0, 0.0, 0.0], "milp_actual_cost": 29782.091045, "milp_forecast_soc": [0.5, 0.05, 0.05, 0.05, 0.525, 1.0, 1.0, 1.0, 0.5263157894736842, 0.0, 0.0, 0.0, 0.0, 0.05, 0.525, 1.0, 1.0, 1.0, 0.4736842105263158, 0.4736842105263158, 0.0, 0.0, 0.0, 0.0, 0.0], "milp_forecast_cost": 46122.80662771122, "battery": {"capacity_kwh": 43.13, "cmax_kw": 21.565, "dmax_kw": 21.565, "eta_c": 0.95, "eta_d": 0.95, "soc_init": 0.5, "soc_target": 0.0}}

# TEST_EXAMPLE = {"date": "2019-01-12", "user_message": "{\"capacity_kwh\":43.13,\"max_c_kw\":7.1883333333333335,\"max_d_kw\":7.1883333333333335,\"eta_c\":0.95,\"eta_d\":0.95,\"soc0\":0.5,\"soc_min\":0.0,\"soc_max\":1.0,\"soc_target\":0.0,\"dt_hours\":1.0,\"allow_export\":true,\"forecast_prices_today\":[55.6757,53.0724,51.0018,47.3662,47.4073,55.01,68.3115,73.1417,83.7939,65.4162,65.2555,59.1089,56.4706,51.5673,55.1069,62.445,69.6633,80.9967,74.2242,74.5211,68.362,64.1417,63.0168,57.1728],\"forecast_demand_today\":[27.2116,25.6195,24.584,24.2211,24.1777,24.8347,26.1967,27.9592,34.9061,34.6584,34.9736,33.8194,34.1818,36.3375,37.4211,34.6604,35.1692,37.8039,39.1928,39.4898,37.932,35.106,32.2228,30.2691],\"actual_prices_yesterday\":[40.29,39.1,38.28,36.53,36.53,36.02,37.47,38.05,37.9,40.72,41.67,41.67,42.75,41.61,41.0,44.01,46.2,53.05,51.34,49.55,50.63,46.76,42.71,37.47],\"forecast_prices_yesterday\":[47.0389,43.3513,42.6604,36.0262,33.4557,43.6875,49.4515,78.9105,86.6604,58.7368,69.0214,65.0664,55.1343,59.7509,60.3883,80.1915,83.6062,79.62,69.465,74.5211,60.3787,56.3515,55.5102,49.3141],\"naive_soc\":[0.5,0.5,0.525,0.683333,0.841667,1.0,1.0,0.824561,0.649123,0.473684,0.402193,0.402193,0.402193,0.560526,0.71886,0.877193,0.877193,0.701754,0.526316,0.350877,0.175439,0.0,0.0,0.0,0.0],\"naive_cost\":46923.0263,\"yesterday_error_stats\":{\"mean\":-17.7912,\"std\":13.0406,\"max_positive\":3.0743,\"max_negative\":-48.7604}}", "assistant_message": "{\"soc\":[0.5,0.324561,0.324561,0.482895,0.641228,0.799561,0.957895,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.877193,0.701754,0.526316,0.350877,0.175439,0.0,0.0,0.0],\"cost\":29962.4494,\"improvement_over_naive\":16960.5769}", "actual_prices_today": [47.86, 44.52, 41.42, 38.67, 38.23, 39.48, 43.15, 43.77, 45.37, 46.44, 46.44, 48.0, 48.15, 45.76, 45.84, 45.46, 48.34, 55.12, 56.2, 56.42, 55.14, 53.84, 46.48, 43.18], "actual_demand_today": [25.2, 23.42, 22.35, 21.83, 21.76, 22.32, 23.75, 25.19, 26.84, 28.62, 29.23, 29.41, 29.23, 27.87, 27.6, 28.14, 29.56, 32.27, 33.19, 33.83, 32.79, 30.99, 28.65, 26.17], "forecast_prices_today": [55.67568033, 53.07239038, 51.00181992, 47.36623821, 47.40726052, 55.01003384, 68.31150847, 73.14173882, 83.79392558, 65.41623894, 65.25547287, 59.10891214, 56.47060763, 51.56733, 55.1069477, 62.44495389, 69.66330869, 80.99666667, 74.2241846, 74.52112903, 68.36196225, 64.14167772, 63.01678135, 57.17275082], "milp_actual_soc": [0.5, 0.32456140350877194, 0.32456140350877194, 0.48289473684210527, 0.6412280701754386, 0.7995614035087719, 0.9578947368421052, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.8771929824561403, 0.7017543859649122, 0.5263157894736842, 0.3508771929824561, 0.17543859649122806, 0.0, 0.0, 0.0], "milp_actual_cost": 29962.449374912285, "milp_forecast_soc": [0.5, 0.5, 0.525, 0.6833333333333333, 0.8416666666666667, 1.0, 1.0, 0.8245614035087719, 0.6491228070175439, 0.4736842105263158, 0.4021929824561403, 0.4021929824561403, 0.4021929824561403, 0.5605263157894737, 0.718859649122807, 0.8771929824561403, 0.8771929824561403, 0.7017543859649122, 0.5263157894736842, 0.3508771929824561, 0.17543859649122806, 0.0, 0.0, 0.0, 0.0], "milp_forecast_cost": 46923.02627718287, "battery": {"capacity_kwh": 43.13, "cmax_kw": 7.1883333333333335, "dmax_kw": 7.1883333333333335, "eta_c": 0.95, "eta_d": 0.95, "soc_init": 0.5, "soc_target": 0.0}}

# TEST_EXAMPLE = {"date": "2019-01-12", "user_message": "{\"capacity_kwh\":43.13,\"max_c_kw\":5.39125,\"max_d_kw\":5.39125,\"eta_c\":0.95,\"eta_d\":0.95,\"soc0\":0.5,\"soc_min\":0.0,\"soc_max\":1.0,\"soc_target\":0.0,\"dt_hours\":1.0,\"allow_export\":true,\"forecast_prices_today\":[55.6757,53.0724,51.0018,47.3662,47.4073,55.01,68.3115,73.1417,83.7939,65.4162,65.2555,59.1089,56.4706,51.5673,55.1069,62.445,69.6633,80.9967,74.2242,74.5211,68.362,64.1417,63.0168,57.1728],\"forecast_demand_today\":[27.2116,25.6195,24.584,24.2211,24.1777,24.8347,26.1967,27.9592,34.9061,34.6584,34.9736,33.8194,34.1818,36.3375,37.4211,34.6604,35.1692,37.8039,39.1928,39.4898,37.932,35.106,32.2228,30.2691],\"actual_prices_yesterday\":[40.29,39.1,38.28,36.53,36.53,36.02,37.47,38.05,37.9,40.72,41.67,41.67,42.75,41.61,41.0,44.01,46.2,53.05,51.34,49.55,50.63,46.76,42.71,37.47],\"forecast_prices_yesterday\":[47.0389,43.3513,42.6604,36.0262,33.4557,43.6875,49.4515,78.9105,86.6604,58.7368,69.0214,65.0664,55.1343,59.7509,60.3883,80.1915,83.6062,79.62,69.465,74.5211,60.3787,56.3515,55.5102,49.3141],\"naive_soc\":[0.5,0.5,0.61875,0.7375,0.85625,0.975,1.0,0.868421,0.736842,0.605263,0.473684,0.342105,0.342105,0.460855,0.579605,0.698355,0.698355,0.566776,0.435197,0.303618,0.172039,0.040461,0.0,0.0,0.0],\"naive_cost\":47102.6947,\"yesterday_error_stats\":{\"mean\":-17.7912,\"std\":13.0406,\"max_positive\":3.0743,\"max_negative\":-48.7604}}", "assistant_message": "{\"soc\":[0.5,0.40625,0.40625,0.525,0.64375,0.7625,0.88125,1.0,1.0,1.0,1.0,1.0,0.921053,0.789474,0.789474,0.789474,0.789474,0.657895,0.526316,0.394737,0.263158,0.131579,0.0,0.0,0.0],\"cost\":30054.2661,\"improvement_over_naive\":17048.4286}", "actual_prices_today": [47.86, 44.52, 41.42, 38.67, 38.23, 39.48, 43.15, 43.77, 45.37, 46.44, 46.44, 48.0, 48.15, 45.76, 45.84, 45.46, 48.34, 55.12, 56.2, 56.42, 55.14, 53.84, 46.48, 43.18], "actual_demand_today": [25.2, 23.42, 22.35, 21.83, 21.76, 22.32, 23.75, 25.19, 26.84, 28.62, 29.23, 29.41, 29.23, 27.87, 27.6, 28.14, 29.56, 32.27, 33.19, 33.83, 32.79, 30.99, 28.65, 26.17], "forecast_prices_today": [55.67568033, 53.07239038, 51.00181992, 47.36623821, 47.40726052, 55.01003384, 68.31150847, 73.14173882, 83.79392558, 65.41623894, 65.25547287, 59.10891214, 56.47060763, 51.56733, 55.1069477, 62.44495389, 69.66330869, 80.99666667, 74.2241846, 74.52112903, 68.36196225, 64.14167772, 63.01678135, 57.17275082], "milp_actual_soc": [0.5, 0.4062499999999999, 0.4062499999999999, 0.5249999999999999, 0.6437499999999999, 0.7625, 0.88125, 1.0, 1.0, 1.0, 1.0, 1.0, 0.9210526315789472, 0.7894736842105262, 0.7894736842105262, 0.7894736842105262, 0.7894736842105262, 0.6578947368421052, 0.5263157894736842, 0.39473684210526316, 0.2631578947368421, 0.13157894736842105, 0.0, 0.0, 0.0], "milp_actual_cost": 30054.266102187503, "milp_forecast_soc": [0.5, 0.5, 0.61875, 0.7375, 0.8562500000000001, 0.9750000000000001, 1.0, 0.868421052631579, 0.736842105263158, 0.605263157894737, 0.47368421052631593, 0.3421052631578949, 0.3421052631578949, 0.46085526315789493, 0.579605263157895, 0.698355263157895, 0.698355263157895, 0.566776315789474, 0.43519736842105294, 0.3036184210526319, 0.17203947368421088, 0.040460526315789835, 0.0, 0.0, 0.0], "milp_forecast_cost": 47102.69471963732, "battery": {"capacity_kwh": 43.13, "cmax_kw": 5.39125, "dmax_kw": 5.39125, "eta_c": 0.95, "eta_d": 0.95, "soc_init": 0.5, "soc_target": 0.0}}

# TEST_EXAMPLE = {"date": "2019-01-12", "user_message": "{\"capacity_kwh\":43.13,\"max_c_kw\":10.7825,\"max_d_kw\":10.7825,\"eta_c\":0.9,\"eta_d\":0.9,\"soc0\":0.5,\"soc_min\":0.0,\"soc_max\":1.0,\"soc_target\":0.0,\"dt_hours\":1.0,\"allow_export\":true,\"forecast_prices_today\":[55.6757,53.0724,51.0018,47.3662,47.4073,55.01,68.3115,73.1417,83.7939,65.4162,65.2555,59.1089,56.4706,51.5673,55.1069,62.445,69.6633,80.9967,74.2242,74.5211,68.362,64.1417,63.0168,57.1728],\"forecast_demand_today\":[27.2116,25.6195,24.584,24.2211,24.1777,24.8347,26.1967,27.9592,34.9061,34.6584,34.9736,33.8194,34.1818,36.3375,37.4211,34.6604,35.1692,37.8039,39.1928,39.4898,37.932,35.106,32.2228,30.2691],\"actual_prices_yesterday\":[40.29,39.1,38.28,36.53,36.53,36.02,37.47,38.05,37.9,40.72,41.67,41.67,42.75,41.61,41.0,44.01,46.2,53.05,51.34,49.55,50.63,46.76,42.71,37.47],\"forecast_prices_yesterday\":[47.0389,43.3513,42.6604,36.0262,33.4557,43.6875,49.4515,78.9105,86.6604,58.7368,69.0214,65.0664,55.1343,59.7509,60.3883,80.1915,83.6062,79.62,69.465,74.5211,60.3787,56.3515,55.5102,49.3141],\"naive_soc\":[0.5,0.5,0.5,0.55,0.775,1.0,1.0,1.0,0.722222,0.444444,0.444444,0.444444,0.444444,0.444444,0.669444,0.894444,0.894444,0.833333,0.555556,0.277778,0.0,0.0,0.0,0.0,0.0],\"naive_cost\":47056.2865,\"yesterday_error_stats\":{\"mean\":-17.7912,\"std\":13.0406,\"max_positive\":3.0743,\"max_negative\":-48.7604}}", "assistant_message": "{\"soc\":[0.5,0.5,0.5,0.5,0.725,0.95,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.833333,0.555556,0.277778,0.0,0.0,0.0,0.0],\"cost\":30080.3755,\"improvement_over_naive\":16975.911}", "actual_prices_today": [47.86, 44.52, 41.42, 38.67, 38.23, 39.48, 43.15, 43.77, 45.37, 46.44, 46.44, 48.0, 48.15, 45.76, 45.84, 45.46, 48.34, 55.12, 56.2, 56.42, 55.14, 53.84, 46.48, 43.18], "actual_demand_today": [25.2, 23.42, 22.35, 21.83, 21.76, 22.32, 23.75, 25.19, 26.84, 28.62, 29.23, 29.41, 29.23, 27.87, 27.6, 28.14, 29.56, 32.27, 33.19, 33.83, 32.79, 30.99, 28.65, 26.17], "forecast_prices_today": [55.67568033, 53.07239038, 51.00181992, 47.36623821, 47.40726052, 55.01003384, 68.31150847, 73.14173882, 83.79392558, 65.41623894, 65.25547287, 59.10891214, 56.47060763, 51.56733, 55.1069477, 62.44495389, 69.66330869, 80.99666667, 74.2241846, 74.52112903, 68.36196225, 64.14167772, 63.01678135, 57.17275082], "milp_actual_soc": [0.5, 0.5, 0.5, 0.5, 0.725, 0.95, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.8333333333333334, 0.5555555555555556, 0.2777777777777778, 0.0, 0.0, 0.0, 0.0], "milp_actual_cost": 30080.375476666668, "milp_forecast_soc": [0.5, 0.5, 0.5, 0.55, 0.775, 1.0, 1.0, 1.0, 0.7222222222222222, 0.4444444444444444, 0.4444444444444444, 0.4444444444444444, 0.4444444444444444, 0.4444444444444444, 0.6694444444444444, 0.8944444444444445, 0.8944444444444445, 0.8333333333333334, 0.5555555555555556, 0.2777777777777778, 0.0, 0.0, 0.0, 0.0, 0.0], "milp_forecast_cost": 47056.286466883656, "battery": {"capacity_kwh": 43.13, "cmax_kw": 10.7825, "dmax_kw": 10.7825, "eta_c": 0.9, "eta_d": 0.9, "soc_init": 0.5, "soc_target": 0.0}}

# TEST_EXAMPLE = {"date": "2019-01-12", "user_message": "{\"capacity_kwh\":43.13,\"max_c_kw\":10.7825,\"max_d_kw\":10.7825,\"eta_c\":0.85,\"eta_d\":0.85,\"soc0\":0.5,\"soc_min\":0.0,\"soc_max\":1.0,\"soc_target\":0.0,\"dt_hours\":1.0,\"allow_export\":true,\"forecast_prices_today\":[55.6757,53.0724,51.0018,47.3662,47.4073,55.01,68.3115,73.1417,83.7939,65.4162,65.2555,59.1089,56.4706,51.5673,55.1069,62.445,69.6633,80.9967,74.2242,74.5211,68.362,64.1417,63.0168,57.1728],\"forecast_demand_today\":[27.2116,25.6195,24.584,24.2211,24.1777,24.8347,26.1967,27.9592,34.9061,34.6584,34.9736,33.8194,34.1818,36.3375,37.4211,34.6604,35.1692,37.8039,39.1928,39.4898,37.932,35.106,32.2228,30.2691],\"actual_prices_yesterday\":[40.29,39.1,38.28,36.53,36.53,36.02,37.47,38.05,37.9,40.72,41.67,41.67,42.75,41.61,41.0,44.01,46.2,53.05,51.34,49.55,50.63,46.76,42.71,37.47],\"forecast_prices_yesterday\":[47.0389,43.3513,42.6604,36.0262,33.4557,43.6875,49.4515,78.9105,86.6604,58.7368,69.0214,65.0664,55.1343,59.7509,60.3883,80.1915,83.6062,79.62,69.465,74.5211,60.3787,56.3515,55.5102,49.3141],\"naive_soc\":[0.5,0.5,0.5,0.575,0.7875,1.0,1.0,1.0,0.963971,0.669853,0.669853,0.669853,0.669853,0.669853,0.882353,0.882353,0.882353,0.882353,0.588235,0.294118,0.0,0.0,0.0,0.0,0.0],\"naive_cost\":47391.2746,\"yesterday_error_stats\":{\"mean\":-17.7912,\"std\":13.0406,\"max_positive\":3.0743,\"max_negative\":-48.7604}}", "assistant_message": "{\"soc\":[0.5,0.5,0.5,0.5,0.7125,0.925,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.882353,0.588235,0.294118,0.0,0.0,0.0,0.0],\"cost\":30254.8879,\"improvement_over_naive\":17136.3867}", "actual_prices_today": [47.86, 44.52, 41.42, 38.67, 38.23, 39.48, 43.15, 43.77, 45.37, 46.44, 46.44, 48.0, 48.15, 45.76, 45.84, 45.46, 48.34, 55.12, 56.2, 56.42, 55.14, 53.84, 46.48, 43.18], "actual_demand_today": [25.2, 23.42, 22.35, 21.83, 21.76, 22.32, 23.75, 25.19, 26.84, 28.62, 29.23, 29.41, 29.23, 27.87, 27.6, 28.14, 29.56, 32.27, 33.19, 33.83, 32.79, 30.99, 28.65, 26.17], "forecast_prices_today": [55.67568033, 53.07239038, 51.00181992, 47.36623821, 47.40726052, 55.01003384, 68.31150847, 73.14173882, 83.79392558, 65.41623894, 65.25547287, 59.10891214, 56.47060763, 51.56733, 55.1069477, 62.44495389, 69.66330869, 80.99666667, 74.2241846, 74.52112903, 68.36196225, 64.14167772, 63.01678135, 57.17275082], "milp_actual_soc": [0.5, 0.5, 0.5, 0.5, 0.7125, 0.925, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.8823529411764706, 0.5882352941176471, 0.29411764705882354, 0.0, 0.0, 0.0, 0.0], "milp_actual_cost": 30254.887913529412, "milp_forecast_soc": [0.5, 0.5, 0.5, 0.575, 0.7875, 1.0, 1.0, 1.0, 0.963970588235294, 0.6698529411764705, 0.6698529411764705, 0.6698529411764705, 0.6698529411764705, 0.6698529411764705, 0.8823529411764706, 0.8823529411764706, 0.8823529411764706, 0.8823529411764706, 0.5882352941176471, 0.29411764705882354, 0.0, 0.0, 0.0, 0.0, 0.0], "milp_forecast_cost": 47391.274595456714, "battery": {"capacity_kwh": 43.13, "cmax_kw": 10.7825, "dmax_kw": 10.7825, "eta_c": 0.85, "eta_d": 0.85, "soc_init": 0.5, "soc_target": 0.0}}

# TEST_EXAMPLE = {"date": "2019-01-12", "user_message": "{\"capacity_kwh\":43.13,\"max_c_kw\":10.7825,\"max_d_kw\":10.7825,\"eta_c\":0.8,\"eta_d\":0.8,\"soc0\":0.5,\"soc_min\":0.0,\"soc_max\":1.0,\"soc_target\":0.0,\"dt_hours\":1.0,\"allow_export\":true,\"forecast_prices_today\":[55.6757,53.0724,51.0018,47.3662,47.4073,55.01,68.3115,73.1417,83.7939,65.4162,65.2555,59.1089,56.4706,51.5673,55.1069,62.445,69.6633,80.9967,74.2242,74.5211,68.362,64.1417,63.0168,57.1728],\"forecast_demand_today\":[27.2116,25.6195,24.584,24.2211,24.1777,24.8347,26.1967,27.9592,34.9061,34.6584,34.9736,33.8194,34.1818,36.3375,37.4211,34.6604,35.1692,37.8039,39.1928,39.4898,37.932,35.106,32.2228,30.2691],\"actual_prices_yesterday\":[40.29,39.1,38.28,36.53,36.53,36.02,37.47,38.05,37.9,40.72,41.67,41.67,42.75,41.61,41.0,44.01,46.2,53.05,51.34,49.55,50.63,46.76,42.71,37.47],\"forecast_prices_yesterday\":[47.0389,43.3513,42.6604,36.0262,33.4557,43.6875,49.4515,78.9105,86.6604,58.7368,69.0214,65.0664,55.1343,59.7509,60.3883,80.1915,83.6062,79.62,69.465,74.5211,60.3787,56.3515,55.5102,49.3141],\"naive_soc\":[0.5,0.5,0.5,0.5,0.7,0.9,0.9,0.9,0.9,0.5875,0.5875,0.5875,0.5875,0.5875,0.5875,0.5875,0.5875,0.5875,0.275,0.275,0.0,0.0,0.0,0.0,0.0],\"naive_cost\":47634.5128,\"yesterday_error_stats\":{\"mean\":-17.7912,\"std\":13.0406,\"max_positive\":3.0743,\"max_negative\":-48.7604}}", "assistant_message": "{\"soc\":[0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.3125,0.0,0.0,0.0,0.0,0.0],\"cost\":30350.1393,\"improvement_over_naive\":17284.3736}", "actual_prices_today": [47.86, 44.52, 41.42, 38.67, 38.23, 39.48, 43.15, 43.77, 45.37, 46.44, 46.44, 48.0, 48.15, 45.76, 45.84, 45.46, 48.34, 55.12, 56.2, 56.42, 55.14, 53.84, 46.48, 43.18], "actual_demand_today": [25.2, 23.42, 22.35, 21.83, 21.76, 22.32, 23.75, 25.19, 26.84, 28.62, 29.23, 29.41, 29.23, 27.87, 27.6, 28.14, 29.56, 32.27, 33.19, 33.83, 32.79, 30.99, 28.65, 26.17], "forecast_prices_today": [55.67568033, 53.07239038, 51.00181992, 47.36623821, 47.40726052, 55.01003384, 68.31150847, 73.14173882, 83.79392558, 65.41623894, 65.25547287, 59.10891214, 56.47060763, 51.56733, 55.1069477, 62.44495389, 69.66330869, 80.99666667, 74.2241846, 74.52112903, 68.36196225, 64.14167772, 63.01678135, 57.17275082], "milp_actual_soc": [0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.3125, 0.0, 0.0, 0.0, 0.0, 0.0], "milp_actual_cost": 30350.13925, "milp_forecast_soc": [0.5, 0.5, 0.5, 0.5, 0.7, 0.9, 0.9, 0.9, 0.9, 0.5874999999999999, 0.5874999999999999, 0.5874999999999999, 0.5874999999999999, 0.5874999999999999, 0.5874999999999999, 0.5874999999999999, 0.5874999999999999, 0.5874999999999999, 0.2749999999999999, 0.2749999999999999, 0.0, 0.0, 0.0, 0.0, 0.0], "milp_forecast_cost": 47634.512816023256, "battery": {"capacity_kwh": 43.13, "cmax_kw": 10.7825, "dmax_kw": 10.7825, "eta_c": 0.8, "eta_d": 0.8, "soc_init": 0.5, "soc_target": 0.0}}

# TEST_EXAMPLE = {"date": "2019-01-12", "user_message": "{\"capacity_kwh\":43.13,\"max_c_kw\":10.7825,\"max_d_kw\":10.7825,\"eta_c\":0.95,\"eta_d\":0.95,\"soc0\":0.4,\"soc_min\":0.0,\"soc_max\":1.0,\"soc_target\":0.0,\"dt_hours\":1.0,\"allow_export\":true,\"forecast_prices_today\":[55.6757,53.0724,51.0018,47.3662,47.4073,55.01,68.3115,73.1417,83.7939,65.4162,65.2555,59.1089,56.4706,51.5673,55.1069,62.445,69.6633,80.9967,74.2242,74.5211,68.362,64.1417,63.0168,57.1728],\"forecast_demand_today\":[27.2116,25.6195,24.584,24.2211,24.1777,24.8347,26.1967,27.9592,34.9061,34.6584,34.9736,33.8194,34.1818,36.3375,37.4211,34.6604,35.1692,37.8039,39.1928,39.4898,37.932,35.106,32.2228,30.2691],\"actual_prices_yesterday\":[40.29,39.1,38.28,36.53,36.53,36.02,37.47,38.05,37.9,40.72,41.67,41.67,42.75,41.61,41.0,44.01,46.2,53.05,51.34,49.55,50.63,46.76,42.71,37.47],\"forecast_prices_yesterday\":[47.0389,43.3513,42.6604,36.0262,33.4557,43.6875,49.4515,78.9105,86.6604,58.7368,69.0214,65.0664,55.1343,59.7509,60.3883,80.1915,83.6062,79.62,69.465,74.5211,60.3787,56.3515,55.5102,49.3141],\"naive_soc\":[0.4,0.4,0.4,0.525,0.7625,1.0,1.0,0.736842,0.473684,0.210526,0.210526,0.210526,0.2875,0.525,0.7625,1.0,1.0,0.789474,0.526316,0.263158,0.0,0.0,0.0,0.0,0.0],\"naive_cost\":46866.7397,\"yesterday_error_stats\":{\"mean\":-17.7912,\"std\":13.0406,\"max_positive\":3.0743,\"max_negative\":-48.7604}}", "assistant_message": "{\"soc\":[0.4,0.136842,0.136842,0.2875,0.525,0.7625,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.789474,0.526316,0.263158,0.0,0.0,0.0,0.0],\"cost\":30059.8607,\"improvement_over_naive\":16806.879}", "actual_prices_today": [47.86, 44.52, 41.42, 38.67, 38.23, 39.48, 43.15, 43.77, 45.37, 46.44, 46.44, 48.0, 48.15, 45.76, 45.84, 45.46, 48.34, 55.12, 56.2, 56.42, 55.14, 53.84, 46.48, 43.18], "actual_demand_today": [25.2, 23.42, 22.35, 21.83, 21.76, 22.32, 23.75, 25.19, 26.84, 28.62, 29.23, 29.41, 29.23, 27.87, 27.6, 28.14, 29.56, 32.27, 33.19, 33.83, 32.79, 30.99, 28.65, 26.17], "forecast_prices_today": [55.67568033, 53.07239038, 51.00181992, 47.36623821, 47.40726052, 55.01003384, 68.31150847, 73.14173882, 83.79392558, 65.41623894, 65.25547287, 59.10891214, 56.47060763, 51.56733, 55.1069477, 62.44495389, 69.66330869, 80.99666667, 74.2241846, 74.52112903, 68.36196225, 64.14167772, 63.01678135, 57.17275082], "milp_actual_soc": [0.4, 0.13684210526315793, 0.13684210526315793, 0.2874999999999999, 0.5249999999999999, 0.7625, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.7894736842105263, 0.5263157894736842, 0.2631578947368421, 0.0, 0.0, 0.0, 0.0], "milp_actual_cost": 30059.860730000004, "milp_forecast_soc": [0.4, 0.4, 0.4, 0.5249999999999999, 0.7625, 1.0, 1.0, 0.736842105263158, 0.4736842105263159, 0.21052631578947378, 0.21052631578947378, 0.21052631578947378, 0.2874999999999999, 0.5249999999999999, 0.7625, 1.0, 1.0, 0.7894736842105263, 0.5263157894736842, 0.2631578947368421, 0.0, 0.0, 0.0, 0.0, 0.0], "milp_forecast_cost": 46866.73968663818, "battery": {"capacity_kwh": 43.13, "cmax_kw": 10.7825, "dmax_kw": 10.7825, "eta_c": 0.95, "eta_d": 0.95, "soc_init": 0.4, "soc_target": 0.0}}

# TEST_EXAMPLE = {"date": "2019-01-12", "user_message": "{\"capacity_kwh\":43.13,\"max_c_kw\":10.7825,\"max_d_kw\":10.7825,\"eta_c\":0.95,\"eta_d\":0.95,\"soc0\":0.2,\"soc_min\":0.0,\"soc_max\":1.0,\"soc_target\":0.0,\"dt_hours\":1.0,\"allow_export\":true,\"forecast_prices_today\":[55.6757,53.0724,51.0018,47.3662,47.4073,55.01,68.3115,73.1417,83.7939,65.4162,65.2555,59.1089,56.4706,51.5673,55.1069,62.445,69.6633,80.9967,74.2242,74.5211,68.362,64.1417,63.0168,57.1728],\"forecast_demand_today\":[27.2116,25.6195,24.584,24.2211,24.1777,24.8347,26.1967,27.9592,34.9061,34.6584,34.9736,33.8194,34.1818,36.3375,37.4211,34.6604,35.1692,37.8039,39.1928,39.4898,37.932,35.106,32.2228,30.2691],\"actual_prices_yesterday\":[40.29,39.1,38.28,36.53,36.53,36.02,37.47,38.05,37.9,40.72,41.67,41.67,42.75,41.61,41.0,44.01,46.2,53.05,51.34,49.55,50.63,46.76,42.71,37.47],\"forecast_prices_yesterday\":[47.0389,43.3513,42.6604,36.0262,33.4557,43.6875,49.4515,78.9105,86.6604,58.7368,69.0214,65.0664,55.1343,59.7509,60.3883,80.1915,83.6062,79.62,69.465,74.5211,60.3787,56.3515,55.5102,49.3141],\"naive_soc\":[0.2,0.2,0.2875,0.525,0.7625,1.0,1.0,0.736842,0.473684,0.210526,0.210526,0.210526,0.2875,0.525,0.7625,1.0,1.0,0.789474,0.526316,0.263158,0.0,0.0,0.0,0.0,0.0],\"naive_cost\":47338.0616,\"yesterday_error_stats\":{\"mean\":-17.7912,\"std\":13.0406,\"max_positive\":3.0743,\"max_negative\":-48.7604}}", "assistant_message": "{\"soc\":[0.2,0.0,0.0,0.2375,0.475,0.7125,0.95,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.789474,0.526316,0.263158,0.0,0.0,0.0,0.0],\"cost\":30444.9671,\"improvement_over_naive\":16893.0944}", "actual_prices_today": [47.86, 44.52, 41.42, 38.67, 38.23, 39.48, 43.15, 43.77, 45.37, 46.44, 46.44, 48.0, 48.15, 45.76, 45.84, 45.46, 48.34, 55.12, 56.2, 56.42, 55.14, 53.84, 46.48, 43.18], "actual_demand_today": [25.2, 23.42, 22.35, 21.83, 21.76, 22.32, 23.75, 25.19, 26.84, 28.62, 29.23, 29.41, 29.23, 27.87, 27.6, 28.14, 29.56, 32.27, 33.19, 33.83, 32.79, 30.99, 28.65, 26.17], "forecast_prices_today": [55.67568033, 53.07239038, 51.00181992, 47.36623821, 47.40726052, 55.01003384, 68.31150847, 73.14173882, 83.79392558, 65.41623894, 65.25547287, 59.10891214, 56.47060763, 51.56733, 55.1069477, 62.44495389, 69.66330869, 80.99666667, 74.2241846, 74.52112903, 68.36196225, 64.14167772, 63.01678135, 57.17275082], "milp_actual_soc": [0.2, 0.0, 0.0, 0.2375, 0.475, 0.7124999999999999, 0.95, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.7894736842105263, 0.5263157894736842, 0.2631578947368421, 0.0, 0.0, 0.0, 0.0], "milp_actual_cost": 30444.967138000004, "milp_forecast_soc": [0.2, 0.2, 0.2874999999999999, 0.5249999999999999, 0.7625, 1.0, 1.0, 0.736842105263158, 0.4736842105263159, 0.21052631578947378, 0.21052631578947378, 0.21052631578947378, 0.2874999999999999, 0.5249999999999999, 0.7625, 1.0, 1.0, 0.7894736842105263, 0.5263157894736842, 0.2631578947368421, 0.0, 0.0, 0.0, 0.0, 0.0], "milp_forecast_cost": 47338.061552664134, "battery": {"capacity_kwh": 43.13, "cmax_kw": 10.7825, "dmax_kw": 10.7825, "eta_c": 0.95, "eta_d": 0.95, "soc_init": 0.2, "soc_target": 0.0}}

# TEST_EXAMPLE = {"date": "2019-01-12", "user_message": "{\"capacity_kwh\":43.13,\"max_c_kw\":10.7825,\"max_d_kw\":10.7825,\"eta_c\":0.95,\"eta_d\":0.95,\"soc0\":0.6,\"soc_min\":0.0,\"soc_max\":1.0,\"soc_target\":0.0,\"dt_hours\":1.0,\"allow_export\":true,\"forecast_prices_today\":[55.6757,53.0724,51.0018,47.3662,47.4073,55.01,68.3115,73.1417,83.7939,65.4162,65.2555,59.1089,56.4706,51.5673,55.1069,62.445,69.6633,80.9967,74.2242,74.5211,68.362,64.1417,63.0168,57.1728],\"forecast_demand_today\":[27.2116,25.6195,24.584,24.2211,24.1777,24.8347,26.1967,27.9592,34.9061,34.6584,34.9736,33.8194,34.1818,36.3375,37.4211,34.6604,35.1692,37.8039,39.1928,39.4898,37.932,35.106,32.2228,30.2691],\"actual_prices_yesterday\":[40.29,39.1,38.28,36.53,36.53,36.02,37.47,38.05,37.9,40.72,41.67,41.67,42.75,41.61,41.0,44.01,46.2,53.05,51.34,49.55,50.63,46.76,42.71,37.47],\"forecast_prices_yesterday\":[47.0389,43.3513,42.6604,36.0262,33.4557,43.6875,49.4515,78.9105,86.6604,58.7368,69.0214,65.0664,55.1343,59.7509,60.3883,80.1915,83.6062,79.62,69.465,74.5211,60.3787,56.3515,55.5102,49.3141],\"naive_soc\":[0.6,0.525,0.525,0.525,0.7625,1.0,1.0,0.736842,0.473684,0.210526,0.210526,0.210526,0.2875,0.525,0.7625,1.0,1.0,0.789474,0.526316,0.263158,0.0,0.0,0.0,0.0,0.0],\"naive_cost\":46406.2123,\"yesterday_error_stats\":{\"mean\":-17.7912,\"std\":13.0406,\"max_positive\":3.0743,\"max_negative\":-48.7604}}", "assistant_message": "{\"soc\":[0.6,0.336842,0.2875,0.2875,0.525,0.7625,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.789474,0.526316,0.263158,0.0,0.0,0.0,0.0],\"cost\":29686.5465,\"improvement_over_naive\":16719.6658}", "actual_prices_today": [47.86, 44.52, 41.42, 38.67, 38.23, 39.48, 43.15, 43.77, 45.37, 46.44, 46.44, 48.0, 48.15, 45.76, 45.84, 45.46, 48.34, 55.12, 56.2, 56.42, 55.14, 53.84, 46.48, 43.18], "actual_demand_today": [25.2, 23.42, 22.35, 21.83, 21.76, 22.32, 23.75, 25.19, 26.84, 28.62, 29.23, 29.41, 29.23, 27.87, 27.6, 28.14, 29.56, 32.27, 33.19, 33.83, 32.79, 30.99, 28.65, 26.17], "forecast_prices_today": [55.67568033, 53.07239038, 51.00181992, 47.36623821, 47.40726052, 55.01003384, 68.31150847, 73.14173882, 83.79392558, 65.41623894, 65.25547287, 59.10891214, 56.47060763, 51.56733, 55.1069477, 62.44495389, 69.66330869, 80.99666667, 74.2241846, 74.52112903, 68.36196225, 64.14167772, 63.01678135, 57.17275082], "milp_actual_soc": [0.6, 0.3368421052631579, 0.2874999999999999, 0.2874999999999999, 0.5249999999999999, 0.7625, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.7894736842105263, 0.5263157894736842, 0.2631578947368421, 0.0, 0.0, 0.0, 0.0], "milp_actual_cost": 29686.546461250004, "milp_forecast_soc": [0.6, 0.5249999999999999, 0.5249999999999999, 0.5249999999999999, 0.7625, 1.0, 1.0, 0.736842105263158, 0.4736842105263159, 0.21052631578947378, 0.21052631578947378, 0.21052631578947378, 0.2874999999999999, 0.5249999999999999, 0.7625, 1.0, 1.0, 0.7894736842105263, 0.5263157894736842, 0.2631578947368421, 0.0, 0.0, 0.0, 0.0, 0.0], "milp_forecast_cost": 46406.21229699209, "battery": {"capacity_kwh": 43.13, "cmax_kw": 10.7825, "dmax_kw": 10.7825, "eta_c": 0.95, "eta_d": 0.95, "soc_init": 0.6, "soc_target": 0.0}}

# TEST_EXAMPLE = {"date": "2019-01-12", "user_message": "{\"capacity_kwh\":43.13,\"max_c_kw\":10.7825,\"max_d_kw\":10.7825,\"eta_c\":0.95,\"eta_d\":0.95,\"soc0\":0.8,\"soc_min\":0.0,\"soc_max\":1.0,\"soc_target\":0.0,\"dt_hours\":1.0,\"allow_export\":true,\"forecast_prices_today\":[55.6757,53.0724,51.0018,47.3662,47.4073,55.01,68.3115,73.1417,83.7939,65.4162,65.2555,59.1089,56.4706,51.5673,55.1069,62.445,69.6633,80.9967,74.2242,74.5211,68.362,64.1417,63.0168,57.1728],\"forecast_demand_today\":[27.2116,25.6195,24.584,24.2211,24.1777,24.8347,26.1967,27.9592,34.9061,34.6584,34.9736,33.8194,34.1818,36.3375,37.4211,34.6604,35.1692,37.8039,39.1928,39.4898,37.932,35.106,32.2228,30.2691],\"actual_prices_yesterday\":[40.29,39.1,38.28,36.53,36.53,36.02,37.47,38.05,37.9,40.72,41.67,41.67,42.75,41.61,41.0,44.01,46.2,53.05,51.34,49.55,50.63,46.76,42.71,37.47],\"forecast_prices_yesterday\":[47.0389,43.3513,42.6604,36.0262,33.4557,43.6875,49.4515,78.9105,86.6604,58.7368,69.0214,65.0664,55.1343,59.7509,60.3883,80.1915,83.6062,79.62,69.465,74.5211,60.3787,56.3515,55.5102,49.3141],\"naive_soc\":[0.8,0.536842,0.525,0.525,0.7625,1.0,1.0,0.736842,0.473684,0.210526,0.210526,0.210526,0.2875,0.525,0.7625,1.0,1.0,0.789474,0.526316,0.263158,0.0,0.0,0.0,0.0,0.0],\"naive_cost\":45951.2299,\"yesterday_error_stats\":{\"mean\":-17.7912,\"std\":13.0406,\"max_positive\":3.0743,\"max_negative\":-48.7604}}", "assistant_message": "{\"soc\":[0.8,0.536842,0.2875,0.2875,0.525,0.7625,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.789474,0.526316,0.263158,0.0,0.0,0.0,0.0],\"cost\":29321.7184,\"improvement_over_naive\":16629.5115}", "actual_prices_today": [47.86, 44.52, 41.42, 38.67, 38.23, 39.48, 43.15, 43.77, 45.37, 46.44, 46.44, 48.0, 48.15, 45.76, 45.84, 45.46, 48.34, 55.12, 56.2, 56.42, 55.14, 53.84, 46.48, 43.18], "actual_demand_today": [25.2, 23.42, 22.35, 21.83, 21.76, 22.32, 23.75, 25.19, 26.84, 28.62, 29.23, 29.41, 29.23, 27.87, 27.6, 28.14, 29.56, 32.27, 33.19, 33.83, 32.79, 30.99, 28.65, 26.17], "forecast_prices_today": [55.67568033, 53.07239038, 51.00181992, 47.36623821, 47.40726052, 55.01003384, 68.31150847, 73.14173882, 83.79392558, 65.41623894, 65.25547287, 59.10891214, 56.47060763, 51.56733, 55.1069477, 62.44495389, 69.66330869, 80.99666667, 74.2241846, 74.52112903, 68.36196225, 64.14167772, 63.01678135, 57.17275082], "milp_actual_soc": [0.8, 0.536842105263158, 0.2874999999999999, 0.2874999999999999, 0.5249999999999999, 0.7625, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.7894736842105263, 0.5263157894736842, 0.2631578947368421, 0.0, 0.0, 0.0, 0.0], "milp_actual_cost": 29321.718417250002, "milp_forecast_soc": [0.8, 0.536842105263158, 0.5249999999999999, 0.5249999999999999, 0.7625, 1.0, 1.0, 0.736842105263158, 0.4736842105263159, 0.21052631578947378, 0.21052631578947378, 0.21052631578947378, 0.2874999999999999, 0.5249999999999999, 0.7625, 1.0, 1.0, 0.7894736842105263, 0.5263157894736842, 0.2631578947368421, 0.0, 0.0, 0.0, 0.0, 0.0], "milp_forecast_cost": 45951.2299482167, "battery": {"capacity_kwh": 43.13, "cmax_kw": 10.7825, "dmax_kw": 10.7825, "eta_c": 0.95, "eta_d": 0.95, "soc_init": 0.8, "soc_target": 0.0}}

# TEST_EXAMPLE = {"date": "2019-01-12", "user_message": "{\"capacity_kwh\":43.13,\"max_c_kw\":10.7825,\"max_d_kw\":10.7825,\"eta_c\":0.95,\"eta_d\":0.95,\"soc0\":0.5,\"soc_min\":0.0,\"soc_max\":1.0,\"soc_target\":0.0,\"dt_hours\":1.0,\"allow_export\":true,\"forecast_prices_today\":[55.6757,53.0724,51.0018,47.3662,47.4073,55.01,68.3115,73.1417,83.7939,65.4162,65.2555,59.1089,56.4706,51.5673,55.1069,62.445,69.6633,80.9967,74.2242,74.5211,68.362,64.1417,63.0168,57.1728],\"forecast_demand_today\":[27.2116,25.6195,24.584,24.2211,24.1777,24.8347,26.1967,27.9592,34.9061,34.6584,34.9736,33.8194,34.1818,36.3375,37.4211,34.6604,35.1692,37.8039,39.1928,39.4898,37.932,35.106,32.2228,30.2691],\"actual_prices_yesterday\":[40.29,39.1,38.28,36.53,36.53,36.02,37.47,38.05,37.9,40.72,41.67,41.67,42.75,41.61,41.0,44.01,46.2,53.05,51.34,49.55,50.63,46.76,42.71,37.47],\"forecast_prices_yesterday\":[47.0389,43.3513,42.6604,36.0262,33.4557,43.6875,49.4515,78.9105,86.6604,58.7368,69.0214,65.0664,55.1343,59.7509,60.3883,80.1915,83.6062,79.62,69.465,74.5211,60.3787,56.3515,55.5102,49.3141],\"naive_soc\":[0.5,0.5,0.5,0.525,0.7625,1.0,1.0,0.736842,0.473684,0.210526,0.210526,0.210526,0.2875,0.525,0.7625,1.0,1.0,0.789474,0.526316,0.263158,0.0,0.0,0.0,0.0,0.0],\"naive_cost\":46635.1914,\"yesterday_error_stats\":{\"mean\":-17.7912,\"std\":13.0406,\"max_positive\":3.0743,\"max_negative\":-48.7604}}", "assistant_message": "{\"soc\":[0.5,0.236842,0.236842,0.2875,0.525,0.7625,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.789474,0.526316,0.263158,0.0,0.0,0.0,0.0],\"cost\":29871.8139,\"improvement_over_naive\":16763.3775}", "actual_prices_today": [47.86, 44.52, 41.42, 38.67, 38.23, 39.48, 43.15, 43.77, 45.37, 46.44, 46.44, 48.0, 48.15, 45.76, 45.84, 45.46, 48.34, 55.12, 56.2, 56.42, 55.14, 53.84, 46.48, 43.18], "actual_demand_today": [25.2, 23.42, 22.35, 21.83, 21.76, 22.32, 23.75, 25.19, 26.84, 28.62, 29.23, 29.41, 29.23, 27.87, 27.6, 28.14, 29.56, 32.27, 33.19, 33.83, 32.79, 30.99, 28.65, 26.17], "forecast_prices_today": [55.67568033, 53.07239038, 51.00181992, 47.36623821, 47.40726052, 55.01003384, 68.31150847, 73.14173882, 83.79392558, 65.41623894, 65.25547287, 59.10891214, 56.47060763, 51.56733, 55.1069477, 62.44495389, 69.66330869, 80.99666667, 74.2241846, 74.52112903, 68.36196225, 64.14167772, 63.01678135, 57.17275082], "milp_actual_soc": [0.5, 0.2368421052631579, 0.2368421052631579, 0.2874999999999999, 0.5249999999999999, 0.7625, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.7894736842105263, 0.5263157894736842, 0.2631578947368421, 0.0, 0.0, 0.0, 0.0], "milp_actual_cost": 29871.813930000004, "milp_forecast_soc": [0.5, 0.5, 0.5, 0.5249999999999999, 0.7625, 1.0, 1.0, 0.736842105263158, 0.4736842105263159, 0.21052631578947378, 0.21052631578947378, 0.21052631578947378, 0.2874999999999999, 0.5249999999999999, 0.7625, 1.0, 1.0, 0.7894736842105263, 0.5263157894736842, 0.2631578947368421, 0.0, 0.0, 0.0, 0.0, 0.0], "milp_forecast_cost": 46635.19142420138, "battery": {"capacity_kwh": 43.13, "cmax_kw": 10.7825, "dmax_kw": 10.7825, "eta_c": 0.95, "eta_d": 0.95, "soc_init": 0.5, "soc_target": 0.0}}

# TEST_EXAMPLE = {"date": "2019-12-01", "user_message": "{\"capacity_kwh\":43.13,\"max_c_kw\":10.7825,\"max_d_kw\":10.7825,\"eta_c\":0.95,\"eta_d\":0.95,\"soc0\":0.5,\"soc_min\":0.0,\"soc_max\":1.0,\"soc_target\":0.0,\"dt_hours\":1.0,\"allow_export\":true,\"forecast_prices_today\":[46.2343,41.5005,39.3378,38.0062,39.032,38.2413,42.1012,47.0631,53.1575,52.2232,56.221,54.2465,50.4386,52.6064,49.8465,57.7811,62.4085,67.3157,56.2016,60.3596,62.0148,62.2724,53.1024,45.9017],\"forecast_demand_today\":[26.8617,25.2153,23.9225,23.8409,23.5584,24.0203,27.171,29.1947,29.2061,29.9812,30.4728,33.6008,30.9495,30.4918,28.9571,29.4635,33.2702,35.6648,36.3692,36.6453,34.4189,32.3334,29.9367,26.8444],\"actual_prices_yesterday\":[43.59,38.05,36.22,35.39,35.39,36.03,45.83,52.5,57.0,51.73,50.0,46.73,45.0,44.0,45.47,53.91,60.85,67.65,66.73,67.93,64.78,59.67,55.01,48.06],\"forecast_prices_yesterday\":[57.0772,55.0874,51.5993,47.2783,45.9009,51.9436,57.7168,64.2002,64.7076,64.3879,63.2415,63.5237,62.6621,61.6693,54.7222,63.4767,67.4161,79.62,74.6026,74.5211,69.3621,64.7601,61.8262,57.9785],\"naive_soc\":[0.5,0.236842,0.236842,0.2875,0.525,0.7625,1.0,1.0,1.0,1.0,1.0,0.736842,0.736842,0.7625,0.7625,1.0,1.0,0.736842,0.473684,0.473684,0.473684,0.263158,0.0,0.0,0.0],\"naive_cost\":35379.4368,\"yesterday_error_stats\":{\"mean\":-11.3234,\"std\":3.9608,\"max_positive\":-4.5821,\"max_negative\":-17.6693}}", "assistant_message": "{\"soc\":[0.5,0.5,0.5,0.5,0.525,0.7625,1.0,1.0,1.0,0.736842,0.473684,0.473684,0.473684,0.473684,0.711184,0.948684,1.0,1.0,0.736842,0.473684,0.210526,0.0,0.0,0.0,0.0],\"cost\":49517.1645,\"improvement_over_naive\":-14137.7277}", "actual_prices_today": [65.1, 62.12, 59.05, 59.0, 56.63, 56.63, 61.66, 66.61, 73.0, 73.0, 65.0, 63.82, 62.19, 59.5, 56.99, 61.92, 65.63, 82.47, 82.79, 83.5, 78.62, 72.94, 64.53, 60.32], "actual_demand_today": [28.68, 27.12, 25.99, 25.62, 25.66, 26.06, 28.58, 32.44, 35.25, 36.63, 36.22, 35.17, 33.26, 31.54, 31.52, 32.14, 33.53, 37.37, 38.78, 38.96, 36.4, 33.51, 30.89, 28.02], "forecast_prices_today": [46.23430042, 41.50053732, 39.33778151, 38.00616949, 39.03200476, 38.24127885, 42.10119966, 47.06307124, 53.15752937, 52.22315992, 56.22102261, 54.24650748, 50.43856725, 52.60644333, 49.84653038, 57.78106286, 62.40845962, 67.31571429, 56.20156832, 60.35956522, 62.01479177, 62.27242035, 53.10237809, 45.90174945], "milp_actual_soc": [0.5, 0.5, 0.5, 0.5, 0.5249999999999999, 0.7625, 1.0, 1.0, 1.0, 0.736842105263158, 0.4736842105263159, 0.4736842105263159, 0.4736842105263159, 0.4736842105263159, 0.7111842105263159, 0.9486842105263158, 1.0, 1.0, 0.736842105263158, 0.4736842105263159, 0.21052631578947378, 0.0, 0.0, 0.0, 0.0], "milp_actual_cost": 49517.16446026317, "milp_forecast_soc": [0.5, 0.2368421052631579, 0.2368421052631579, 0.2874999999999999, 0.5249999999999999, 0.7625, 1.0, 1.0, 1.0, 1.0, 1.0, 0.736842105263158, 0.736842105263158, 0.7625, 0.7625, 1.0, 1.0, 0.736842105263158, 0.4736842105263159, 0.4736842105263159, 0.4736842105263159, 0.2631578947368421, 0.0, 0.0, 0.0], "milp_forecast_cost": 35379.43676891254, "battery": {"capacity_kwh": 43.13, "cmax_kw": 10.7825, "dmax_kw": 10.7825, "eta_c": 0.95, "eta_d": 0.95, "soc_init": 0.5, "soc_target": 0.0}}

# TEST_EXAMPLE = {"date": "2019-12-01", "user_message": "{\"capacity_kwh\":43.13,\"max_c_kw\":5.39125,\"max_d_kw\":5.39125,\"eta_c\":0.95,\"eta_d\":0.95,\"soc0\":0.5,\"soc_min\":0.0,\"soc_max\":1.0,\"soc_target\":0.0,\"dt_hours\":1.0,\"allow_export\":true,\"forecast_prices_today\":[46.2343,41.5005,39.3378,38.0062,39.032,38.2413,42.1012,47.0631,53.1575,52.2232,56.221,54.2465,50.4386,52.6064,49.8465,57.7811,62.4085,67.3157,56.2016,60.3596,62.0148,62.2724,53.1024,45.9017],\"forecast_demand_today\":[26.8617,25.2153,23.9225,23.8409,23.5584,24.0203,27.171,29.1947,29.2061,29.9812,30.4728,33.6008,30.9495,30.4918,28.9571,29.4635,33.2702,35.6648,36.3692,36.6453,34.4189,32.3334,29.9367,26.8444],\"actual_prices_yesterday\":[43.59,38.05,36.22,35.39,35.39,36.03,45.83,52.5,57.0,51.73,50.0,46.73,45.0,44.0,45.47,53.91,60.85,67.65,66.73,67.93,64.78,59.67,55.01,48.06],\"forecast_prices_yesterday\":[57.0772,55.0874,51.5993,47.2783,45.9009,51.9436,57.7168,64.2002,64.7076,64.3879,63.2415,63.5237,62.6621,61.6693,54.7222,63.4767,67.4161,79.62,74.6026,74.5211,69.3621,64.7601,61.8262,57.9785],\"naive_soc\":[0.5,0.40625,0.525,0.64375,0.7625,0.88125,1.0,1.0,1.0,1.0,1.0,0.868421,0.868421,0.868421,0.868421,0.921053,0.789474,0.657895,0.526316,0.394737,0.263158,0.131579,0.0,0.0,0.0],\"naive_cost\":35545.9731,\"yesterday_error_stats\":{\"mean\":-11.3234,\"std\":3.9608,\"max_positive\":-4.5821,\"max_negative\":-17.6693}}", "assistant_message": "{\"soc\":[0.5,0.5,0.5,0.61875,0.7375,0.85625,0.975,0.975,0.843421,0.711842,0.580263,0.580263,0.580263,0.580263,0.580263,0.699013,0.699013,0.657895,0.526316,0.394737,0.263158,0.131579,0.0,0.0,0.0],\"cost\":49902.6059,\"improvement_over_naive\":-14356.6329}", "actual_prices_today": [65.1, 62.12, 59.05, 59.0, 56.63, 56.63, 61.66, 66.61, 73.0, 73.0, 65.0, 63.82, 62.19, 59.5, 56.99, 61.92, 65.63, 82.47, 82.79, 83.5, 78.62, 72.94, 64.53, 60.32], "actual_demand_today": [28.68, 27.12, 25.99, 25.62, 25.66, 26.06, 28.58, 32.44, 35.25, 36.63, 36.22, 35.17, 33.26, 31.54, 31.52, 32.14, 33.53, 37.37, 38.78, 38.96, 36.4, 33.51, 30.89, 28.02], "forecast_prices_today": [46.23430042, 41.50053732, 39.33778151, 38.00616949, 39.03200476, 38.24127885, 42.10119966, 47.06307124, 53.15752937, 52.22315992, 56.22102261, 54.24650748, 50.43856725, 52.60644333, 49.84653038, 57.78106286, 62.40845962, 67.31571429, 56.20156832, 60.35956522, 62.01479177, 62.27242035, 53.10237809, 45.90174945], "milp_actual_soc": [0.5, 0.5, 0.5, 0.61875, 0.7375, 0.8562500000000001, 0.9750000000000001, 0.9750000000000001, 0.8434210526315791, 0.711842105263158, 0.580263157894737, 0.580263157894737, 0.580263157894737, 0.580263157894737, 0.580263157894737, 0.6990131578947371, 0.6990131578947371, 0.6578947368421052, 0.5263157894736842, 0.39473684210526316, 0.2631578947368421, 0.13157894736842105, 0.0, 0.0, 0.0], "milp_actual_cost": 49902.60594453125, "milp_forecast_soc": [0.5, 0.4062499999999999, 0.5249999999999999, 0.6437499999999999, 0.7625, 0.88125, 1.0, 1.0, 1.0, 1.0, 1.0, 0.868421052631579, 0.868421052631579, 0.868421052631579, 0.868421052631579, 0.9210526315789472, 0.7894736842105262, 0.6578947368421052, 0.5263157894736842, 0.39473684210526316, 0.2631578947368421, 0.13157894736842105, 0.0, 0.0, 0.0], "milp_forecast_cost": 35545.97307057377, "battery": {"capacity_kwh": 43.13, "cmax_kw": 5.39125, "dmax_kw": 5.39125, "eta_c": 0.95, "eta_d": 0.95, "soc_init": 0.5, "soc_target": 0.0}}

# TEST_EXAMPLE = {"date": "2019-12-01", "user_message": "{\"capacity_kwh\":43.13,\"max_c_kw\":7.1883333333333335,\"max_d_kw\":7.1883333333333335,\"eta_c\":0.95,\"eta_d\":0.95,\"soc0\":0.5,\"soc_min\":0.0,\"soc_max\":1.0,\"soc_target\":0.0,\"dt_hours\":1.0,\"allow_export\":true,\"forecast_prices_today\":[46.2343,41.5005,39.3378,38.0062,39.032,38.2413,42.1012,47.0631,53.1575,52.2232,56.221,54.2465,50.4386,52.6064,49.8465,57.7811,62.4085,67.3157,56.2016,60.3596,62.0148,62.2724,53.1024,45.9017],\"forecast_demand_today\":[26.8617,25.2153,23.9225,23.8409,23.5584,24.0203,27.171,29.1947,29.2061,29.9812,30.4728,33.6008,30.9495,30.4918,28.9571,29.4635,33.2702,35.6648,36.3692,36.6453,34.4189,32.3334,29.9367,26.8444],\"actual_prices_yesterday\":[43.59,38.05,36.22,35.39,35.39,36.03,45.83,52.5,57.0,51.73,50.0,46.73,45.0,44.0,45.47,53.91,60.85,67.65,66.73,67.93,64.78,59.67,55.01,48.06],\"forecast_prices_yesterday\":[57.0772,55.0874,51.5993,47.2783,45.9009,51.9436,57.7168,64.2002,64.7076,64.3879,63.2415,63.5237,62.6621,61.6693,54.7222,63.4767,67.4161,79.62,74.6026,74.5211,69.3621,64.7601,61.8262,57.9785],\"naive_soc\":[0.5,0.324561,0.366667,0.525,0.683333,0.841667,1.0,1.0,1.0,1.0,1.0,0.824561,0.824561,0.841667,0.841667,1.0,0.877193,0.701754,0.526316,0.526316,0.350877,0.175439,0.0,0.0,0.0],\"naive_cost\":35460.8311,\"yesterday_error_stats\":{\"mean\":-11.3234,\"std\":3.9608,\"max_positive\":-4.5821,\"max_negative\":-17.6693}}", "assistant_message": "{\"soc\":[0.5,0.5,0.5,0.525,0.683333,0.841667,1.0,1.0,0.911404,0.735965,0.560526,0.560526,0.560526,0.560526,0.71886,0.877193,0.877193,0.877193,0.701754,0.526316,0.350877,0.175439,0.0,0.0,0.0],\"cost\":49737.0598,\"improvement_over_naive\":-14276.2287}", "actual_prices_today": [65.1, 62.12, 59.05, 59.0, 56.63, 56.63, 61.66, 66.61, 73.0, 73.0, 65.0, 63.82, 62.19, 59.5, 56.99, 61.92, 65.63, 82.47, 82.79, 83.5, 78.62, 72.94, 64.53, 60.32], "actual_demand_today": [28.68, 27.12, 25.99, 25.62, 25.66, 26.06, 28.58, 32.44, 35.25, 36.63, 36.22, 35.17, 33.26, 31.54, 31.52, 32.14, 33.53, 37.37, 38.78, 38.96, 36.4, 33.51, 30.89, 28.02], "forecast_prices_today": [46.23430042, 41.50053732, 39.33778151, 38.00616949, 39.03200476, 38.24127885, 42.10119966, 47.06307124, 53.15752937, 52.22315992, 56.22102261, 54.24650748, 50.43856725, 52.60644333, 49.84653038, 57.78106286, 62.40845962, 67.31571429, 56.20156832, 60.35956522, 62.01479177, 62.27242035, 53.10237809, 45.90174945], "milp_actual_soc": [0.5, 0.5, 0.5, 0.525, 0.6833333333333333, 0.8416666666666667, 1.0, 1.0, 0.9114035087719298, 0.7359649122807017, 0.5605263157894737, 0.5605263157894737, 0.5605263157894737, 0.5605263157894737, 0.718859649122807, 0.8771929824561403, 0.8771929824561403, 0.8771929824561403, 0.7017543859649122, 0.5263157894736842, 0.3508771929824561, 0.17543859649122806, 0.0, 0.0, 0.0], "milp_actual_cost": 49737.05981725, "milp_forecast_soc": [0.5, 0.32456140350877194, 0.36666666666666664, 0.525, 0.6833333333333333, 0.8416666666666667, 1.0, 1.0, 1.0, 1.0, 1.0, 0.8245614035087719, 0.8245614035087719, 0.8416666666666667, 0.8416666666666667, 1.0, 0.8771929824561403, 0.7017543859649122, 0.5263157894736842, 0.5263157894736842, 0.3508771929824561, 0.17543859649122806, 0.0, 0.0, 0.0], "milp_forecast_cost": 35460.831128878286, "battery": {"capacity_kwh": 43.13, "cmax_kw": 7.1883333333333335, "dmax_kw": 7.1883333333333335, "eta_c": 0.95, "eta_d": 0.95, "soc_init": 0.5, "soc_target": 0.0}}

# TEST_EXAMPLE = {"date": "2019-12-01", "user_message": "{\"capacity_kwh\":43.13,\"max_c_kw\":21.565,\"max_d_kw\":21.565,\"eta_c\":0.95,\"eta_d\":0.95,\"soc0\":0.5,\"soc_min\":0.0,\"soc_max\":1.0,\"soc_target\":0.0,\"dt_hours\":1.0,\"allow_export\":true,\"forecast_prices_today\":[46.2343,41.5005,39.3378,38.0062,39.032,38.2413,42.1012,47.0631,53.1575,52.2232,56.221,54.2465,50.4386,52.6064,49.8465,57.7811,62.4085,67.3157,56.2016,60.3596,62.0148,62.2724,53.1024,45.9017],\"forecast_demand_today\":[26.8617,25.2153,23.9225,23.8409,23.5584,24.0203,27.171,29.1947,29.2061,29.9812,30.4728,33.6008,30.9495,30.4918,28.9571,29.4635,33.2702,35.6648,36.3692,36.6453,34.4189,32.3334,29.9367,26.8444],\"actual_prices_yesterday\":[43.59,38.05,36.22,35.39,35.39,36.03,45.83,52.5,57.0,51.73,50.0,46.73,45.0,44.0,45.47,53.91,60.85,67.65,66.73,67.93,64.78,59.67,55.01,48.06],\"forecast_prices_yesterday\":[57.0772,55.0874,51.5993,47.2783,45.9009,51.9436,57.7168,64.2002,64.7076,64.3879,63.2415,63.5237,62.6621,61.6693,54.7222,63.4767,67.4161,79.62,74.6026,74.5211,69.3621,64.7601,61.8262,57.9785],\"naive_soc\":[0.5,0.0,0.0,0.0,0.475,0.525,1.0,1.0,1.0,1.0,1.0,0.473684,0.473684,0.525,0.525,1.0,1.0,0.526316,0.0,0.0,0.0,0.0,0.0,0.0,0.0],\"naive_cost\":35262.4202,\"yesterday_error_stats\":{\"mean\":-11.3234,\"std\":3.9608,\"max_positive\":-4.5821,\"max_negative\":-17.6693}}", "assistant_message": "{\"soc\":[0.5,0.05,0.05,0.05,0.05,0.525,1.0,1.0,1.0,0.473684,0.0,0.0,0.0,0.0,0.475,0.95,1.0,1.0,1.0,0.526316,0.0,0.0,0.0,0.0,0.0],\"cost\":49259.5644,\"improvement_over_naive\":-13997.1441}", "actual_prices_today": [65.1, 62.12, 59.05, 59.0, 56.63, 56.63, 61.66, 66.61, 73.0, 73.0, 65.0, 63.82, 62.19, 59.5, 56.99, 61.92, 65.63, 82.47, 82.79, 83.5, 78.62, 72.94, 64.53, 60.32], "actual_demand_today": [28.68, 27.12, 25.99, 25.62, 25.66, 26.06, 28.58, 32.44, 35.25, 36.63, 36.22, 35.17, 33.26, 31.54, 31.52, 32.14, 33.53, 37.37, 38.78, 38.96, 36.4, 33.51, 30.89, 28.02], "forecast_prices_today": [46.23430042, 41.50053732, 39.33778151, 38.00616949, 39.03200476, 38.24127885, 42.10119966, 47.06307124, 53.15752937, 52.22315992, 56.22102261, 54.24650748, 50.43856725, 52.60644333, 49.84653038, 57.78106286, 62.40845962, 67.31571429, 56.20156832, 60.35956522, 62.01479177, 62.27242035, 53.10237809, 45.90174945], "milp_actual_soc": [0.5, 0.05, 0.05, 0.05, 0.05, 0.525, 1.0, 1.0, 1.0, 0.4736842105263158, 0.0, 0.0, 0.0, 0.0, 0.475, 0.95, 1.0, 1.0, 1.0, 0.5263157894736842, 0.0, 0.0, 0.0, 0.0, 0.0], "milp_actual_cost": 49259.56435250001, "milp_forecast_soc": [0.5, 0.0, 0.0, 0.0, 0.475, 0.525, 1.0, 1.0, 1.0, 1.0, 1.0, 0.4736842105263158, 0.4736842105263158, 0.525, 0.525, 1.0, 1.0, 0.5263157894736842, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], "milp_forecast_cost": 35262.420244759734, "battery": {"capacity_kwh": 43.13, "cmax_kw": 21.565, "dmax_kw": 21.565, "eta_c": 0.95, "eta_d": 0.95, "soc_init": 0.5, "soc_target": 0.0}}

# TEST_EXAMPLE = {"date": "2019-12-01", "user_message": "{\"capacity_kwh\":43.13,\"max_c_kw\":10.7825,\"max_d_kw\":10.7825,\"eta_c\":0.9,\"eta_d\":0.9,\"soc0\":0.5,\"soc_min\":0.0,\"soc_max\":1.0,\"soc_target\":0.0,\"dt_hours\":1.0,\"allow_export\":true,\"forecast_prices_today\":[46.2343,41.5005,39.3378,38.0062,39.032,38.2413,42.1012,47.0631,53.1575,52.2232,56.221,54.2465,50.4386,52.6064,49.8465,57.7811,62.4085,67.3157,56.2016,60.3596,62.0148,62.2724,53.1024,45.9017],\"forecast_demand_today\":[26.8617,25.2153,23.9225,23.8409,23.5584,24.0203,27.171,29.1947,29.2061,29.9812,30.4728,33.6008,30.9495,30.4918,28.9571,29.4635,33.2702,35.6648,36.3692,36.6453,34.4189,32.3334,29.9367,26.8444],\"actual_prices_yesterday\":[43.59,38.05,36.22,35.39,35.39,36.03,45.83,52.5,57.0,51.73,50.0,46.73,45.0,44.0,45.47,53.91,60.85,67.65,66.73,67.93,64.78,59.67,55.01,48.06],\"forecast_prices_yesterday\":[57.0772,55.0874,51.5993,47.2783,45.9009,51.9436,57.7168,64.2002,64.7076,64.3879,63.2415,63.5237,62.6621,61.6693,54.7222,63.4767,67.4161,79.62,74.6026,74.5211,69.3621,64.7601,61.8262,57.9785],\"naive_soc\":[0.5,0.5,0.5,0.5,0.725,0.775,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.722222,0.444444,0.444444,0.444444,0.277778,0.0,0.0,0.0],\"naive_cost\":35603.8624,\"yesterday_error_stats\":{\"mean\":-11.3234,\"std\":3.9608,\"max_positive\":-4.5821,\"max_negative\":-17.6693}}", "assistant_message": "{\"soc\":[0.5,0.5,0.5,0.5,0.55,0.775,1.0,1.0,1.0,0.775,0.775,0.775,0.775,0.775,0.775,1.0,1.0,1.0,0.722222,0.444444,0.166667,0.0,0.0,0.0,0.0],\"cost\":49911.9738,\"improvement_over_naive\":-14308.1114}", "actual_prices_today": [65.1, 62.12, 59.05, 59.0, 56.63, 56.63, 61.66, 66.61, 73.0, 73.0, 65.0, 63.82, 62.19, 59.5, 56.99, 61.92, 65.63, 82.47, 82.79, 83.5, 78.62, 72.94, 64.53, 60.32], "actual_demand_today": [28.68, 27.12, 25.99, 25.62, 25.66, 26.06, 28.58, 32.44, 35.25, 36.63, 36.22, 35.17, 33.26, 31.54, 31.52, 32.14, 33.53, 37.37, 38.78, 38.96, 36.4, 33.51, 30.89, 28.02], "forecast_prices_today": [46.23430042, 41.50053732, 39.33778151, 38.00616949, 39.03200476, 38.24127885, 42.10119966, 47.06307124, 53.15752937, 52.22315992, 56.22102261, 54.24650748, 50.43856725, 52.60644333, 49.84653038, 57.78106286, 62.40845962, 67.31571429, 56.20156832, 60.35956522, 62.01479177, 62.27242035, 53.10237809, 45.90174945], "milp_actual_soc": [0.5, 0.5, 0.5, 0.5, 0.55, 0.775, 1.0, 1.0, 1.0, 0.775, 0.775, 0.775, 0.775, 0.775, 0.775, 1.0, 1.0, 1.0, 0.7222222222222222, 0.4444444444444444, 0.16666666666666666, 0.0, 0.0, 0.0, 0.0], "milp_actual_cost": 49911.973765555565, "milp_forecast_soc": [0.5, 0.5, 0.5, 0.5, 0.725, 0.775, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.7222222222222222, 0.4444444444444444, 0.4444444444444444, 0.4444444444444444, 0.2777777777777778, 0.0, 0.0, 0.0], "milp_forecast_cost": 35603.86238744107, "battery": {"capacity_kwh": 43.13, "cmax_kw": 10.7825, "dmax_kw": 10.7825, "eta_c": 0.9, "eta_d": 0.9, "soc_init": 0.5, "soc_target": 0.0}}

# TEST_EXAMPLE = {"date": "2019-12-01", "user_message": "{\"capacity_kwh\":43.13,\"max_c_kw\":10.7825,\"max_d_kw\":10.7825,\"eta_c\":0.85,\"eta_d\":0.85,\"soc0\":0.5,\"soc_min\":0.0,\"soc_max\":1.0,\"soc_target\":0.0,\"dt_hours\":1.0,\"allow_export\":true,\"forecast_prices_today\":[46.2343,41.5005,39.3378,38.0062,39.032,38.2413,42.1012,47.0631,53.1575,52.2232,56.221,54.2465,50.4386,52.6064,49.8465,57.7811,62.4085,67.3157,56.2016,60.3596,62.0148,62.2724,53.1024,45.9017],\"forecast_demand_today\":[26.8617,25.2153,23.9225,23.8409,23.5584,24.0203,27.171,29.1947,29.2061,29.9812,30.4728,33.6008,30.9495,30.4918,28.9571,29.4635,33.2702,35.6648,36.3692,36.6453,34.4189,32.3334,29.9367,26.8444],\"actual_prices_yesterday\":[43.59,38.05,36.22,35.39,35.39,36.03,45.83,52.5,57.0,51.73,50.0,46.73,45.0,44.0,45.47,53.91,60.85,67.65,66.73,67.93,64.78,59.67,55.01,48.06],\"forecast_prices_yesterday\":[57.0772,55.0874,51.5993,47.2783,45.9009,51.9436,57.7168,64.2002,64.7076,64.3879,63.2415,63.5237,62.6621,61.6693,54.7222,63.4767,67.4161,79.62,74.6026,74.5211,69.3621,64.7601,61.8262,57.9785],\"naive_soc\":[0.5,0.5,0.5,0.5,0.7125,0.7875,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.705882,0.411765,0.411765,0.411765,0.294118,0.0,0.0,0.0],\"naive_cost\":35792.612,\"yesterday_error_stats\":{\"mean\":-11.3234,\"std\":3.9608,\"max_positive\":-4.5821,\"max_negative\":-17.6693}}", "assistant_message": "{\"soc\":[0.5,0.5,0.5,0.5,0.5,0.7125,0.925,0.925,0.925,0.925,0.925,0.925,0.925,0.925,0.925,0.925,0.925,0.925,0.630882,0.336765,0.042647,0.0,0.0,0.0,0.0],\"cost\":50179.3904,\"improvement_over_naive\":-14386.7784}", "actual_prices_today": [65.1, 62.12, 59.05, 59.0, 56.63, 56.63, 61.66, 66.61, 73.0, 73.0, 65.0, 63.82, 62.19, 59.5, 56.99, 61.92, 65.63, 82.47, 82.79, 83.5, 78.62, 72.94, 64.53, 60.32], "actual_demand_today": [28.68, 27.12, 25.99, 25.62, 25.66, 26.06, 28.58, 32.44, 35.25, 36.63, 36.22, 35.17, 33.26, 31.54, 31.52, 32.14, 33.53, 37.37, 38.78, 38.96, 36.4, 33.51, 30.89, 28.02], "forecast_prices_today": [46.23430042, 41.50053732, 39.33778151, 38.00616949, 39.03200476, 38.24127885, 42.10119966, 47.06307124, 53.15752937, 52.22315992, 56.22102261, 54.24650748, 50.43856725, 52.60644333, 49.84653038, 57.78106286, 62.40845962, 67.31571429, 56.20156832, 60.35956522, 62.01479177, 62.27242035, 53.10237809, 45.90174945], "milp_actual_soc": [0.5, 0.5, 0.5, 0.5, 0.5, 0.7125, 0.925, 0.925, 0.925, 0.925, 0.925, 0.925, 0.925, 0.925, 0.925, 0.925, 0.925, 0.925, 0.6308823529411764, 0.3367647058823529, 0.04264705882352937, 0.0, 0.0, 0.0, 0.0], "milp_actual_cost": 50179.39042825001, "milp_forecast_soc": [0.5, 0.5, 0.5, 0.5, 0.7125, 0.7875, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.7058823529411764, 0.41176470588235287, 0.41176470588235287, 0.41176470588235287, 0.29411764705882354, 0.0, 0.0, 0.0], "milp_forecast_cost": 35792.6120037133, "battery": {"capacity_kwh": 43.13, "cmax_kw": 10.7825, "dmax_kw": 10.7825, "eta_c": 0.85, "eta_d": 0.85, "soc_init": 0.5, "soc_target": 0.0}}


# TEST_EXAMPLE = {"date": "2019-12-01", "user_message": "{\"capacity_kwh\":43.13,\"max_c_kw\":10.7825,\"max_d_kw\":10.7825,\"eta_c\":0.8,\"eta_d\":0.8,\"soc0\":0.5,\"soc_min\":0.0,\"soc_max\":1.0,\"soc_target\":0.0,\"dt_hours\":1.0,\"allow_export\":true,\"forecast_prices_today\":[46.2343,41.5005,39.3378,38.0062,39.032,38.2413,42.1012,47.0631,53.1575,52.2232,56.221,54.2465,50.4386,52.6064,49.8465,57.7811,62.4085,67.3157,56.2016,60.3596,62.0148,62.2724,53.1024,45.9017],\"forecast_demand_today\":[26.8617,25.2153,23.9225,23.8409,23.5584,24.0203,27.171,29.1947,29.2061,29.9812,30.4728,33.6008,30.9495,30.4918,28.9571,29.4635,33.2702,35.6648,36.3692,36.6453,34.4189,32.3334,29.9367,26.8444],\"actual_prices_yesterday\":[43.59,38.05,36.22,35.39,35.39,36.03,45.83,52.5,57.0,51.73,50.0,46.73,45.0,44.0,45.47,53.91,60.85,67.65,66.73,67.93,64.78,59.67,55.01,48.06],\"forecast_prices_yesterday\":[57.0772,55.0874,51.5993,47.2783,45.9009,51.9436,57.7168,64.2002,64.7076,64.3879,63.2415,63.5237,62.6621,61.6693,54.7222,63.4767,67.4161,79.62,74.6026,74.5211,69.3621,64.7601,61.8262,57.9785],\"naive_soc\":[0.5,0.5,0.5,0.5,0.7,0.8,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.6875,0.375,0.375,0.375,0.3125,0.0,0.0,0.0],\"naive_cost\":35988.2385,\"yesterday_error_stats\":{\"mean\":-11.3234,\"std\":3.9608,\"max_positive\":-4.5821,\"max_negative\":-17.6693}}", "assistant_message": "{\"soc\":[0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.3125,0.0,0.0,0.0,0.0,0.0],\"cost\":50327.3899,\"improvement_over_naive\":-14339.1515}", "actual_prices_today": [65.1, 62.12, 59.05, 59.0, 56.63, 56.63, 61.66, 66.61, 73.0, 73.0, 65.0, 63.82, 62.19, 59.5, 56.99, 61.92, 65.63, 82.47, 82.79, 83.5, 78.62, 72.94, 64.53, 60.32], "actual_demand_today": [28.68, 27.12, 25.99, 25.62, 25.66, 26.06, 28.58, 32.44, 35.25, 36.63, 36.22, 35.17, 33.26, 31.54, 31.52, 32.14, 33.53, 37.37, 38.78, 38.96, 36.4, 33.51, 30.89, 28.02], "forecast_prices_today": [46.23430042, 41.50053732, 39.33778151, 38.00616949, 39.03200476, 38.24127885, 42.10119966, 47.06307124, 53.15752937, 52.22315992, 56.22102261, 54.24650748, 50.43856725, 52.60644333, 49.84653038, 57.78106286, 62.40845962, 67.31571429, 56.20156832, 60.35956522, 62.01479177, 62.27242035, 53.10237809, 45.90174945], "milp_actual_soc": [0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.3125, 0.0, 0.0, 0.0, 0.0, 0.0], "milp_actual_cost": 50327.38994500002, "milp_forecast_soc": [0.5, 0.5, 0.5, 0.5, 0.7, 0.8, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.6875, 0.375, 0.375, 0.375, 0.3125, 0.0, 0.0, 0.0], "milp_forecast_cost": 35988.23845971305, "battery": {"capacity_kwh": 43.13, "cmax_kw": 10.7825, "dmax_kw": 10.7825, "eta_c": 0.8, "eta_d": 0.8, "soc_init": 0.5, "soc_target": 0.0}}


# TEST_EXAMPLE = {"date": "2019-12-01", "user_message": "{\"capacity_kwh\":43.13,\"max_c_kw\":10.7825,\"max_d_kw\":10.7825,\"eta_c\":0.95,\"eta_d\":0.95,\"soc0\":0.2,\"soc_min\":0.0,\"soc_max\":1.0,\"soc_target\":0.0,\"dt_hours\":1.0,\"allow_export\":true,\"forecast_prices_today\":[46.2343,41.5005,39.3378,38.0062,39.032,38.2413,42.1012,47.0631,53.1575,52.2232,56.221,54.2465,50.4386,52.6064,49.8465,57.7811,62.4085,67.3157,56.2016,60.3596,62.0148,62.2724,53.1024,45.9017],\"forecast_demand_today\":[26.8617,25.2153,23.9225,23.8409,23.5584,24.0203,27.171,29.1947,29.2061,29.9812,30.4728,33.6008,30.9495,30.4918,28.9571,29.4635,33.2702,35.6648,36.3692,36.6453,34.4189,32.3334,29.9367,26.8444],\"actual_prices_yesterday\":[43.59,38.05,36.22,35.39,35.39,36.03,45.83,52.5,57.0,51.73,50.0,46.73,45.0,44.0,45.47,53.91,60.85,67.65,66.73,67.93,64.78,59.67,55.01,48.06],\"forecast_prices_yesterday\":[57.0772,55.0874,51.5993,47.2783,45.9009,51.9436,57.7168,64.2002,64.7076,64.3879,63.2415,63.5237,62.6621,61.6693,54.7222,63.4767,67.4161,79.62,74.6026,74.5211,69.3621,64.7601,61.8262,57.9785],\"naive_soc\":[0.2,0.0,0.05,0.2875,0.525,0.7625,1.0,1.0,1.0,1.0,1.0,0.736842,0.736842,0.7625,0.7625,1.0,1.0,0.736842,0.473684,0.473684,0.473684,0.263158,0.0,0.0,0.0],\"naive_cost\":35926.976,\"yesterday_error_stats\":{\"mean\":-11.3234,\"std\":3.9608,\"max_positive\":-4.5821,\"max_negative\":-17.6693}}", "assistant_message": "{\"soc\":[0.2,0.2,0.2,0.2875,0.525,0.7625,1.0,1.0,1.0,0.736842,0.473684,0.473684,0.473684,0.473684,0.711184,0.948684,1.0,1.0,0.736842,0.473684,0.210526,0.0,0.0,0.0,0.0],\"cost\":50320.9431,\"improvement_over_naive\":-14393.9671}", "actual_prices_today": [65.1, 62.12, 59.05, 59.0, 56.63, 56.63, 61.66, 66.61, 73.0, 73.0, 65.0, 63.82, 62.19, 59.5, 56.99, 61.92, 65.63, 82.47, 82.79, 83.5, 78.62, 72.94, 64.53, 60.32], "actual_demand_today": [28.68, 27.12, 25.99, 25.62, 25.66, 26.06, 28.58, 32.44, 35.25, 36.63, 36.22, 35.17, 33.26, 31.54, 31.52, 32.14, 33.53, 37.37, 38.78, 38.96, 36.4, 33.51, 30.89, 28.02], "forecast_prices_today": [46.23430042, 41.50053732, 39.33778151, 38.00616949, 39.03200476, 38.24127885, 42.10119966, 47.06307124, 53.15752937, 52.22315992, 56.22102261, 54.24650748, 50.43856725, 52.60644333, 49.84653038, 57.78106286, 62.40845962, 67.31571429, 56.20156832, 60.35956522, 62.01479177, 62.27242035, 53.10237809, 45.90174945], "milp_actual_soc": [0.2, 0.2, 0.2, 0.2874999999999999, 0.5249999999999999, 0.7625, 1.0, 1.0, 1.0, 0.736842105263158, 0.4736842105263159, 0.4736842105263159, 0.4736842105263159, 0.4736842105263159, 0.7111842105263159, 0.9486842105263158, 1.0, 1.0, 0.736842105263158, 0.4736842105263159, 0.21052631578947378, 0.0, 0.0, 0.0, 0.0], "milp_actual_cost": 50320.94308526317, "milp_forecast_soc": [0.2, 0.0, 0.05, 0.2874999999999999, 0.5249999999999999, 0.7625, 1.0, 1.0, 1.0, 1.0, 1.0, 0.736842105263158, 0.736842105263158, 0.7625, 0.7625, 1.0, 1.0, 0.736842105263158, 0.4736842105263159, 0.4736842105263159, 0.4736842105263159, 0.2631578947368421, 0.0, 0.0, 0.0], "milp_forecast_cost": 35926.97601893828, "battery": {"capacity_kwh": 43.13, "cmax_kw": 10.7825, "dmax_kw": 10.7825, "eta_c": 0.95, "eta_d": 0.95, "soc_init": 0.2, "soc_target": 0.0}}

# TEST_EXAMPLE = {"date": "2019-12-01", "user_message": "{\"capacity_kwh\":43.13,\"max_c_kw\":10.7825,\"max_d_kw\":10.7825,\"eta_c\":0.95,\"eta_d\":0.95,\"soc0\":0.4,\"soc_min\":0.0,\"soc_max\":1.0,\"soc_target\":0.0,\"dt_hours\":1.0,\"allow_export\":true,\"forecast_prices_today\":[46.2343,41.5005,39.3378,38.0062,39.032,38.2413,42.1012,47.0631,53.1575,52.2232,56.221,54.2465,50.4386,52.6064,49.8465,57.7811,62.4085,67.3157,56.2016,60.3596,62.0148,62.2724,53.1024,45.9017],\"forecast_demand_today\":[26.8617,25.2153,23.9225,23.8409,23.5584,24.0203,27.171,29.1947,29.2061,29.9812,30.4728,33.6008,30.9495,30.4918,28.9571,29.4635,33.2702,35.6648,36.3692,36.6453,34.4189,32.3334,29.9367,26.8444],\"actual_prices_yesterday\":[43.59,38.05,36.22,35.39,35.39,36.03,45.83,52.5,57.0,51.73,50.0,46.73,45.0,44.0,45.47,53.91,60.85,67.65,66.73,67.93,64.78,59.67,55.01,48.06],\"forecast_prices_yesterday\":[57.0772,55.0874,51.5993,47.2783,45.9009,51.9436,57.7168,64.2002,64.7076,64.3879,63.2415,63.5237,62.6621,61.6693,54.7222,63.4767,67.4161,79.62,74.6026,74.5211,69.3621,64.7601,61.8262,57.9785],\"naive_soc\":[0.4,0.136842,0.136842,0.2875,0.525,0.7625,1.0,1.0,1.0,1.0,1.0,0.736842,0.736842,0.7625,0.7625,1.0,1.0,0.736842,0.473684,0.473684,0.473684,0.263158,0.0,0.0,0.0],\"naive_cost\":35558.0303,\"yesterday_error_stats\":{\"mean\":-11.3234,\"std\":3.9608,\"max_positive\":-4.5821,\"max_negative\":-17.6693}}", "assistant_message": "{\"soc\":[0.4,0.4,0.4,0.4,0.525,0.7625,1.0,1.0,1.0,0.736842,0.473684,0.473684,0.473684,0.473684,0.711184,0.948684,1.0,1.0,0.736842,0.473684,0.210526,0.0,0.0,0.0,0.0],\"cost\":49785.0245,\"improvement_over_naive\":-14226.9942}", "actual_prices_today": [65.1, 62.12, 59.05, 59.0, 56.63, 56.63, 61.66, 66.61, 73.0, 73.0, 65.0, 63.82, 62.19, 59.5, 56.99, 61.92, 65.63, 82.47, 82.79, 83.5, 78.62, 72.94, 64.53, 60.32], "actual_demand_today": [28.68, 27.12, 25.99, 25.62, 25.66, 26.06, 28.58, 32.44, 35.25, 36.63, 36.22, 35.17, 33.26, 31.54, 31.52, 32.14, 33.53, 37.37, 38.78, 38.96, 36.4, 33.51, 30.89, 28.02], "forecast_prices_today": [46.23430042, 41.50053732, 39.33778151, 38.00616949, 39.03200476, 38.24127885, 42.10119966, 47.06307124, 53.15752937, 52.22315992, 56.22102261, 54.24650748, 50.43856725, 52.60644333, 49.84653038, 57.78106286, 62.40845962, 67.31571429, 56.20156832, 60.35956522, 62.01479177, 62.27242035, 53.10237809, 45.90174945], "milp_actual_soc": [0.4, 0.4, 0.4, 0.4, 0.5249999999999999, 0.7625, 1.0, 1.0, 1.0, 0.736842105263158, 0.4736842105263159, 0.4736842105263159, 0.4736842105263159, 0.4736842105263159, 0.7111842105263159, 0.9486842105263158, 1.0, 1.0, 0.736842105263158, 0.4736842105263159, 0.21052631578947378, 0.0, 0.0, 0.0, 0.0], "milp_actual_cost": 49785.02446026317, "milp_forecast_soc": [0.4, 0.13684210526315793, 0.13684210526315793, 0.2874999999999999, 0.5249999999999999, 0.7625, 1.0, 1.0, 1.0, 1.0, 1.0, 0.736842105263158, 0.736842105263158, 0.7625, 0.7625, 1.0, 1.0, 0.736842105263158, 0.4736842105263159, 0.4736842105263159, 0.4736842105263159, 0.2631578947368421, 0.0, 0.0, 0.0], "milp_forecast_cost": 35558.03029696795, "battery": {"capacity_kwh": 43.13, "cmax_kw": 10.7825, "dmax_kw": 10.7825, "eta_c": 0.95, "eta_d": 0.95, "soc_init": 0.4, "soc_target": 0.0}}

# TEST_EXAMPLE = {"date": "2019-12-01", "user_message": "{\"capacity_kwh\":43.13,\"max_c_kw\":10.7825,\"max_d_kw\":10.7825,\"eta_c\":0.95,\"eta_d\":0.95,\"soc0\":0.6,\"soc_min\":0.0,\"soc_max\":1.0,\"soc_target\":0.0,\"dt_hours\":1.0,\"allow_export\":true,\"forecast_prices_today\":[46.2343,41.5005,39.3378,38.0062,39.032,38.2413,42.1012,47.0631,53.1575,52.2232,56.221,54.2465,50.4386,52.6064,49.8465,57.7811,62.4085,67.3157,56.2016,60.3596,62.0148,62.2724,53.1024,45.9017],\"forecast_demand_today\":[26.8617,25.2153,23.9225,23.8409,23.5584,24.0203,27.171,29.1947,29.2061,29.9812,30.4728,33.6008,30.9495,30.4918,28.9571,29.4635,33.2702,35.6648,36.3692,36.6453,34.4189,32.3334,29.9367,26.8444],\"actual_prices_yesterday\":[43.59,38.05,36.22,35.39,35.39,36.03,45.83,52.5,57.0,51.73,50.0,46.73,45.0,44.0,45.47,53.91,60.85,67.65,66.73,67.93,64.78,59.67,55.01,48.06],\"forecast_prices_yesterday\":[57.0772,55.0874,51.5993,47.2783,45.9009,51.9436,57.7168,64.2002,64.7076,64.3879,63.2415,63.5237,62.6621,61.6693,54.7222,63.4767,67.4161,79.62,74.6026,74.5211,69.3621,64.7601,61.8262,57.9785],\"naive_soc\":[0.6,0.336842,0.336842,0.336842,0.574342,0.7625,1.0,1.0,1.0,1.0,1.0,0.736842,0.736842,0.7625,0.7625,1.0,1.0,0.736842,0.473684,0.473684,0.473684,0.263158,0.0,0.0,0.0],\"naive_cost\":35201.5282,\"yesterday_error_stats\":{\"mean\":-11.3234,\"std\":3.9608,\"max_positive\":-4.5821,\"max_negative\":-17.6693}}", "assistant_message": "{\"soc\":[0.6,0.525,0.525,0.525,0.525,0.7625,1.0,1.0,1.0,0.736842,0.473684,0.473684,0.473684,0.473684,0.711184,0.948684,1.0,1.0,0.736842,0.473684,0.210526,0.0,0.0,0.0,0.0],\"cost\":49250.1463,\"improvement_over_naive\":-14048.6181}", "actual_prices_today": [65.1, 62.12, 59.05, 59.0, 56.63, 56.63, 61.66, 66.61, 73.0, 73.0, 65.0, 63.82, 62.19, 59.5, 56.99, 61.92, 65.63, 82.47, 82.79, 83.5, 78.62, 72.94, 64.53, 60.32], "actual_demand_today": [28.68, 27.12, 25.99, 25.62, 25.66, 26.06, 28.58, 32.44, 35.25, 36.63, 36.22, 35.17, 33.26, 31.54, 31.52, 32.14, 33.53, 37.37, 38.78, 38.96, 36.4, 33.51, 30.89, 28.02], "forecast_prices_today": [46.23430042, 41.50053732, 39.33778151, 38.00616949, 39.03200476, 38.24127885, 42.10119966, 47.06307124, 53.15752937, 52.22315992, 56.22102261, 54.24650748, 50.43856725, 52.60644333, 49.84653038, 57.78106286, 62.40845962, 67.31571429, 56.20156832, 60.35956522, 62.01479177, 62.27242035, 53.10237809, 45.90174945], "milp_actual_soc": [0.6, 0.5249999999999999, 0.5249999999999999, 0.5249999999999999, 0.5249999999999999, 0.7625, 1.0, 1.0, 1.0, 0.736842105263158, 0.4736842105263159, 0.4736842105263159, 0.4736842105263159, 0.4736842105263159, 0.7111842105263159, 0.9486842105263158, 1.0, 1.0, 0.736842105263158, 0.4736842105263159, 0.21052631578947378, 0.0, 0.0, 0.0, 0.0], "milp_actual_cost": 49250.14634651317, "milp_forecast_soc": [0.6, 0.3368421052631579, 0.3368421052631579, 0.3368421052631579, 0.5743421052631579, 0.7625, 1.0, 1.0, 1.0, 1.0, 1.0, 0.736842105263158, 0.736842105263158, 0.7625, 0.7625, 1.0, 1.0, 0.736842105263158, 0.4736842105263159, 0.4736842105263159, 0.4736842105263159, 0.2631578947368421, 0.0, 0.0, 0.0], "milp_forecast_cost": 35201.52822101093, "battery": {"capacity_kwh": 43.13, "cmax_kw": 10.7825, "dmax_kw": 10.7825, "eta_c": 0.95, "eta_d": 0.95, "soc_init": 0.6, "soc_target": 0.0}}

TEST_EXAMPLE = {"date": "2019-12-01", "user_message": "{\"capacity_kwh\":43.13,\"max_c_kw\":10.7825,\"max_d_kw\":10.7825,\"eta_c\":0.95,\"eta_d\":0.95,\"soc0\":0.8,\"soc_min\":0.0,\"soc_max\":1.0,\"soc_target\":0.0,\"dt_hours\":1.0,\"allow_export\":true,\"forecast_prices_today\":[46.2343,41.5005,39.3378,38.0062,39.032,38.2413,42.1012,47.0631,53.1575,52.2232,56.221,54.2465,50.4386,52.6064,49.8465,57.7811,62.4085,67.3157,56.2016,60.3596,62.0148,62.2724,53.1024,45.9017],\"forecast_demand_today\":[26.8617,25.2153,23.9225,23.8409,23.5584,24.0203,27.171,29.1947,29.2061,29.9812,30.4728,33.6008,30.9495,30.4918,28.9571,29.4635,33.2702,35.6648,36.3692,36.6453,34.4189,32.3334,29.9367,26.8444],\"actual_prices_yesterday\":[43.59,38.05,36.22,35.39,35.39,36.03,45.83,52.5,57.0,51.73,50.0,46.73,45.0,44.0,45.47,53.91,60.85,67.65,66.73,67.93,64.78,59.67,55.01,48.06],\"forecast_prices_yesterday\":[57.0772,55.0874,51.5993,47.2783,45.9009,51.9436,57.7168,64.2002,64.7076,64.3879,63.2415,63.5237,62.6621,61.6693,54.7222,63.4767,67.4161,79.62,74.6026,74.5211,69.3621,64.7601,61.8262,57.9785],\"naive_soc\":[0.8,0.536842,0.536842,0.536842,0.774342,0.774342,1.0,1.0,1.0,1.0,1.0,0.736842,0.736842,0.7625,0.7625,1.0,1.0,0.736842,0.473684,0.473684,0.473684,0.263158,0.0,0.0,0.0],\"naive_cost\":34847.5427,\"yesterday_error_stats\":{\"mean\":-11.3234,\"std\":3.9608,\"max_positive\":-4.5821,\"max_negative\":-17.6693}}", "assistant_message": "{\"soc\":[0.8,0.536842,0.536842,0.536842,0.536842,0.7625,1.0,1.0,1.0,0.736842,0.473684,0.473684,0.473684,0.473684,0.711184,0.948684,1.0,1.0,0.736842,0.473684,0.210526,0.0,0.0,0.0,0.0],\"cost\":48717.8126,\"improvement_over_naive\":-13870.2699}", "actual_prices_today": [65.1, 62.12, 59.05, 59.0, 56.63, 56.63, 61.66, 66.61, 73.0, 73.0, 65.0, 63.82, 62.19, 59.5, 56.99, 61.92, 65.63, 82.47, 82.79, 83.5, 78.62, 72.94, 64.53, 60.32], "actual_demand_today": [28.68, 27.12, 25.99, 25.62, 25.66, 26.06, 28.58, 32.44, 35.25, 36.63, 36.22, 35.17, 33.26, 31.54, 31.52, 32.14, 33.53, 37.37, 38.78, 38.96, 36.4, 33.51, 30.89, 28.02], "forecast_prices_today": [46.23430042, 41.50053732, 39.33778151, 38.00616949, 39.03200476, 38.24127885, 42.10119966, 47.06307124, 53.15752937, 52.22315992, 56.22102261, 54.24650748, 50.43856725, 52.60644333, 49.84653038, 57.78106286, 62.40845962, 67.31571429, 56.20156832, 60.35956522, 62.01479177, 62.27242035, 53.10237809, 45.90174945], "milp_actual_soc": [0.8, 0.536842105263158, 0.536842105263158, 0.536842105263158, 0.536842105263158, 0.7625, 1.0, 1.0, 1.0, 0.736842105263158, 0.4736842105263159, 0.4736842105263159, 0.4736842105263159, 0.4736842105263159, 0.7111842105263159, 0.9486842105263158, 1.0, 1.0, 0.736842105263158, 0.4736842105263159, 0.21052631578947378, 0.0, 0.0, 0.0, 0.0], "milp_actual_cost": 48717.81263394738, "milp_forecast_soc": [0.8, 0.536842105263158, 0.536842105263158, 0.536842105263158, 0.774342105263158, 0.774342105263158, 1.0, 1.0, 1.0, 1.0, 1.0, 0.736842105263158, 0.736842105263158, 0.7625, 0.7625, 1.0, 1.0, 0.736842105263158, 0.4736842105263159, 0.4736842105263159, 0.4736842105263159, 0.2631578947368421, 0.0, 0.0, 0.0], "milp_forecast_cost": 34847.54273700964, "battery": {"capacity_kwh": 43.13, "cmax_kw": 10.7825, "dmax_kw": 10.7825, "eta_c": 0.95, "eta_d": 0.95, "soc_init": 0.8, "soc_target": 0.0}}


# ============================================================================
# System Prompt (same as original)
# ============================================================================

SYSTEM_PROMPT = """\
You are an expert energy trader operating a battery storage system for electricity price arbitrage.

## Battery Model

You control a battery with these parameters (provided per day):
- capacity_kwh: total energy capacity (kWh)
- max_c_kw / max_d_kw: max charge / discharge power (kW)
- eta_c / eta_d: charge / discharge efficiency
- soc0: initial state-of-charge (fraction, 0 to 1)
- soc_min / soc_max: SOC bounds (fractions)
- soc_target: minimum final SOC at end of day (fraction)
- dt_hours: time step duration (hours)
- allow_export: whether selling back to grid is allowed

## SOC Dynamics

At each hour t (0 to 23), you choose charge power c[t] >= 0 and discharge power d[t] >= 0.
You cannot charge and discharge simultaneously.

SOC updates as:
  soc[t+1] = soc[t] + (eta_c * c[t] * dt - d[t] * dt / eta_d) / capacity_kwh

Constraints:
  0 <= c[t] <= max_c_kw
  0 <= d[t] <= max_d_kw
  c[t] and d[t] cannot both be > 0 at the same time
  soc_min <= soc[t] <= soc_max   for all t = 0..24
  soc[0] = soc0
  soc[24] >= soc_target

## Grid Interaction

Net grid power: net[t] = demand[t] + c[t] - d[t]
If allow_export is true:
  import[t] - export[t] = net[t],  import[t] >= 0, export[t] >= 0
If allow_export is false:
  import[t] >= net[t],  import[t] >= 0

## Objective (minimize total cost)

If allow_export:
  cost = sum over t of (price[t] * import[t] * dt - price[t] * export[t] * dt)
If not allow_export:
  cost = sum over t of (price[t] * import[t] * dt)

## Your Task

Given the battery parameters, forecast prices for today, forecast demand for today, and yesterday's price forecast errors, determine:
1. The optimal SOC trajectory: 25 values [soc[0], soc[1], ..., soc[24]]
2. The resulting objective cost

## How to Reason

1. Identify the cheapest and most expensive hours from the price forecast
2. Plan to charge during cheap hours and discharge during expensive hours
3. Account for efficiency losses (you lose energy on both charge and discharge)
4. Respect all constraints (SOC bounds, power limits, no simultaneous charge/discharge)
5. Calculate the resulting cost carefully
6. Consider yesterday's forecast errors - if forecasts were systematically biased, adjust

## Output Format

After your reasoning, you MUST end with EXACTLY this format:

SOC_TRAJECTORY: [s0, s1, s2, ..., s24]
OBJECTIVE_COST: <number>

Where s0..s24 are 25 comma-separated floats (SOC fractions) and the cost is a single number.
These must be the LAST two lines of your response.
"""


# ============================================================================
# Format Input Prompt
# ============================================================================

def format_user_prompt(user_message_json: str) -> str:
    """Format the user_message JSON into a readable prompt."""
    data = json.loads(user_message_json) if isinstance(user_message_json, str) else user_message_json

    lines = []
    lines.append("## Battery Parameters")
    lines.append(f"- Capacity: {data['capacity_kwh']} kWh")
    lines.append(f"- Max charge: {data['max_c_kw']} kW, Max discharge: {data['max_d_kw']} kW")
    lines.append(f"- Charge efficiency: {data['eta_c']}, Discharge efficiency: {data['eta_d']}")
    lines.append(f"- Initial SOC: {data['soc0']}")
    lines.append(f"- SOC bounds: [{data['soc_min']}, {data['soc_max']}]")
    lines.append(f"- SOC target (end of day): {data['soc_target']}")
    lines.append(f"- Time step: {data['dt_hours']} hours")
    lines.append(f"- Allow export: {data['allow_export']}")

    prices = data['forecast_prices_today']
    demand = data['forecast_demand_today']

    lines.append("")
    lines.append("## Hourly Forecast Data")
    lines.append("| Hour | Price (€/MWh) | Demand (MW) |")
    lines.append("|------|---------------|-------------|")
    for h in range(24):
        lines.append(f"| {h:02d}   | {prices[h]:13.2f} | {demand[h]:11.2f} |")

    mean_price = np.mean(prices)
    lines.append(f"\nMean forecast price: {mean_price:.2f} €/MWh")
    cheapest = sorted(range(24), key=lambda h: prices[h])
    lines.append(f"Cheapest hours: {cheapest[:6]}")
    lines.append(f"Most expensive hours: {cheapest[-6:][::-1]}")

    if 'yesterday_error_stats' in data and data['yesterday_error_stats']:
        err = data['yesterday_error_stats']
        lines.append("")
        lines.append("## Yesterday's Forecast Error Stats")
        lines.append(f"- Mean error: {err['mean']:.2f} (forecast - actual)")
        lines.append(f"- Std: {err['std']:.2f}")
        lines.append(f"- Max positive error: {err['max_positive']:.2f}")
        lines.append(f"- Max negative error: {err['max_negative']:.2f}")

    if 'actual_prices_yesterday' in data:
        lines.append("")
        lines.append("## Yesterday's Actual vs Forecast Prices")
        lines.append("| Hour | Actual | Forecast | Error |")
        lines.append("|------|--------|----------|-------|")
        act_y = data['actual_prices_yesterday']
        frc_y = data['forecast_prices_yesterday']
        for h in range(min(24, len(act_y), len(frc_y))):
            lines.append(f"| {h:02d}   | {act_y[h]:6.2f} | {frc_y[h]:8.2f} | {frc_y[h]-act_y[h]:+6.2f} |")

    if 'naive_soc' in data:
        lines.append("")
        lines.append("## Naive Optimizer SOC Trajectory (for reference)")
        lines.append(f"SOC: {[round(s, 4) for s in data['naive_soc']]}")
        lines.append(f"Naive cost: {data['naive_cost']:.2f}")

    return "\n".join(lines)


# ============================================================================
# Parse SOC + Cost from Model Output
# ============================================================================

def parse_soc_and_cost(text: str) -> Tuple[Optional[List[float]], Optional[float]]:
    soc = None
    cost = None

    soc_match = re.search(r'SOC_TRAJECTORY:\s*\[([^\]]+)\]', text, re.IGNORECASE)
    if soc_match:
        try:
            soc = [float(x.strip()) for x in soc_match.group(1).split(',')]
        except ValueError:
            pass

    cost_match = re.search(r'OBJECTIVE_COST:\s*([-\d.eE+]+)', text, re.IGNORECASE)
    if cost_match:
        try:
            cost = float(cost_match.group(1))
        except ValueError:
            pass

    return soc, cost


# ============================================================================
# Derive Hourly Operational Data from SOC Trajectory
# ============================================================================

def derive_hourly_data(
    soc_trajectory: List[float],
    battery: Dict[str, float],
    actual_prices: List[float],
    forecast_prices: List[float],
    actual_demand: List[float],
    forecast_demand: List[float],
    allow_export: bool,
) -> List[Dict]:
    """
    From a 25-element SOC trajectory, derive per-hour:
    charge_MW, discharge_MW, import_MW, export_MW, profit_step
    and combine with prices/demand data.
    Returns 24 row dicts.
    """
    C = battery['capacity_kwh']
    eta_c = battery['eta_c']
    eta_d = battery['eta_d']
    dt = 1.0  # hours

    rows = []
    for t in range(24):
        delta_soc = soc_trajectory[t + 1] - soc_trajectory[t]
        delta_energy = delta_soc * C  # kWh

        if delta_energy >= 0:
            # Charging
            c_t = delta_energy / (eta_c * dt)  # kW
            d_t = 0.0
        else:
            # Discharging: delta_soc * C = -d * dt / eta_d  =>  d = -delta_soc * C * eta_d / dt
            c_t = 0.0
            d_t = -delta_soc * C * eta_d / dt  # kW

        charge_val = round(c_t, 2)
        discharge_val = round(d_t, 2)

        # Net grid power
        net = actual_demand[t] + c_t - d_t

        if allow_export:
            if net >= 0:
                imp_t = net
                exp_t = 0.0
            else:
                imp_t = 0.0
                exp_t = -net
        else:
            imp_t = max(net, 0.0)
            exp_t = 0.0

        baseline_cost = actual_prices[t] * actual_demand[t] * dt
        actual_cost = actual_prices[t] * imp_t * dt - actual_prices[t] * exp_t * dt
        profit_step = baseline_cost - actual_cost

        rows.append({
            'prices_actual': round(actual_prices[t], 2),
            'prices_forecast': round(forecast_prices[t], 2),
            'actual_demand': round(actual_demand[t], 2),
            'forecast_demand': round(forecast_demand[t], 2),
            'soc': round(soc_trajectory[t], 4),
            'charge_MW': charge_val,
            'discharge_MW': discharge_val,
            'import_MW': round(imp_t, 2),
            'export_MW': round(exp_t, 2),
            'profit_step': round(profit_step, 2),
        })

    return rows


# ============================================================================
# OpenAI API Call
# ============================================================================

def call_openai(client: OpenAI, user_message_json: str, config: Config, temperature: float) -> Tuple[str, float]:
    """Call GPT-4. Returns (generated_text, api_cost_usd)."""
    formatted = format_user_prompt(user_message_json)

    user_prompt = f"""Please analyze the following battery storage data and determine the optimal charging/discharging schedule.

{formatted}

Reason step by step:
1. Identify cheap vs expensive hours
2. Plan charge/discharge schedule respecting all constraints
3. Compute the SOC trajectory and total cost

End your response with:
SOC_TRAJECTORY: [s0, s1, ..., s24]
OBJECTIVE_COST: <number>"""

    try:
        response = client.chat.completions.create(
            model=config.model,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt},
            ],
            temperature=temperature,
            max_tokens=config.max_tokens,
        )

        text = response.choices[0].message.content
        inp_tok = response.usage.prompt_tokens
        out_tok = response.usage.completion_tokens

        if "gpt-4" in config.model and "mini" not in config.model:
            api_cost = (inp_tok * 2.5 + out_tok * 10) / 1_000_000
        elif "gpt-4-mini" in config.model:
            api_cost = (inp_tok * 0.15 + out_tok * 0.6) / 1_000_000
        else:
            api_cost = (inp_tok * 5 + out_tok * 15) / 1_000_000

        return text, api_cost

    except Exception as e:
        print(f"  API Error: {e}")
        return "", 0.0


# ============================================================================
# Main Loop
# ============================================================================

def run(config: Config):
    print("=" * 70)
    print("GPT-4 Storage Arbitrage → Temperature Sweep (2019-12-01)")
    print("=" * 70)

    OPENAI_API_KEY = ""

    client = OpenAI(api_key=OPENAI_API_KEY)

    ex = TEST_EXAMPLE
    date = ex["date"]
    user_msg = ex["user_message"]
    user_data = json.loads(user_msg) if isinstance(user_msg, str) else user_msg

    # Battery params
    batt_raw = ex.get("battery", {})
    battery = {
        'capacity_kwh': batt_raw.get('capacity_kwh', user_data.get('capacity_kwh', 49.44)),
        'cmax_kw': batt_raw.get('cmax_kw', user_data.get('max_c_kw', 12.36)),
        'dmax_kw': batt_raw.get('dmax_kw', user_data.get('max_d_kw', 12.36)),
        'eta_c': batt_raw.get('eta_c', user_data.get('eta_c', 0.95)),
        'eta_d': batt_raw.get('eta_d', user_data.get('eta_d', 0.95)),
        'soc_init': user_data.get('soc0', 0.5),
    }
    allow_export = user_data.get('allow_export', True)

    forecast_prices = user_data.get('forecast_prices_today', [])
    forecast_demand = user_data.get('forecast_demand_today', [])
    actual_prices = ex.get("actual_prices_today", forecast_prices)
    actual_demand = ex.get("actual_demand_today", forecast_demand)

    # --- Prepare CSV ---
    csv_path = config.output_csv
    columns = [
        'temperature', 'prices_actual', 'prices_forecast', 'actual_demand', 'forecast_demand',
        'soc', 'charge_MW', 'discharge_MW', 'import_MW', 'export_MW', 'profit_step'
    ]

    with open(csv_path, 'w', newline='') as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=columns)
        writer.writeheader()

        total_api_cost = 0.0
        failed_runs = 0

        for temp in config.temperatures:
            start_t = time.time()
            raw_text, api_cost = call_openai(client, user_msg, config, temp)
            elapsed = time.time() - start_t
            total_api_cost += api_cost

            # Parse SOC trajectory
            pred_soc, pred_cost = parse_soc_and_cost(raw_text)

            if pred_soc and len(pred_soc) == 25:
                # Derive hourly operational data
                hourly_rows = derive_hourly_data(
                    soc_trajectory=pred_soc,
                    battery=battery,
                    actual_prices=actual_prices,
                    forecast_prices=forecast_prices,
                    actual_demand=actual_demand,
                    forecast_demand=forecast_demand,
                    allow_export=allow_export,
                )
                for row in hourly_rows:
                    row['temperature'] = temp
                    writer.writerow(row)
                csvfile.flush()

                print(f"Temp {temp:.1f} ({date}) ✓  24 rows written  "
                      f"API ${api_cost:.3f}  {elapsed:.1f}s")
            else:
                # Failed parse
                failed_runs += 1
                n_vals = len(pred_soc) if pred_soc else 0
                print(f"Temp {temp:.1f} ({date}) ✗  SOC parse failed ({n_vals} values)  "
                      f"API ${api_cost:.3f}  {elapsed:.1f}s")
                for t in range(24):
                    row = {
                        'temperature': temp,
                        'prices_actual': round(actual_prices[t], 2) if t < len(actual_prices) else 0,
                        'prices_forecast': round(forecast_prices[t], 2) if t < len(forecast_prices) else 0,
                        'actual_demand': round(actual_demand[t], 2) if t < len(actual_demand) else 0,
                        'forecast_demand': round(forecast_demand[t], 2) if t < len(forecast_demand) else 0,
                        'soc': '',
                        'charge_MW': '',
                        'discharge_MW': '',
                        'import_MW': '',
                        'export_MW': '',
                        'profit_step': '',
                    }
                    writer.writerow(row)
                csvfile.flush()

            time.sleep(config.delay_between_calls)

    print("\n" + "=" * 70)
    print(f"Done! {len(config.temperatures)} temperature settings processed.")
    print(f"Failed parses: {failed_runs}")
    print(f"Total API cost: ${total_api_cost:.4f}")
    print(f"Output: {csv_path}")
    print("=" * 70)


# ============================================================================
# Entry Point
# ============================================================================

if __name__ == "__main__":
    config = Config()
    run(config)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPT-4 Storage Arbitrage → Temperature Sweep (2019-12-01)
Temp 0.0 (2019-12-01) ✓  24 rows written  API $0.011  17.0s
Temp 0.1 (2019-12-01) ✓  24 rows written  API $0.010  16.5s
Temp 0.2 (2019-12-01) ✓  24 rows written  API $0.011  16.6s
Temp 0.3 (2019-12-01) ✓  24 rows written  API $0.011  14.2s
Temp 0.4 (2019-12-01) ✓  24 rows written  API $0.010  12.2s
Temp 0.5 (2019-12-01) ✓  24 rows written  API $0.011  14.0s
Temp 0.6 (2019-12-01) ✓  24 rows written  API $0.011  14.5s
Temp 0.7 (2019-12-01) ✓  24 rows written  API $0.011  23.3s
Temp 0.8 (2019-12-01) ✗  SOC parse failed (24 values)  API $0.018  45.0s
Temp 0.9 (2019-12-01) ✓  24 rows written  API $0.010  15.3s
Temp 1.0 (2019-12-01) ✗  SOC parse failed (0 values)  API $0.014  19.5s

Done! 11 temperature settings processed.
Failed parses: 2
Total API cost: $0.1288
Output: /content/drive/MyDrive/energy_finetuni

In [ ]:
# -*- coding: utf-8 -*-
"""
GPT-4o Storage Arbitrage — EXHAUSTIVE Check & MINIMAL Correction
Reads   : gpt4_hourly_arbitrage_cleaned.csv  (24 rows/day × 365 days)
Writes  : gpt4_hourly_arbitrage_corrected.csv (same format, all feasible)

Philosophy:
  - We keep GPT's charge/discharge decisions as close to original as possible.
  - We do NOT replace with an LP solution.
  - We only make the MINIMUM adjustments needed to satisfy all constraints.

Correction steps (applied only to violated days):
  Step 1: Clip c[t] to [0, cmax], d[t] to [0, dmax]
  Step 2: Net out simultaneous charge & discharge (keep dominant direction)
  Step 3: Forward-simulate SOC. If SOC breaches bounds, reduce the c or d
           that caused the breach — just enough to hit the boundary.
  Step 4: If soc[24] < soc_target, greedily add minimal charge at the
           cheapest available hours (or reduce discharge) until target is met,
           then re-run Step 3 to ensure no new SOC breaches.
  Step 5: Recompute import/export/profit from corrected c, d + actual data.

Checks performed (17 constraint families):
  [C1]  SOC bounds             [C10] Non-negative export
  [C2]  Initial SOC            [C11] Export restriction
  [C3]  Final SOC target       [C12] CSV SOC vs dynamics SOC
  [C4]  Charge power limits    [C13] 4h rolling energy cap
  [C5]  Discharge power limits [C14] Charge headroom per step
  [C6]  No simultaneous c & d  [C15] Discharge available per step
  [C7]  SOC dynamics consist.  [C16] NaN / missing data
  [C8]  Grid balance           [C17] Profit consistency
  [C9]  Non-negative import
"""

import csv
import math
import copy
import numpy as np
from dataclasses import dataclass
from typing import List, Tuple, Optional, Dict


# ============================================================================
# Battery Configuration  (match your JSONL defaults — edit if needed)
# ============================================================================

@dataclass
class BatteryConfig:
    capacity_kwh: float = 43.13
    cmax_kw: float = 10.7825     # max charge power
    dmax_kw: float = 10.7825       # max discharge power
    eta_c: float = 0.95          # charge efficiency
    eta_d: float = 0.95          # discharge efficiency
    soc_min: float = 0.0
    soc_max: float = 1.0
    soc_target: float = 0.0      # minimum final SOC (end of day)
    dt: float = 1.0              # time-step in hours
    allow_export: bool = True


# ============================================================================
# File Paths
# ============================================================================

INPUT_CSV  = "/content/drive/MyDrive/energy_finetuning_35/RF_0.8_Dec1_Table_5_gpt4_hourly_arbitrage_temp_sweep.csv"
OUTPUT_CSV = "/content/drive/MyDrive/energy_finetuning_35/RF_0.8_Dec1_Table_5_gpt4_hourly_arbitrage_temp_sweep_corrected.csv"
JSONL_FILE = None   # e.g. "test_qwen_prevday_ITALY_1.jsonl"
ROLLING_WINDOW_H = 2


# ============================================================================
# Helpers
# ============================================================================

def safe_float(val, default=0.0) -> Tuple[float, bool]:
    if val is None or val == '':
        return default, False
    try:
        v = float(val)
        if math.isnan(v) or math.isinf(v):
            return default, False
        return v, True
    except (ValueError, TypeError):
        return default, False


def read_csv_days(path: str) -> List[List[Dict]]:
    with open(path, "r") as f:
        reader = csv.DictReader(f)
        all_rows = list(reader)
    days = []
    for start in range(0, len(all_rows), 24):
        chunk = all_rows[start : start + 24]
        if len(chunk) == 24:
            days.append(chunk)
        else:
            print(f"  Warning: incomplete day at row {start} ({len(chunk)} rows), skipping")
    return days


def load_jsonl_days(path: str) -> List[Dict]:
    import json
    records = []
    with open(path) as f:
        for line in f:
            records.append(json.loads(line))
    return records


def battery_from_jsonl(rec: Dict) -> Tuple[BatteryConfig, float]:
    import json
    ud = rec.get("user_message", {})
    if isinstance(ud, str):
        ud = json.loads(ud)
    batt_raw = rec.get("battery", {})
    cfg = BatteryConfig(
        capacity_kwh = batt_raw.get("capacity_kwh", ud.get("capacity_kwh", 49.44)),
        cmax_kw      = batt_raw.get("cmax_kw",      ud.get("max_c_kw",    12.36)),
        dmax_kw      = batt_raw.get("dmax_kw",      ud.get("max_d_kw",    12.36)),
        eta_c        = batt_raw.get("eta_c",         ud.get("eta_c",       0.95)),
        eta_d        = batt_raw.get("eta_d",         ud.get("eta_d",       0.95)),
        soc_min      = ud.get("soc_min", 0.0),
        soc_max      = ud.get("soc_max", 1.0),
        soc_target   = ud.get("soc_target", 0.0),
        allow_export = ud.get("allow_export", True),
    )
    soc0 = ud.get("soc0", 0.5)
    return cfg, soc0


# ============================================================================
# EXHAUSTIVE Feasibility Checker  (17 constraint families)
# ============================================================================

def exhaustive_check(
    rows: List[Dict],
    batt: BatteryConfig,
    soc0: float,
    tol: float = 1e-3,
    rolling_window: int = ROLLING_WINDOW_H,
) -> Tuple[bool, List[str]]:
    T = 24
    C = batt.capacity_kwh
    eta_c = batt.eta_c
    eta_d = batt.eta_d
    dt = batt.dt
    violations = []

    charge = []; discharge = []; soc_csv = []
    imp_csv = []; exp_csv = []; demand_a = []
    prices_a = []; profit_csv = []

    for t, r in enumerate(rows):
        c_val, c_ok  = safe_float(r.get('charge_MW'))
        d_val, d_ok  = safe_float(r.get('discharge_MW'))
        s_val, s_ok  = safe_float(r.get('soc'))
        i_val, i_ok  = safe_float(r.get('import_MW'))
        e_val, e_ok  = safe_float(r.get('export_MW'))
        da_val, _    = safe_float(r.get('actual_demand'))
        pa_val, _    = safe_float(r.get('prices_actual'))
        pr_val, pr_ok = safe_float(r.get('profit_step'))

        if not c_ok: violations.append(f"[C16] t={t}: charge_MW invalid")
        if not d_ok: violations.append(f"[C16] t={t}: discharge_MW invalid")
        if not s_ok: violations.append(f"[C16] t={t}: soc invalid")
        if not i_ok: violations.append(f"[C16] t={t}: import_MW invalid")
        if not e_ok: violations.append(f"[C16] t={t}: export_MW invalid")

        charge.append(c_val); discharge.append(d_val); soc_csv.append(s_val)
        imp_csv.append(i_val); exp_csv.append(e_val); demand_a.append(da_val)
        prices_a.append(pa_val); profit_csv.append(pr_val)

    # Reconstruct SOC from dynamics
    soc_dyn = [soc0]
    for t in range(T):
        nxt = soc_dyn[t] + (eta_c * charge[t] * dt - discharge[t] * dt / eta_d) / C
        soc_dyn.append(nxt)

    # [C1] SOC bounds
    for t in range(T + 1):
        if soc_dyn[t] < batt.soc_min - tol:
            violations.append(f"[C1] soc[{t}]={soc_dyn[t]:.6f} < {batt.soc_min}")
        if soc_dyn[t] > batt.soc_max + tol:
            violations.append(f"[C1] soc[{t}]={soc_dyn[t]:.6f} > {batt.soc_max}")

    # [C2] Initial SOC
    if abs(soc_dyn[0] - soc0) > tol:
        violations.append(f"[C2] soc[0]={soc_dyn[0]:.6f} != soc0={soc0}")

    # [C3] Final SOC target
    if soc_dyn[T] < batt.soc_target - tol:
        violations.append(f"[C3] soc[24]={soc_dyn[T]:.6f} < target={batt.soc_target}")

    # [C4] Charge limits
    for t in range(T):
        if charge[t] < -tol:
            violations.append(f"[C4] t={t}: charge={charge[t]:.4f} < 0")
        if charge[t] > batt.cmax_kw + tol:
            violations.append(f"[C4] t={t}: charge={charge[t]:.4f} > {batt.cmax_kw}")

    # [C5] Discharge limits
    for t in range(T):
        if discharge[t] < -tol:
            violations.append(f"[C5] t={t}: discharge={discharge[t]:.4f} < 0")
        if discharge[t] > batt.dmax_kw + tol:
            violations.append(f"[C5] t={t}: discharge={discharge[t]:.4f} > {batt.dmax_kw}")

    # [C6] Simultaneous c & d
    for t in range(T):
        if charge[t] > tol and discharge[t] > tol:
            violations.append(f"[C6] t={t}: c={charge[t]:.4f} & d={discharge[t]:.4f}")

    # [C7] SOC dynamics self-consistency (CSV column)
    for t in range(T - 1):
        expected = soc_csv[t] + (eta_c * charge[t] * dt - discharge[t] * dt / eta_d) / C
        if abs(soc_csv[t + 1] - expected) > tol:
            violations.append(f"[C7] t={t}: CSV soc[{t+1}]={soc_csv[t+1]:.6f} != {expected:.6f}")

    # [C8] Grid balance
    for t in range(T):
        net = demand_a[t] + charge[t] - discharge[t]
        bal = imp_csv[t] - exp_csv[t]
        if abs(bal - net) > tol:
            violations.append(f"[C8] t={t}: imp-exp={bal:.4f} != net={net:.4f}")

    # [C9] Non-negative import
    for t in range(T):
        if imp_csv[t] < -tol:
            violations.append(f"[C9] t={t}: import={imp_csv[t]:.4f} < 0")

    # [C10] Non-negative export
    for t in range(T):
        if exp_csv[t] < -tol:
            violations.append(f"[C10] t={t}: export={exp_csv[t]:.4f} < 0")

    # [C11] Export restriction
    if not batt.allow_export:
        for t in range(T):
            if exp_csv[t] > tol:
                violations.append(f"[C11] t={t}: export={exp_csv[t]:.4f} but not allowed")

    # [C12] CSV SOC vs dynamics SOC
    for t in range(T):
        if abs(soc_csv[t] - soc_dyn[t]) > tol:
            violations.append(f"[C12] t={t}: CSV soc={soc_csv[t]:.6f} != dyn={soc_dyn[t]:.6f}")

    # [C13] Rolling-window energy
    ce = [charge[t] * dt * eta_c for t in range(T)]
    de = [discharge[t] * dt / eta_d for t in range(T)]
    for s in range(T - rolling_window + 1):
        wc = sum(ce[s:s+rolling_window])
        wd = sum(de[s:s+rolling_window])
        if wc > C + tol:
            violations.append(f"[C13] h{s}-{s+rolling_window-1}: charge_e={wc:.2f} > C={C}")
        if wd > C + tol:
            violations.append(f"[C13] h{s}-{s+rolling_window-1}: disch_e={wd:.2f} > C={C}")

    # [C14] Charge headroom
    for t in range(T):
        headroom = (batt.soc_max - soc_dyn[t]) * C
        ein = charge[t] * dt * eta_c
        if ein > headroom + tol:
            violations.append(f"[C14] t={t}: charge_e={ein:.4f} > headroom={headroom:.4f}")

    # [C15] Discharge available
    for t in range(T):
        avail = (soc_dyn[t] - batt.soc_min) * C
        eout = discharge[t] * dt / eta_d
        if eout > avail + tol:
            violations.append(f"[C15] t={t}: disch_e={eout:.4f} > avail={avail:.4f}")

    # [C17] Profit consistency
    for t in range(T):
        baseline = prices_a[t] * demand_a[t] * dt
        act_cost = prices_a[t] * imp_csv[t] * dt - prices_a[t] * exp_csv[t] * dt
        exp_profit = baseline - act_cost
        threshold = max(0.1, abs(exp_profit) * 0.01)
        if abs(profit_csv[t] - exp_profit) > threshold:
            violations.append(f"[C17] t={t}: profit={profit_csv[t]:.2f} != {exp_profit:.2f}")

    return (len(violations) == 0), violations


# ============================================================================
# MINIMAL Correction  (preserve GPT's intent, fix only what's broken)
# ============================================================================

def forward_simulate_and_clamp(c, d, batt, soc0):
    """
    Forward-simulate SOC from c, d decisions.
    If SOC breaches bounds at any step, reduce the c or d that caused it
    just enough to land exactly on the boundary.
    Returns (c, d, soc) — all modified in-place-safe copies.
    """
    T = 24
    C = batt.capacity_kwh
    eta_c = batt.eta_c
    eta_d = batt.eta_d
    dt = batt.dt

    c = list(c)  # copy
    d = list(d)

    soc = [soc0]
    for t in range(T):
        delta = (eta_c * c[t] * dt - d[t] * dt / eta_d) / C
        nxt = soc[t] + delta

        if nxt > batt.soc_max:
            # SOC too high — reduce charge at this hour
            # We want: soc[t] + (eta_c * c_new * dt - d[t]*dt/eta_d) / C = soc_max
            # => eta_c * c_new * dt = (soc_max - soc[t]) * C + d[t]*dt/eta_d
            needed = (batt.soc_max - soc[t]) * C + d[t] * dt / eta_d
            c[t] = max(0.0, needed / (eta_c * dt))
            nxt = batt.soc_max

        elif nxt < batt.soc_min:
            # SOC too low — reduce discharge at this hour
            # We want: soc[t] + (eta_c * c[t] * dt - d_new*dt/eta_d) / C = soc_min
            # => d_new * dt / eta_d = (soc[t] - soc_min) * C + eta_c * c[t] * dt
            available = (soc[t] - batt.soc_min) * C + eta_c * c[t] * dt
            d[t] = max(0.0, available * eta_d / dt)
            nxt = batt.soc_min

        soc.append(nxt)

    return c, d, soc


def minimal_correct(
    c_orig: List[float],
    d_orig: List[float],
    batt: BatteryConfig,
    soc0: float,
    forecast_prices: List[float],
    max_iterations: int = 3,
) -> Tuple[List[float], List[float], List[float]]:
    """
    Minimally correct GPT's charge/discharge decisions for feasibility.

    Step 1: Clip power limits
    Step 2: Net out simultaneous c & d
    Step 3: Forward-simulate SOC, clamp at boundaries
    Step 4: Fix final SOC target (greedy: add charge at cheapest hours)
    Step 5: Re-run step 3 to ensure no new breaches from step 4
    Repeat steps 3-5 up to max_iterations for convergence.

    Returns (c_corrected, d_corrected, soc_25_point).
    """
    T = 24
    C = batt.capacity_kwh
    eta_c = batt.eta_c
    eta_d = batt.eta_d
    dt = batt.dt

    c = [float(x) for x in c_orig]
    d = [float(x) for x in d_orig]

    # ------ Step 1: Clip to power limits ------
    for t in range(T):
        c[t] = max(0.0, min(c[t], batt.cmax_kw))
        d[t] = max(0.0, min(d[t], batt.dmax_kw))

    # ------ Step 2: Net out simultaneous c & d ------
    for t in range(T):
        if c[t] > 1e-6 and d[t] > 1e-6:
            # Convert to energy-equivalent for fair comparison
            ce = c[t] * eta_c * dt     # kWh into battery
            de = d[t] * dt / eta_d     # kWh out of battery
            if ce >= de:
                c[t] = (ce - de) / (eta_c * dt)
                d[t] = 0.0
            else:
                c[t] = 0.0
                d[t] = (de - ce) * eta_d / dt

    # ------ Steps 3-5: Iterate to convergence ------
    for iteration in range(max_iterations):

        # Step 3: Forward simulate + clamp
        c, d, soc = forward_simulate_and_clamp(c, d, batt, soc0)

        # Step 4: Fix final SOC target
        if soc[T] >= batt.soc_target - 1e-6:
            break   # all good

        deficit_kwh = (batt.soc_target - soc[T]) * C   # kWh needed in battery

        # Greedy: add charge (or reduce discharge) at cheapest hours first
        hour_order = sorted(range(T), key=lambda t: forecast_prices[t])

        for t in hour_order:
            if deficit_kwh <= 1e-6:
                break

            # Option A: Increase charge at hour t
            power_headroom = batt.cmax_kw - c[t]
            soc_headroom = (batt.soc_max - soc[t]) * C / (eta_c * dt) if soc[t] < batt.soc_max else 0.0
            max_add_power = max(0.0, min(power_headroom, soc_headroom))

            if max_add_power > 1e-6:
                energy_can_add = max_add_power * eta_c * dt
                energy_to_add = min(energy_can_add, deficit_kwh)
                c_add = energy_to_add / (eta_c * dt)
                c[t] += c_add
                deficit_kwh -= energy_to_add
                if deficit_kwh <= 1e-6:
                    break

            # Option B: Reduce discharge at hour t
            if d[t] > 1e-6:
                energy_can_save = d[t] * dt / eta_d   # kWh freed in battery
                energy_to_save = min(energy_can_save, deficit_kwh)
                d_reduce = energy_to_save * eta_d / dt
                d[t] = max(0.0, d[t] - d_reduce)
                deficit_kwh -= energy_to_save

        # After greedy fix, re-run forward simulation (Step 5 → loops back to Step 3)
        # The loop continues to ensure convergence

    # Final forward simulation to get clean SOC
    c, d, soc = forward_simulate_and_clamp(c, d, batt, soc0)

    return c, d, soc


# ============================================================================
# Build corrected rows from corrected c, d, soc
# ============================================================================

def build_corrected_rows(
    c: List[float],
    d: List[float],
    soc: List[float],
    actual_prices: List[float],
    forecast_prices: List[float],
    actual_demand: List[float],
    forecast_demand: List[float],
    batt: BatteryConfig,
) -> List[Dict]:
    rows = []
    dt = batt.dt
    for t in range(24):
        # Recompute import/export from corrected c, d + actual demand
        net = actual_demand[t] + c[t] - d[t]
        if batt.allow_export:
            if net >= 0:
                imp_t = net
                exp_t = 0.0
            else:
                imp_t = 0.0
                exp_t = -net
        else:
            imp_t = max(net, 0.0)
            exp_t = 0.0

        # profit_step = savings vs no-battery baseline
        baseline = actual_prices[t] * actual_demand[t] * dt
        actual_cost = actual_prices[t] * imp_t * dt - actual_prices[t] * exp_t * dt
        profit_step = baseline - actual_cost

        rows.append({
            'prices_actual':   round(actual_prices[t], 2),
            'prices_forecast': round(forecast_prices[t], 2),
            'actual_demand':   round(actual_demand[t], 2),
            'forecast_demand': round(forecast_demand[t], 2),
            'soc':             round(soc[t], 6),
            'charge_MW':       round(c[t], 4),
            'discharge_MW':    round(d[t], 4),
            'import_MW':       round(imp_t, 4),
            'export_MW':       round(exp_t, 4),
            'profit_step':     round(profit_step, 2),
        })
    return rows


# ============================================================================
# Main Pipeline
# ============================================================================

def main():
    print("=" * 70)
    print("Storage Arbitrage — EXHAUSTIVE Check & MINIMAL Correction")
    print("=" * 70)

    days = read_csv_days(INPUT_CSV)
    n_days = len(days)
    print(f"Read {n_days} days ({n_days * 24} rows) from {INPUT_CSV}")

    jsonl_records = None
    if JSONL_FILE:
        jsonl_records = load_jsonl_days(JSONL_FILE)
        print(f"Loaded {len(jsonl_records)} JSONL records from {JSONL_FILE}")

    default_batt = BatteryConfig()
    default_soc0 = 0.5

    columns = [
        'prices_actual', 'prices_forecast', 'actual_demand', 'forecast_demand',
        'soc', 'charge_MW', 'discharge_MW', 'import_MW', 'export_MW', 'profit_step'
    ]

    stats = {
        'feasible': 0,
        'violated': 0,
        'corrected_clean': 0,
        'corrected_residual': 0,
        'violation_types': {},
        'changes_made': [],   # (day_idx, n_hours_changed)
    }

    corrected_days = []

    for day_idx, day_rows in enumerate(days):
        # --- Battery config & soc0 ---
        if jsonl_records and day_idx < len(jsonl_records):
            batt, soc0 = battery_from_jsonl(jsonl_records[day_idx])
        else:
            batt = default_batt
            try:
                soc0 = float(day_rows[0]['soc'])
            except (ValueError, KeyError):
                soc0 = default_soc0

        # --- Price / demand vectors ---
        actual_prices   = [safe_float(r['prices_actual'])[0]   for r in day_rows]
        forecast_prices = [safe_float(r['prices_forecast'])[0] for r in day_rows]
        actual_demand   = [safe_float(r['actual_demand'])[0]   for r in day_rows]
        forecast_demand = [safe_float(r['forecast_demand'])[0] for r in day_rows]

        # --- Step 1: Check feasibility ---
        feasible, violations = exhaustive_check(day_rows, batt, soc0)

        if feasible:
            stats['feasible'] += 1
            corrected_days.append(day_rows)
            if day_idx < 5 or day_idx % 50 == 0:
                print(f"Day {day_idx:3d}  ✓ feasible")
            continue

        # --- Tally violation types ---
        stats['violated'] += 1
        for v in violations:
            tag = v.split(']')[0] + ']' if ']' in v else 'unknown'
            stats['violation_types'][tag] = stats['violation_types'].get(tag, 0) + 1

        n_v = len(violations)

        # --- Step 2: Extract GPT's original c, d ---
        c_orig = [safe_float(r['charge_MW'])[0]    for r in day_rows]
        d_orig = [safe_float(r['discharge_MW'])[0]  for r in day_rows]

        # --- Step 3: Minimal correction ---
        c_fix, d_fix, soc_fix = minimal_correct(
            c_orig, d_orig, batt, soc0, forecast_prices
        )

        # --- Step 4: Build corrected rows ---
        new_rows = build_corrected_rows(
            c_fix, d_fix, soc_fix,
            actual_prices, forecast_prices,
            actual_demand, forecast_demand, batt,
        )

        # --- Step 5: Verify the correction ---
        post_ok, post_violations = exhaustive_check(new_rows, batt, soc0)

        # Count how many hours actually changed
        hours_changed = sum(
            1 for t in range(24)
            if abs(c_fix[t] - c_orig[t]) > 0.01 or abs(d_fix[t] - d_orig[t]) > 0.01
        )
        stats['changes_made'].append((day_idx, hours_changed))

        if post_ok:
            stats['corrected_clean'] += 1
            if day_idx < 10 or day_idx % 50 == 0:
                print(f"Day {day_idx:3d}  ✗→✓ {n_v} violations fixed, "
                      f"{hours_changed}/24 hours adjusted")
        else:
            stats['corrected_residual'] += 1
            print(f"Day {day_idx:3d}  ⚠ {n_v} violations, {len(post_violations)} remain "
                  f"after correction, {hours_changed}/24 hours adjusted")
            for v in post_violations[:3]:
                print(f"           {v}")

        corrected_days.append(new_rows)

    # --- Write corrected CSV ---
    with open(OUTPUT_CSV, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=columns)
        writer.writeheader()
        for day_block in corrected_days:
            for row in day_block:
                out = {col: row.get(col, '') for col in columns}
                writer.writerow(out)

    total_rows = sum(len(d) for d in corrected_days)

    # --- Summary ---
    print()
    print("=" * 70)
    print(f"OUTPUT: {OUTPUT_CSV}  ({total_rows} rows, {n_days} days)")
    print("=" * 70)
    print(f"  Already feasible        : {stats['feasible']:4d} days")
    print(f"  Corrected (clean)       : {stats['corrected_clean']:4d} days")
    print(f"  Corrected (residual ⚠)  : {stats['corrected_residual']:4d} days")
    print(f"  Total with violations   : {stats['violated']:4d} days")

    if stats['changes_made']:
        changes = [h for _, h in stats['changes_made']]
        print(f"\n  Hours changed per corrected day:")
        print(f"    Mean: {np.mean(changes):.1f}  "
              f"Median: {np.median(changes):.0f}  "
              f"Max: {max(changes)}  "
              f"Min: {min(changes)}")

    if stats['violation_types']:
        print(f"\n  Violation breakdown (pre-correction):")
        for tag in sorted(stats['violation_types'].keys()):
            print(f"    {tag:8s} : {stats['violation_types'][tag]:4d} occurrences")
    print("=" * 70)


if __name__ == "__main__":
    main()

Storage Arbitrage — EXHAUSTIVE Check & MINIMAL Correction
Read 11 days (264 rows) from /content/drive/MyDrive/energy_finetuning_35/RF_0.8_Dec1_Table_5_gpt4_hourly_arbitrage_temp_sweep.csv
Day   0  ✗→✓ 2 violations fixed, 1/24 hours adjusted
Day   1  ✗→✓ 15 violations fixed, 2/24 hours adjusted
Day   2  ✗→✓ 9 violations fixed, 0/24 hours adjusted
Day   3  ✗→✓ 9 violations fixed, 0/24 hours adjusted
Day   4  ✗→✓ 15 violations fixed, 2/24 hours adjusted
Day   5  ✗→✓ 14 violations fixed, 1/24 hours adjusted
Day   6  ✗→✓ 9 violations fixed, 0/24 hours adjusted
Day   8  ✗→✓ 192 violations fixed, 0/24 hours adjusted
Day   9  ✗→✓ 18 violations fixed, 1/24 hours adjusted

OUTPUT: /content/drive/MyDrive/energy_finetuning_35/RF_0.8_Dec1_Table_5_gpt4_hourly_arbitrage_temp_sweep_corrected.csv  (264 rows, 11 days)
  Already feasible        :    1 days
  Corrected (clean)       :   10 days
  Corrected (residual ⚠)  :    0 days
  Total with violations   :   10 days

  Hours changed per corrected day:
